In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatib

In [2]:
import numpy as np 
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
print("All Packages Loaded successfully!")

/kaggle/input/datasets/banwait13/models/games_index.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/model.safetensors.index.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/config.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/preprocessor_config.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/video_preprocessor_config.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/model-00005-of-00005.safetensors
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/model-00001-of-00005.safetensors
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/README.md
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/model-00002-of-00005.safetensors
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/tokenizer.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/vocab.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4/tokenizer_config.json
/kaggle/input/datasets/banwait13/models/Qwen3.6-27B

In [3]:
!pip install --no-index --find-links=/kaggle/input/notebooks/banwait13/datasets-for-arc-agi/wheels sentence-transformers faiss-cpu bitsandbytes

Looking in links: /kaggle/input/notebooks/banwait13/datasets-for-arc-agi/wheels
Processing /kaggle/input/notebooks/banwait13/datasets-for-arc-agi/wheels/faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/banwait13/datasets-for-arc-agi/wheels/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl


# File WriteUP

In [4]:
%%writefile /kaggle/working/my_agent.py
"""
ARC-AGI-3 Agent  -  ITERATION 6 (Offline, in-process, no network LLM)

HARNESS REALITY (from v-o-i-d.ipynb):
  * This whole file is written to /kaggle/working/my_agent.py and run as `myagent`
    inside ARC-AGI-3-Agents. The framework talks to the game at http://gateway:8001;
    OUR code must make NO network calls of its own. There is NO vLLM/localhost.
  * All weights are mounted from a Kaggle dataset and loaded IN-PROCESS from local
    paths (see CONFIG). sentence-transformers + faiss-cpu are installed from wheels.
  * Hardware: 1x RTX 6000 Pro (Blackwell, 96GB). Qwen ~27B loads int8 (bitsandbytes)
    when available, else bf16 (~54GB, still fits). It is PRELOADED in a background
    thread at agent init so the first budgeted call doesn't stall the action loop.

DESIGN (mapped to roadmap.md):
  * World models are a VERIFIABLE DSL. PRIMARY synthesis is ENUMERATIVE and needs
    NO LLM and NO GPU -- it searches a small DSL program space against real,
    state-changing observations and only trusts a program that reproduces them
    (roadmap C4, V2, S9). Object-scoped ops (translate_color / slide_color / ...)
    cover the "a distinct-colored avatar moves" family that dominates ARC-AGI-3.
  * The local Qwen is a BUDGETED fallback (<=5 calls/level, roadmap V1): launched
    only when already loaded, generation hard-capped by max_time, and the launcher
    thread is covered by a watchdog + generation counter so a stall can never
    wedge the agent. LLM candidates are verified against FRESH transitions.
  * GRPO/PPO deleted (roadmap C5/S6/V8). No online training, no value head (V5).
    Exploration = count-based novelty with per-level epsilon decay.
  * MCTS: heuristic leaf value (novelty + activity), ACTION6 is ONE parametric
    node keyed to (cached) object centers, ZERO LLM calls in the loop (C4, V9).
  * Promotion needs verify >= 0.8 on changed transitions; a promoted model whose
    live accuracy (EWMA) collapses is DEMOTED and its rules banned so enumeration
    is forced to the next hypothesis instead of rediscovering the same one.
"""

import os
import json
import time
import math
import random
import traceback
import pickle
from collections import deque
from typing import Any, List, Tuple, Dict, Optional, Callable
from dataclasses import dataclass
import threading

import numpy as np
try:
    import torch                       # required on Kaggle (LLM); optional for local smoke tests
except ImportError:
    torch = None

# ==========================================
# CONFIG  (local dataset paths; NO network)
# ==========================================
ARC_DATA_ROOT = "/kaggle/input/notebooks/banwait13/datasets-for-arc-agi"
EMBED_MODEL_PATH = f"{ARC_DATA_ROOT}/models/all-MiniLM-L6-v2"
LLM_MODEL_PATH = f"/kaggle/input/datasets/banwait13/models/Qwen3.6-27B-NVFP4"

EMBED_DEVICE = "cpu"                 # MiniLM is tiny; keep the 96GB GPU for the 27B LLM
LLM_LOAD_8BIT = True                 # int8 via bitsandbytes if importable, else bf16 (fits 96GB)
LLM_CALL_BUDGET_PER_LEVEL = 5        # roadmap V1
LLM_MAX_NEW_TOKENS = 512

LLM_GEN_MAX_TIME_S = 90.0            # hard cap inside model.generate (MaxTimeCriteria)
SYNTH_MAX_INFLIGHT_S = 180.0         # watchdog: abandon a synth thread stuck past this
CONTEXT_TOKEN_BUDGET = 12000         # roadmap C2
CHECKPOINT_INTERVAL_S = 900          # 15 min (roadmap V7)

MIN_CHANGED_SAME_ACTION = 4          # trigger A: >=4 changed transitions for ONE action
MIN_CHANGED_TOTAL = 8                # trigger B: >=8 changed total ...
MIN_ACTIONS_WITH_MIN_SAMPLES = 2     # ... with >=2 actions having >= MIN_SAMPLES_PER_RULE
MIN_SAMPLES_PER_RULE = 2             # don't trust a per-action rule from a single example
MIN_CHANGED_FOR_VERIFY = 4           # verifier needs this much evidence to score at all
VERIFY_PROMOTE_THRESHOLD = 0.8       # >=4/5 exact match on changed transitions
DEMOTE_ACCURACY = 0.35               # EWMA accuracy below this -> demote + ban rules

EPSILON_START = 0.3
EPSILON_FLOOR = 0.05
EPSILON_DECAY = 0.99                 # per step within a level

# Force offline for HF/transformers so a stray lookup can never hang on the network.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if torch is not None else "cpu"

try:
    from sentence_transformers import SentenceTransformer
    import faiss
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False
    print("Warning: sentence-transformers/faiss not found. L3 memory -> flat storage.")

try:
    from arcengine import FrameData, GameAction, GameState
    HAS_ARCENGINE = True
except ImportError:
    HAS_ARCENGINE = False
    class GameAction:                      # noqa: N801  (local self-test stand-in)
        ACTION1 = "ACTION1"; ACTION2 = "ACTION2"; ACTION3 = "ACTION3"
        ACTION4 = "ACTION4"; ACTION5 = "ACTION5"; ACTION6 = "ACTION6"
        ACTION7 = "ACTION7"; RESET = "RESET"
    class GameState:                       # noqa: N801
        NOT_PLAYED = "NOT_PLAYED"; NOT_FINISHED = "NOT_FINISHED"
        WIN = "WIN"; GAME_OVER = "GAME_OVER"
    FrameData = Any

try:
    from agents.agent import Agent         # ARC-AGI-3-Agents framework (Kaggle rerun only)
except ImportError:
    class Agent:                           # noqa: N801
        def __init__(self, *a, **k):
            self.game_id = k.get("game_id", "local")
            self.action_counter = 0

VALID_ACTION_NAMES = {f"ACTION{i}" for i in range(1, 8)}


# ==========================================
# DATA STRUCTURES
# ==========================================

@dataclass
class Transition:
    prev: np.ndarray
    action: str
    next: np.ndarray
    reward: float
    diff_encoding: str
    timestamp: float

    @property
    def changed(self) -> bool:
        return self.prev.shape != self.next.shape or not np.array_equal(self.prev, self.next)

@dataclass
class WorldModel:
    hypothesis: str
    source: str                       # "enumerative" | "llm-dsl" | "llm-code"
    spec: Any                         # program list OR code string
    fn: Optional[Callable] = None
    accuracy: float = 0.0

@dataclass
class GridObject:
    color: int
    size: int
    bbox: Tuple[int, int, int, int]
    center: Tuple[float, float]
    shape_class: str
    mask: np.ndarray


def base_action(action: str) -> str:
    return "ACTION6" if str(action).startswith("ACTION6") else str(action)


def synth_ready(transitions: List[Transition]) -> bool:
    """Trigger for synthesis: >=4 changed transitions for the SAME action, OR
    >=8 changed total with at least 2 actions each having >=2 samples."""
    per: Dict[str, int] = {}
    for t in transitions:
        if t.changed:
            b = base_action(t.action)
            per[b] = per.get(b, 0) + 1
    if not per:
        return False
    if max(per.values()) >= MIN_CHANGED_SAME_ACTION:
        return True
    return (sum(per.values()) >= MIN_CHANGED_TOTAL
            and sum(1 for v in per.values() if v >= MIN_SAMPLES_PER_RULE) >= MIN_ACTIONS_WITH_MIN_SAMPLES)


# ==========================================
# COMPONENT 1: EYES (State Encoder)
# ==========================================

class StateEncoder:
    def __init__(self):
        self.shape_types = ['single_pixel', 'horizontal_line', 'vertical_line',
                            'square', 'rectangle', 'hollow_shape', 'irregular_blob']
        self._obj_cache: Dict[Any, List[GridObject]] = {}

    def detect_background(self, grid: np.ndarray) -> int:
        if grid.size == 0:
            return 0
        h, w = grid.shape
        if h == 0 or w == 0:
            return 0
        border = (np.concatenate([grid[0, :], grid[-1, :], grid[1:-1, 0], grid[1:-1, -1]])
                  if h > 2 and w > 2 else grid.flatten())
        if len(border) == 0:
            return 0
        unique, counts = np.unique(border, return_counts=True)
        if len(unique) > 0 and counts.max() > len(border) * 0.5:
            return int(unique[counts.argmax()])
        unique, counts = np.unique(grid, return_counts=True)
        return int(unique[counts.argmax()]) if len(unique) > 0 else 0

    def objects(self, grid: np.ndarray, bg: Optional[int] = None) -> List[GridObject]:
        """Cached object extraction (Issue 3: MCTS expands many nodes on the same
        grid -- flood fill must not be recomputed per expansion)."""
        if bg is None:
            bg = self.detect_background(grid)
        key = (grid.shape, int(bg), grid.tobytes())
        hit = self._obj_cache.get(key)
        if hit is not None:
            return hit
        objs = self.flood_fill_objects(grid, bg)
        if len(self._obj_cache) > 256:
            self._obj_cache.clear()
        self._obj_cache[key] = objs
        return objs

    def flood_fill_objects(self, grid: np.ndarray, bg_color: int, connectivity: int = 4) -> List[GridObject]:
        visited = np.zeros_like(grid, dtype=bool)
        objects: List[GridObject] = []
        h, w = grid.shape
        for r in range(h):
            for c in range(w):
                if not visited[r, c] and grid[r, c] != bg_color:
                    color = int(grid[r, c])
                    mask = np.zeros_like(grid, dtype=bool)
                    q = [(r, c)]
                    visited[r, c] = True
                    mask[r, c] = True
                    size = 0
                    rmin, cmin, rmax, cmax = r, c, r, c
                    while q:
                        cr, cc = q.pop(0)
                        size += 1
                        rmin, rmax = min(rmin, cr), max(rmax, cr)
                        cmin, cmax = min(cmin, cc), max(cmax, cc)
                        neigh = [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]
                        if connectivity == 8:
                            neigh += [(cr-1, cc-1), (cr-1, cc+1), (cr+1, cc-1), (cr+1, cc+1)]
                        for nr, nc in neigh:
                            if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == color:
                                visited[nr, nc] = True
                                mask[nr, nc] = True
                                q.append((nr, nc))
                    center = (rmin + (rmax - rmin) / 2.0, cmin + (cmax - cmin) / 2.0)
                    shape_class = self.classify_shape(mask, rmin, cmin, rmax, cmax, size)
                    objects.append(GridObject(color, size, (rmin, cmin, rmax, cmax), center, shape_class, mask))
        return objects

    def classify_shape(self, mask, rmin, cmin, rmax, cmax, size) -> str:
        h = rmax - rmin + 1
        w = cmax - cmin + 1
        if size == 1:
            return 'single_pixel'
        if h == 1 and w > 1 and size == w:
            return 'horizontal_line'
        if w == 1 and h > 1 and size == h:
            return 'vertical_line'
        if size == h * w:
            return 'square' if h == w else 'rectangle'
        if h > 2 and w > 2:
            inner = mask[rmin+1:rmax, cmin+1:cmax]
            if not np.any(inner) and size == (2*h + 2*w - 4):
                return 'hollow_shape'
        return 'irregular_blob'

    def rle_encode(self, mask: np.ndarray) -> str:
        flat = mask.flatten()
        if len(flat) == 0:
            return ""
        runs, curr, count = [], flat[0], 1
        for val in flat[1:]:
            if val == curr:
                count += 1
            else:
                runs.append(f"{count}{'T' if curr else 'F'}")
                curr, count = val, 1
        runs.append(f"{count}{'T' if curr else 'F'}")
        return ",".join(runs)

    def encode_state(self, grid: np.ndarray, level: int = 1, requested_objects: List[int] = None) -> str:
        h, w = grid.shape
        bg = self.detect_background(grid)
        unique, counts = np.unique(grid, return_counts=True)
        color_counts = dict(zip([int(x) for x in unique], [int(x) for x in counts]))
        l0 = f"Grid: {h}x{w}. BG: {bg}. Colors: {color_counts}."
        if level == 0:
            return l0
        objects = self.objects(grid, bg)
        obj_strs = [f"Obj{i}(C:{o.color}, S:{o.size}, BB:{o.bbox}, Shp:{o.shape_class})"
                    for i, o in enumerate(objects)]
        l1 = l0 + " Objects: [" + ", ".join(obj_strs) + "]"
        if level == 1:
            return l1
        if level == 2 and requested_objects:
            details = [f"Obj{i}_RLE:{self.rle_encode(objects[i].mask)}"
                       for i in requested_objects if i < len(objects)]
            return l1 + " Details: " + " ".join(details)
        if level == 3:
            if h <= 10 and w <= 10:
                grid_str = "\n".join(" ".join(map(str, row)) for row in grid)
                return l1 + f"\nFull Grid:\n{grid_str}"
            sh, sw = max(1, h // 10), max(1, w // 10)
            ds = grid[::sh, ::sw]
            grid_str = "\n".join(" ".join(map(str, row)) for row in ds)
            return l1 + f"\nDownsampled Grid:\n{grid_str}"
        return l1

    def encode_transition(self, prev: np.ndarray, nxt: np.ndarray, action: str) -> str:
        if prev.shape != nxt.shape:
            return f"Action {action} resized grid {prev.shape} -> {nxt.shape}"
        diff = nxt != prev
        n = int(np.sum(diff))
        if n == 0:
            return f"Action {action}: No change."
        added, ac = np.unique(nxt[diff], return_counts=True)
        removed, rc = np.unique(prev[diff], return_counts=True)
        if set(added.tolist()) == set(removed.tolist()) and len(added) > 0:
            return f"Action {action}: movement/swap of {n} px, colors {added.tolist()}."
        add_d = dict(zip([int(x) for x in added], [int(x) for x in ac]))
        rem_d = dict(zip([int(x) for x in removed], [int(x) for x in rc]))
        return f"Action {action}: {n} px changed. Added: {add_d}, Removed: {rem_d}."


# ==========================================
# THE DSL (verifiable grid transforms)
# ==========================================

class GridDSL:
    @staticmethod
    def identity(g, bg): return g.copy()

    @staticmethod
    def translate(g, bg, dr=0, dc=0):
        out = np.full_like(g, bg)
        h, w = g.shape
        for r in range(h):
            for c in range(w):
                nr, nc = r + dr, c + dc
                if 0 <= nr < h and 0 <= nc < w:
                    out[nr, nc] = g[r, c]
        return out

    @staticmethod
    def recolor(g, bg, src=None, dst=None):
        out = g.copy()
        if src is not None and dst is not None:
            out[g == src] = dst
        return out

    @staticmethod
    def reflect_h(g, bg): return g[:, ::-1].copy()

    @staticmethod
    def reflect_v(g, bg): return g[::-1, :].copy()

    @staticmethod
    def rotate90(g, bg):
        r = np.rot90(g, k=-1)
        return r.copy() if r.shape == g.shape else g.copy()

    @staticmethod
    def gravity(g, bg, direction="down"):
        out = np.full_like(g, bg)
        if direction in ("down", "up"):
            for c in range(g.shape[1]):
                col = g[:, c]
                vals = col[col != bg]
                if direction == "down":
                    out[g.shape[0] - len(vals):, c] = vals
                else:
                    out[:len(vals), c] = vals
        else:
            for r in range(g.shape[0]):
                row = g[r, :]
                vals = row[row != bg]
                if direction == "right":
                    out[r, g.shape[1] - len(vals):] = vals
                else:
                    out[r, :len(vals)] = vals
        return out

    @staticmethod
    def flood_replace(g, bg, r=0, c=0, dst=None):
        out = g.copy()
        h, w = g.shape
        if not (0 <= r < h and 0 <= c < w) or dst is None:
            return out
        target = g[r, c]
        if target == dst:
            return out
        stack, seen = [(r, c)], set()
        while stack:
            cr, cc = stack.pop()
            if (cr, cc) in seen or not (0 <= cr < h and 0 <= cc < w) or g[cr, cc] != target:
                continue
            seen.add((cr, cc))
            out[cr, cc] = dst
            stack += [(cr-1, cc), (cr+1, cc), (cr, cc-1), (cr, cc+1)]
        return out

    # ---- object(color)-scoped ops: "the avatar/object of color X does Y" ----

    @staticmethod
    def translate_color(g, bg, color=None, dr=0, dc=0):
        """Move ONLY the cells of `color` by (dr, dc); vacated cells become bg."""
        if color is None:
            return g.copy()
        out = g.copy()
        src = (g == color)
        out[src] = bg
        h, w = g.shape
        rs, cs = np.nonzero(src)
        nr, nc = rs + dr, cs + dc
        keep = (nr >= 0) & (nr < h) & (nc >= 0) & (nc < w)
        out[nr[keep], nc[keep]] = color
        return out

    @staticmethod
    def slide_color(g, bg, color=None, direction="down"):
        """Rigid-slide ALL cells of `color` in `direction` until any of them would
        hit a wall or a non-bg cell of another color (sokoban/ice-floor movement)."""
        if color is None:
            return g.copy()
        dr, dc = {"down": (1, 0), "up": (-1, 0), "left": (0, -1), "right": (0, 1)}.get(direction, (1, 0))
        h, w = g.shape
        rs, cs = np.nonzero(g == color)
        if len(rs) == 0:
            return g.copy()
        others = (g != bg) & (g != color)
        k = 0
        while True:
            nr, nc = rs + (k + 1) * dr, cs + (k + 1) * dc
            if ((nr < 0) | (nr >= h) | (nc < 0) | (nc >= w)).any():
                break
            if others[nr, nc].any():
                break
            k += 1
        out = g.copy()
        out[rs, cs] = bg
        out[rs + k * dr, cs + k * dc] = color
        return out

    @staticmethod
    def swap_colors(g, bg, a=None, b=None):
        out = g.copy()
        if a is not None and b is not None:
            out[g == a] = b
            out[g == b] = a
        return out

    @staticmethod
    def delete_color(g, bg, color=None):
        out = g.copy()
        if color is not None:
            out[g == color] = bg
        return out

    OPS: Dict[str, Callable] = {}

    @classmethod
    def help(cls) -> str:
        return (
            "Ops (grid + optional args; all shape-preserving):\n"
            "  identity()                         no change\n"
            "  translate(dr, dc)                  shift all non-bg pixels\n"
            "  recolor(src, dst)                  repaint color src as dst\n"
            "  reflect_h() / reflect_v()          mirror horizontally / vertically\n"
            "  rotate90()                         rotate clockwise (square grids)\n"
            "  gravity(direction=down|up|left|right)\n"
            "  flood_replace(r, c, dst)           bucket-fill region at (r,c) with dst\n"
            "  translate_color(color, dr, dc)     move ONLY that color's cells\n"
            "  slide_color(color, direction)      rigid-slide that color until blocked\n"
            "  swap_colors(a, b)                  exchange two colors\n"
            "  delete_color(color)                erase a color to background\n"
        )

GridDSL.OPS = {
    "identity": GridDSL.identity, "translate": GridDSL.translate,
    "recolor": GridDSL.recolor, "reflect_h": GridDSL.reflect_h,
    "reflect_v": GridDSL.reflect_v, "rotate90": GridDSL.rotate90,
    "gravity": GridDSL.gravity, "flood_replace": GridDSL.flood_replace,
    "translate_color": GridDSL.translate_color, "slide_color": GridDSL.slide_color,
    "swap_colors": GridDSL.swap_colors, "delete_color": GridDSL.delete_color,
}


def compile_program(program: List[dict], encoder: StateEncoder) -> Optional[Callable]:
    """DSL rule list -> transition(grid, action). No exec(). None if op unknown."""
    if not isinstance(program, list) or not program:
        return None
    for rule in program:
        if not isinstance(rule, dict) or rule.get("op") not in GridDSL.OPS:
            return None

    def transition(grid: np.ndarray, action: str) -> np.ndarray:
        g = np.asarray(grid)
        bg = encoder.detect_background(g)
        out = g.copy()
        act_base = base_action(action)
        for rule in program:
            want = rule.get("action", "*")
            if want not in ("*", act_base):
                continue
            try:
                out = GridDSL.OPS[rule["op"]](out, bg, **(rule.get("args", {}) or {}))
            except Exception:
                return g.copy()
            if not isinstance(out, np.ndarray) or out.shape != g.shape:
                return g.copy()
        return out
    return transition


def compile_python(code: str) -> Optional[Callable]:
    """Sandboxed fallback for free-form LLM `transition`. Restricted namespace."""
    import ast
    try:
        tree = ast.parse(code)
        if not any(isinstance(n, ast.FunctionDef) and n.name == "transition" for n in tree.body):
            return None
        allowed = {"np": np, "math": math, "dsl": GridDSL, "__builtins__": {
            "range": range, "len": len, "min": min, "max": max, "abs": abs,
            "int": int, "float": float, "enumerate": enumerate, "zip": zip,
        }}
        local: Dict[str, Any] = {}
        exec(code, allowed, local)  # noqa: S102 - restricted + verified before trust
        return local.get("transition")
    except Exception as e:
        print(f"compile_python failed: {e}")
        return None


# ==========================================
# ENUMERATIVE SYNTHESIZER  (primary, no LLM, no GPU)
# ==========================================

class EnumerativeSynthesizer:
    """Search a small DSL program space for a per-action rule set that exactly
    reproduces the observed state-changing transitions. Deterministic, ~ms."""

    def __init__(self, encoder: StateEncoder):
        self.encoder = encoder

    def _changed_colors(self, samples) -> List[int]:
        cols = set()
        for prev, nxt in samples:
            d = prev != nxt
            cols.update(int(x) for x in np.unique(prev[d]))
            cols.update(int(x) for x in np.unique(nxt[d]))
        return sorted(cols)

    def _candidate_ops(self, samples, palette: List[int]):
        cands = []
        # Global geometry first (whole-grid physics).
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr or dc:
                    cands.append(("translate", {"dr": dr, "dc": dc}))
        for d in ("down", "up", "left", "right"):
            cands.append(("gravity", {"direction": d}))
        cands += [("reflect_h", {}), ("reflect_v", {}), ("rotate90", {})]
        # Object(color)-scoped: enumerate ONLY over colors seen changing.
        moved = self._changed_colors(samples)
        bg0 = self.encoder.detect_background(samples[0][0]) if samples else 0
        for col in moved:
            if col == bg0:
                continue                       # "moving the background" is not an object rule
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1),
                           (-2, 0), (2, 0), (0, -2), (0, 2)):
                cands.append(("translate_color", {"color": col, "dr": dr, "dc": dc}))
            for d in ("down", "up", "left", "right"):
                cands.append(("slide_color", {"color": col, "direction": d}))
            cands.append(("delete_color", {"color": col}))
        for i, a in enumerate(moved):
            for b in moved[i + 1:]:
                cands.append(("swap_colors", {"a": a, "b": b}))
        # Recolor last; src must be a color that actually changed.
        for s in moved:
            for d in palette:
                if s != d:
                    cands.append(("recolor", {"src": int(s), "dst": int(d)}))
        return cands

    def _fits(self, op: str, args: dict, samples) -> bool:
        fn = GridDSL.OPS[op]
        for prev, nxt in samples:
            bg = self.encoder.detect_background(prev)
            try:
                out = fn(prev.copy(), bg, **args)
            except Exception:
                return False
            if not (isinstance(out, np.ndarray) and out.shape == nxt.shape and np.array_equal(out, nxt)):
                return False
        return True

    def synthesize(self, transitions: List[Transition], banned: Optional[set] = None) -> Optional[WorldModel]:
        banned = banned or set()
        by_action: Dict[str, list] = {}
        palette = set()
        for t in transitions:
            if not t.changed or t.prev.shape != t.next.shape:
                continue
            by_action.setdefault(base_action(t.action), []).append((t.prev, t.next))
            palette.update(int(x) for x in np.unique(t.prev))
            palette.update(int(x) for x in np.unique(t.next))
        if not by_action:
            return None
        program = []
        for act, samples in by_action.items():
            use = samples[-5:]
            if len(use) < MIN_SAMPLES_PER_RULE:
                continue
            cands = self._candidate_ops(use, sorted(palette))
            for op, args in cands:
                rule = {"action": act, "op": op, "args": args}
                if json.dumps(rule, sort_keys=True) in banned:
                    continue
                if self._fits(op, args, use):
                    program.append(rule)
                    break
        if not program:
            return None
        fn = compile_program(program, self.encoder)
        if fn is None:
            return None
        desc = "; ".join(f"{r['action']}->{r['op']}{r['args']}" for r in program)
        return WorldModel(hypothesis=f"Enumerated: {desc}", source="enumerative", spec=program, fn=fn)


# ==========================================
# COMPONENT 3: BRAIN  (in-process local LLM, offline, lazy, budgeted)
# ==========================================

class LocalLLM:
    """Loads Qwen from a local directory with transformers. NO network. Preloaded
    in a background thread; a missing/OOM model never blocks or crashes the agent.
    Load is lock-guarded so preload + synth threads can't double-load 27B weights."""

    def __init__(self, model_path: str = LLM_MODEL_PATH):
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self._load_attempted = False
        self._load_lock = threading.Lock()
        self.available = torch is not None and os.path.isdir(model_path)

    def is_ready(self) -> bool:
        return self.model is not None

    def ensure_loaded(self) -> bool:
        if self.model is not None:
            return True
        if not self.available:
            return False
        with self._load_lock:
            if self.model is not None:
                return True
            if self._load_attempted:
                return False
            self._load_attempted = True
            try:
                from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
                self.tokenizer = AutoTokenizer.from_pretrained(self.model_path, local_files_only=True, trust_remote_code=True)
                kwargs: Dict[str, Any] = dict(local_files_only=True, device_map="auto",
                                              low_cpu_mem_usage=True, trust_remote_code=True)
                
                # Patch for Kaggle's older transformers version not recognizing new Qwen types
                config = AutoConfig.from_pretrained(self.model_path, local_files_only=True, trust_remote_code=True)
                if getattr(config, "model_type", "") == "qwen3_5":
                    print("[LLM] Patching unrecognized model_type 'qwen3_5' to 'qwen2' for compatibility...")
                    config.model_type = "qwen2"
                kwargs["config"] = config

                quant = "bf16"
                if LLM_LOAD_8BIT:
                    try:
                        import bitsandbytes  # noqa: F401
                        from transformers import BitsAndBytesConfig
                        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
                        quant = "int8"
                    except Exception:
                        kwargs["torch_dtype"] = torch.bfloat16   # 27B bf16 ~54GB fits the 96GB card
                else:
                    kwargs["torch_dtype"] = torch.bfloat16
                t0 = time.time()
                self.model = AutoModelForCausalLM.from_pretrained(self.model_path, **kwargs)
                self.model.eval()
                print(f"[LLM] loaded {self.model_path} ({quant}) in {time.time() - t0:.0f}s")
                return True
            except Exception as e:
                print(f"[LLM] load failed ({e}); running without LLM.")
                self.model = None
                self.available = False
                return False

    def generate(self, prompt: str, max_new_tokens: int = LLM_MAX_NEW_TOKENS, temperature: float = 0.2) -> str:
        if not self.ensure_loaded():
            return ""
        try:
            return self._generate_inner(prompt, max_new_tokens, temperature)
        except Exception as e:
            print(f"[LLM] generate failed: {e}")
            return ""

    def _generate_inner(self, prompt: str, max_new_tokens: int, temperature: float) -> str:
        with torch.no_grad():
            msgs = [{"role": "user", "content": prompt}]
            inputs = self.tokenizer.apply_chat_template(
                msgs, add_generation_prompt=True, return_tensors="pt",
                enable_thinking=False,   # Qwen3: don't burn the token budget on <think>; ignored by other templates
            ).to(self.model.device)
            out = self.model.generate(
                inputs, max_new_tokens=max_new_tokens,
                max_time=LLM_GEN_MAX_TIME_S,
                do_sample=temperature > 0, temperature=max(temperature, 1e-2),
                pad_token_id=self.tokenizer.eos_token_id)
            return self.tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

    @staticmethod
    def _extract_json(text: str) -> dict:
        try:
            return json.loads(text)
        except Exception:
            s, e = text.find("{"), text.rfind("}")
            if 0 <= s < e:
                return json.loads(text[s:e + 1])
            raise

    def synthesize_world_model(self, observations: str) -> Optional[WorldModel]:
        prompt = f"""You are reverse-engineering the hidden rules of a grid puzzle.
Observed action -> effect transitions:
{observations}

{GridDSL.help()}
Return JSON ONLY. Prefer the FEWEST ops that explain the data:
{{"hypothesis": "one sentence", "program": [{{"action": "ACTION1", "op": "gravity", "args": {{"direction": "down"}}}}]}}
Use "action": "*" for a rule applying to every action. If no op fits, return instead:
{{"hypothesis": "...", "code": "def transition(state_grid, action_str):\\n    return state_grid.copy()"}}"""
        raw = self.generate(prompt, max_new_tokens=LLM_MAX_NEW_TOKENS, temperature=0.2)
        if not raw:
            return None
        try:
            res = self._extract_json(raw)
        except Exception:
            return None
        if res.get("program"):
            return WorldModel(res.get("hypothesis", ""), "llm-dsl", res["program"])
        if res.get("code"):
            return WorldModel(res.get("hypothesis", ""), "llm-code", res["code"])
        return None

    def reflect(self, trajectory: str, failure_reason: str) -> str:
        prompt = f"""Trajectory:
{trajectory}
Failure: {failure_reason}
Give an updated one-sentence hypothesis of the puzzle rules.
Return JSON: {{"updated_hypothesis": "..."}}"""
        raw = self.generate(prompt, max_new_tokens=200, temperature=0.5)
        if not raw:
            return "Reflection unavailable."
        try:
            return self._extract_json(raw).get("updated_hypothesis", "Reflection failed.")
        except Exception:
            return "Reflection failed."

    def plan_clicks(self, board_text: str, observations: str, reactive: str,
                    hypothesis: str) -> List[Tuple[int, int]]:
        """Reasoning layer for click-only games (CSP / logic puzzles the coverage
        planner cannot solve blind). Given the board and what clicks have been
        observed to do, return an ORDERED list of (row, col) pixel cells to click
        toward completing the level. Empty list => no useful plan (caller falls back
        to coverage). Coords are 0-63; caller clamps + validates regardless."""
        prompt = f"""You are solving an interactive grid puzzle by CLICKING cells.
The board is a {board_text.count(chr(10)) + 1}-row grid; each char is a cell colour
('.' = background). Rows are numbered from the top (0), columns from the left (0).
A click is at a (row, col) coordinate.

Current board:
{board_text}

What clicks have done so far (action -> effect):
{observations or "nothing observed yet"}
Cells already found to react to a click (row,col): {reactive or "none yet"}
Working hypothesis of the rules: {hypothesis}

Reason about the GOAL (what a completed board looks like) and the click MECHANIC,
then give the shortest click sequence that makes progress toward completing the
level. Return JSON ONLY, at most 12 clicks, coordinates in [0,63]:
{{"goal": "one sentence", "clicks": [[row, col], [row, col]]}}"""
        raw = self.generate(prompt, max_new_tokens=LLM_MAX_NEW_TOKENS, temperature=0.3)
        if not raw:
            return []
        try:
            res = self._extract_json(raw)
        except Exception:
            return []
        out: List[Tuple[int, int]] = []
        for pair in (res.get("clicks") or [])[:12]:
            try:
                r, c = int(pair[0]), int(pair[1])
            except (TypeError, ValueError, IndexError):
                continue
            out.append((r, c))
        return out

    def theorize(self, board_text: str, theory: str, evidence: str,
                 trouble: str) -> Optional[dict]:
        """Schema-harness Phase 2 THEORIZE step (movement games): given the
        board, the locally-learned theory, and why the local loop is stuck,
        propose a BOUNDED revision -- a goal hypothesis and/or grounding edits
        (never free-form code). The caller certifies the proposal against the
        full timeline before adoption (ExecPlanner.propose_theory), so a
        hallucinated theory can never drive planning."""
        prompt = f"""You are reverse-engineering a grid game in which an avatar moves with UP/DOWN/LEFT/RIGHT.
Current board, one char per cell ('.' = background; other chars are colour
indices 0-9/a-f). Rows are numbered from the top (0), columns from the left (0):
{board_text}

Theory learned so far from real transitions:
{theory}

Recent observed transitions:
{evidence or "none"}

Why local reasoning is stuck: {trouble}

Reason about (1) the LEVEL GOAL -- which cell the avatar must reach -- and
(2) whether any learned grounding is wrong: a colour wrongly marked as a wall,
a wall colour not yet marked, or object colours that are decoys, not the goal.
Return JSON ONLY (use null for "goal" if unsure; colours are INTEGER indices):
{{"hypothesis": "one sentence",
 "goal": [row, col],
 "not_obstacle_colors": [],
 "obstacle_colors": [],
 "decoy_colors": []}}"""
        raw = self.generate(prompt, max_new_tokens=LLM_MAX_NEW_TOKENS, temperature=0.3)
        if not raw:
            return None
        try:
            res = self._extract_json(raw)
        except Exception:
            return None
        return res if isinstance(res, dict) else None


_LLM_SINGLETON: Optional[LocalLLM] = None

def get_local_llm() -> LocalLLM:
    """One 27B model per PROCESS, shared across agent instances/games."""
    global _LLM_SINGLETON
    if _LLM_SINGLETON is None:
        _LLM_SINGLETON = LocalLLM()
    return _LLM_SINGLETON


_EMBEDDER = None
_EMBEDDER_TRIED = False

def get_embedder():
    """One MiniLM per process (agents may be constructed once per game)."""
    global _EMBEDDER, _EMBEDDER_TRIED
    if _EMBEDDER is None and not _EMBEDDER_TRIED and HAS_SENTENCE_TRANSFORMERS:
        _EMBEDDER_TRIED = True
        try:
            _EMBEDDER = SentenceTransformer(EMBED_MODEL_PATH, device=EMBED_DEVICE)
        except Exception as e:
            print(f"Embedder init failed ({e}); L3 -> flat storage.")
            _EMBEDDER = None
    return _EMBEDDER


# ==========================================
# COMPONENT 2: MEMORY  (embedder on CPU; rule-based compression)
# ==========================================

class MemoryManager:
    def __init__(self):
        self.l0_working: List[Transition] = []
        self.l1_episodes: List[str] = []
        self.l2_hypothesis: str = "No rules discovered yet."
        self.max_tokens = CONTEXT_TOKEN_BUDGET
        self.embedder = get_embedder()
        self.l3_archive_index = None
        self.l3_archive_texts: List[str] = []
        if self.embedder is not None:
            try:
                self.l3_archive_index = faiss.IndexFlatL2(384)
            except Exception:
                self.l3_archive_index = None

    def estimate_tokens(self, text: str) -> int:
        return int(len(text.split()) / 0.75)

    def add_transition(self, t: Transition):
        self.l0_working.append(t)
        if self.estimate_tokens(self.get_context_for_llm()) > self.max_tokens * 0.9:
            self.compress_l0_to_l1()
        if len(self.l1_episodes) > 5:
            self.embed_and_archive(self.l1_episodes.pop(0))

    def compress_l0_to_l1(self):
        if not self.l0_working:
            return
        chunk = self.l0_working[:10]
        self.l0_working = self.l0_working[10:]
        acts = sorted({t.action for t in chunk})
        summary = (f"Steps: tried {acts} over {len(chunk)} moves, "
                   f"{sum(1 for t in chunk if t.changed)} changed the grid, "
                   f"{sum(1 for t in chunk if t.reward > 0)} rewarded.")
        self.l1_episodes.append(summary)

    def embed_and_archive(self, text: str):
        self.l3_archive_texts.append(text)
        if self.embedder is not None and self.l3_archive_index is not None:
            try:
                emb = self.embedder.encode([text])[0]
                self.l3_archive_index.add(np.array([emb], dtype=np.float32))
            except Exception:
                pass

    def retrieve_similar(self, query: str, k: int = 3) -> List[str]:
        if not self.l3_archive_texts:
            return []
        if self.embedder is not None and self.l3_archive_index is not None:
            try:
                emb = self.embedder.encode([query])[0]
                _, idx = self.l3_archive_index.search(np.array([emb], dtype=np.float32),
                                                      min(k, len(self.l3_archive_texts)))
                return [self.l3_archive_texts[i] for i in idx[0] if i < len(self.l3_archive_texts)]
            except Exception:
                pass
        return self.l3_archive_texts[-k:]

    def observation_digest(self, transitions: List[Transition], max_items: int = 15) -> str:
        """Compact, LLM-ready summary of recent CHANGED transitions (keeps prompts small)."""
        changed = [t for t in transitions if t.changed][-max_items:]
        return "\n".join(f"{t.diff_encoding}" for t in changed) or "No state changes observed yet."

    def get_context_for_llm(self, current_state_desc: str = "") -> str:
        ctx = "=== Hypothesis ===\n" + self.l2_hypothesis + "\n\n"
        if current_state_desc and self.l3_archive_texts:
            sim = self.retrieve_similar(current_state_desc)
            if sim:
                ctx += "=== Relevant Past Episodes ===\n" + "\n".join(sim) + "\n\n"
        if self.l1_episodes:
            ctx += "=== Episode Summaries ===\n" + "\n".join(self.l1_episodes) + "\n\n"
        if self.l0_working:
            ctx += "=== Recent Working Memory ===\n"
            for t in self.l0_working[-10:]:
                ctx += f"Action: {t.action} -> {t.diff_encoding}\n"
        return ctx


# ==========================================
# COMPONENT 4: PLANNER (WorldModel + MCTS)
# ==========================================

class WorldModelManager:
    def __init__(self, llm: LocalLLM, encoder: StateEncoder):
        self.llm = llm
        self.encoder = encoder
        self.enumerator = EnumerativeSynthesizer(encoder)
        self.active_model: Optional[WorldModel] = None
        self.banned_rules: set = set()
        self._lock = threading.Lock()

    def _compile(self, wm: WorldModel) -> Optional[Callable]:
        if wm.source in ("enumerative", "llm-dsl"):
            return compile_program(wm.spec, self.encoder)
        return compile_python(wm.spec)

    def verify(self, fn: Callable, transitions: List[Transition]) -> float:
        changed = [t for t in transitions if t.changed and t.prev.shape == t.next.shape]
        if len(changed) < MIN_CHANGED_FOR_VERIFY:
            return 0.0
        sample = changed[-5:]
        correct = 0
        for t in sample:
            try:
                pred = fn(t.prev.copy(), t.action)
                if isinstance(pred, np.ndarray) and pred.shape == t.next.shape and np.array_equal(pred, t.next):
                    correct += 1
            except Exception:
                pass
        return correct / len(sample)

    def _promote(self, wm: WorldModel, fn: Callable, score: float):
        wm.fn = fn
        wm.accuracy = score
        with self._lock:
            self.active_model = wm
        print(f"[world-model] PROMOTED ({wm.source}, verify={score:.2f}): {wm.hypothesis}")

    def demote(self, wm: WorldModel):
        """Live accuracy collapsed: drop the model and ban its rules so the
        enumerator is forced to the NEXT hypothesis, not the same one again."""
        with self._lock:
            if self.active_model is wm:
                self.active_model = None
        if isinstance(wm.spec, list):
            for rule in wm.spec:
                try:
                    self.banned_rules.add(json.dumps(rule, sort_keys=True))
                except Exception:
                    pass
        print(f"[world-model] DEMOTED (acc={wm.accuracy:.2f}): {wm.hypothesis}")

    def reset_level(self):
        with self._lock:
            self.active_model = None
        self.banned_rules.clear()

    def try_enumerative(self, transitions: List[Transition]) -> bool:
        """Synchronous, cheap. Returns True if a model was promoted."""
        wm = self.enumerator.synthesize(transitions, self.banned_rules)
        if wm is None or wm.fn is None:
            return False
        score = self.verify(wm.fn, transitions)
        if score >= VERIFY_PROMOTE_THRESHOLD:
            self._promote(wm, wm.fn, score)
            return True
        return False

    def synthesize_with_llm(self, observations: str,
                            transitions_provider: Callable[[], List[Transition]]):
        """Background-thread safe LLM fallback (roadmap V10). Verifies against
        FRESH transitions at completion time, so a slow generation can't get
        promoted on stale evidence."""
        try:
            candidate = self.llm.synthesize_world_model(observations)
            if candidate is None:
                return
            fn = self._compile(candidate)
            if fn is None:
                return
            transitions = transitions_provider() or []
            score = self.verify(fn, transitions)
            if score >= VERIFY_PROMOTE_THRESHOLD:
                self._promote(candidate, fn, score)
            else:
                print(f"[world-model] LLM candidate rejected (verify={score:.2f}).")
        except Exception:
            traceback.print_exc()

    def predict(self, state: np.ndarray, action: str) -> np.ndarray:
        with self._lock:
            m = self.active_model
        if m and m.fn:
            try:
                nxt = m.fn(state.copy(), action)
                if isinstance(nxt, np.ndarray) and nxt.shape == state.shape:
                    return nxt
            except Exception:
                pass
        return state.copy()


class ExploitGate:
    """Hysteresis gate (Issue 4): enter EXPLOIT only after ENTER_STREAK consecutive
    correct one-step predictions; fall back to EXPLORE when the rolling accuracy
    over the last 10 changed steps drops below EXIT_AVG."""
    ENTER_STREAK = 5
    EXIT_AVG = 0.5

    def __init__(self):
        self.window: List[float] = []
        self.streak = 0
        self.mode = "EXPLORE"

    def update(self, hit: float):
        self.window.append(hit)
        self.window = self.window[-20:]
        self.streak = self.streak + 1 if hit >= 0.7 else 0
        if self.mode == "EXPLORE" and self.streak >= self.ENTER_STREAK:
            self.mode = "EXPLOIT"
        elif (self.mode == "EXPLOIT" and len(self.window) >= 10
              and float(np.mean(self.window[-10:])) < self.EXIT_AVG):
            self.mode = "EXPLORE"
            self.streak = 0

    def should_exploit(self) -> bool:
        return self.mode == "EXPLOIT"


class MCTSNode:
    __slots__ = ("state", "parent", "action", "children", "visits", "value_sum", "prior", "expanded")

    def __init__(self, state, parent=None, action=None, prior=1.0):
        self.state = state
        self.parent = parent
        self.action = action
        self.children: Dict[str, "MCTSNode"] = {}
        self.visits = 0
        self.value_sum = 0.0
        self.prior = prior
        self.expanded = False

    def value(self):
        return self.value_sum / self.visits if self.visits else 0.0

    def uct_select(self, c_puct=1.414):
        best, best_child = -1e18, None
        for child in self.children.values():
            u = c_puct * child.prior * math.sqrt(self.visits + 1) / (1 + child.visits)
            s = child.value() + u
            if s > best:
                best, best_child = s, child
        return best_child


class MCTSPlanner:
    """Heuristic-value MCTS. No neural value head, no LLM. ACTION6 = one node."""
    def __init__(self, world_model: WorldModelManager, encoder: StateEncoder, counter: "StateCounter"):
        self.world_model = world_model
        self.encoder = encoder
        self.counter = counter

    def _candidate_actions(self, grid: np.ndarray, valid: List[str]) -> List[str]:
        acts = [a for a in valid if not a.startswith("ACTION6")]
        if any(a.startswith("ACTION6") for a in valid):
            centers = [(int(o.center[0]), int(o.center[1]))
                       for o in self.encoder.objects(grid)[:3]]   # cached (Issue 3)
            if not centers:
                centers = [(grid.shape[0] // 2, grid.shape[1] // 2)]
            acts += [f"ACTION6_r{r}_c{c}" for r, c in centers]
        return acts or ["ACTION1"]

    def _leaf_value(self, grid: np.ndarray) -> float:
        novelty = self.counter.peek_bonus(grid)
        bg = self.encoder.detect_background(grid)
        activity = float(np.mean(grid != bg))
        return novelty + 0.1 * activity

    def search(self, root_grid: np.ndarray, valid_actions: List[str], n_iterations=60) -> str:
        root = MCTSNode(root_grid)
        for _ in range(n_iterations):
            node, path = root, [root]
            while node.expanded and node.children:
                nxt = node.uct_select()
                if nxt is None:
                    break
                node = nxt
                path.append(node)
            if not node.expanded:
                cands = self._candidate_actions(node.state, valid_actions)
                prior = 1.0 / len(cands)
                for a in cands:
                    ns = self.world_model.predict(node.state, a)
                    node.children[a] = MCTSNode(ns, parent=node, action=a, prior=prior)
                node.expanded = True
            val = self._leaf_value(node.state)
            for n in reversed(path):
                n.visits += 1
                n.value_sum += val
        if not root.children:
            return valid_actions[0] if valid_actions else "ACTION1"
        return max(root.children.items(), key=lambda kv: kv[1].visits)[0]


# ==========================================
# COMPONENT 5: LEARNER (count-based novelty only; no training)
# ==========================================

class StateCounter:
    def __init__(self):
        self.counts: Dict[int, int] = {}

    def simhash(self, grid: np.ndarray) -> int:
        if grid.shape[0] > 16 or grid.shape[1] > 16:
            sh, sw = max(1, grid.shape[0] // 16), max(1, grid.shape[1] // 16)
            grid = grid[::sh, ::sw]
        return hash(grid.tobytes())

    def get_bonus(self, grid: np.ndarray) -> float:
        h = self.simhash(grid)
        self.counts[h] = self.counts.get(h, 0) + 1
        return 1.0 / math.sqrt(self.counts[h])

    def peek_bonus(self, grid: np.ndarray) -> float:
        h = self.simhash(grid)
        return 1.0 / math.sqrt(self.counts.get(h, 0) + 1)


# ==========================================
# COMPONENT 5.5: GOAL INFERENCE + REACTIVE PLANNER
# ==========================================
#
# Why this exists: the real ARC-AGI-3 frames are a small logical grid rendered
# SCALED into 64x64 (an 8x8 level -> 8x8 pixel blocks; the avatar jumps `scale`
# pixels per action). The enumerative DSL assumes 1-2px moves, so it never fits
# the physics and the agent falls back to random walk (the ~0.2% failure mode).
#
# ReactivePlanner sidesteps that two ways:
#   1) It LEARNS the avatar's per-action displacement DIRECTLY from the observed
#      centroid shift, so it is scale-invariant (an 8px jump is just (dr,dc)=(8,0)).
#   2) It INFERS the goal as the nearest distinctive object the avatar can reach,
#      learns which colors are obstacles by bumping into them, and moves greedily
#      toward the goal with a wall-escape when a preferred move is blocked.
# No LLM, no GPU -- pure geometry, ~microseconds/step.

class ReactivePlanner:
    STUCK_LIMIT = 3          # centroid unchanged this many steps -> try next-best move

    def __init__(self):
        self.action_disp: Dict[str, Tuple[int, int]] = {}   # action -> (dr, dc) of avatar
        self.avatar_color: Optional[int] = None
        self.obstacle_colors: set = set()
        self.non_goal_colors: set = set()      # distinctive objects that gave NO reward (decoys)
        self._last_center: Optional[Tuple[float, float]] = None
        self._stuck = 0
        self._rank_offset = 0
        self._target: Optional[Tuple[int, int]] = None
        self._visits: Dict[Tuple[int, int], int] = {}   # lattice cell -> times seen
        # Game-type evidence (dual-purpose: gathered from actions we take anyway).
        # explained = a changed frame was a clean avatar translation; unexplained =
        # the frame changed in a way no translation accounts for (rotation, sprite
        # cycling, counters). Mostly-unexplained games are MECHANICAL: driving them
        # with a displacement model burns scored actions on a false premise.
        self.explained_moves = 0
        self.unexplained_changes = 0
        # An action whose displacement REVERSES sign is not a movement key --
        # it's a wrapping selector/cursor (tr87: ACTION3 = (0,+28) ... (0,-7) at
        # the wrap). Real movement keys keep a fixed direction; magnitude may
        # jitter (non-integer render scale) but the sign never flips.
        self.disp_contradictions = 0

    def reset(self):
        # Game-TYPE evidence survives GAME_OVER: whether this game is mechanical
        # is a property of the game, not of the attempt. Wiping it made the
        # classifier flip back to "movement" after every lose() (tr87's 128-move
        # budget), handing the turn back to ExecPlanner on a false premise.
        em, uc, dcon = (self.explained_moves, self.unexplained_changes,
                        self.disp_contradictions)
        self.__init__()
        self.explained_moves, self.unexplained_changes = em, uc
        self.disp_contradictions = dcon

    def new_level(self):
        """Layout changed: forget positional pursuit state but KEEP learned
        physics (avatar color + per-action displacement + obstacle colors)."""
        self._last_center = None
        self._stuck = 0
        self._rank_offset = 0
        self._target = None
        self._visits = {}

    # ---- learning ----
    @staticmethod
    def _centroids_by_color(grid, bg):
        out = {}
        for color in np.unique(grid):
            c = int(color)
            if c == bg:
                continue
            rs, cs = np.nonzero(grid == c)
            out[c] = (rs.mean(), cs.mean(), len(rs))
        return out

    def _infer_moved_color(self, prev, nxt, bg):
        cp, cn = self._centroids_by_color(prev, bg), self._centroids_by_color(nxt, bg)
        best, best_shift = None, 0.0
        for color, (pr, pc, pn) in cp.items():
            if color not in cn:
                continue
            nr, ncc, nn = cn[color]
            shift = abs(pr - nr) + abs(pc - ncc)
            # A single move translates the avatar by ~one avatar-size (the render
            # scale). Reject implausibly large jumps: those are level-swap
            # discontinuities, not moves. Cap at 2x the object's own bbox.
            size = max(1, min(pn, nn)) ** 0.5          # ~side length of the block
            if shift > 3.0 * max(size, self._obj_span(prev, int(color))):
                continue
            # prefer a color that translated while keeping most of its mass
            if shift > best_shift and min(pn, nn) >= 0.5 * max(pn, nn):
                best, best_shift = color, shift
        return best if best_shift >= 1.0 else None

    @staticmethod
    def _obj_span(grid, color):
        rs, cs = np.nonzero(grid == color)
        if len(rs) == 0:
            return 1.0
        return float(max(rs.max() - rs.min() + 1, cs.max() - cs.min() + 1))

    def update(self, prev, action, nxt, encoder, reward: float = 0.0):
        action = base_action(action)
        if prev is None or nxt is None or prev.shape != nxt.shape:
            return
        bg = encoder.detect_background(prev)
        if np.array_equal(prev, nxt):
            # No change: if this action normally moves the avatar, it was blocked.
            if action in self.action_disp and self.avatar_color is not None:
                self._mark_obstacle(prev, action, bg)
            return
        color = self._infer_moved_color(prev, nxt, bg)
        if color is None:
            self.unexplained_changes += 1
            return
        pr, pc, _ = self._centroids_by_color(prev, bg)[color]
        nr, ncc, _ = self._centroids_by_color(nxt, bg)[color]
        dr, dc = int(round(nr - pr)), int(round(ncc - pc))
        if dr == 0 and dc == 0:
            return
        # Diagonal-jump guard: movement actions translate along ONE axis (any
        # off-axis component is centroid-rounding noise <=1px). A strongly
        # diagonal displacement is a scene EVENT (key pickup / recolor shifting
        # the color's centroid), not physics -- learning it corrupts action_disp
        # and the ensuing sign-flips misclassify the game as mechanical
        # (keydoor regression 177 -> 939 actions, 2026-07-16). tr87's genuine
        # wrapping selectors are axis-aligned, so this does not mask them.
        if min(abs(dr), abs(dc)) > 1:
            self.unexplained_changes += 1
            return
        self.explained_moves += 1
        self.avatar_color = color
        if action in self.action_disp:
            odr, odc = self.action_disp[action]
            if dr * odr < 0 or dc * odc < 0:    # nonzero component flipped sign
                self.disp_contradictions += 1
        self.action_disp[action] = (dr, dc)
        # Goal inference from reward: the avatar just stepped onto whatever color
        # sat under its new center in `prev`. If that yielded NO reward, it's a
        # decoy, not the goal -- stop targeting it.
        if reward <= 0:
            arr, acc = np.nonzero(nxt == color)
            if len(arr):
                cr, cc = int(round(arr.mean())), int(round(acc.mean()))
                if 0 <= cr < prev.shape[0] and 0 <= cc < prev.shape[1]:
                    under = int(prev[cr, cc])
                    border = self._border_color(prev)
                    if under not in (bg, border, color) and under not in self.obstacle_colors:
                        self.non_goal_colors.add(under)

    def _mark_obstacle(self, grid, action, bg_hint):
        """A blocked move means a wall sits just past the avatar's leading edge
        (or it hit the grid boundary). Sample the band one cell beyond the bbox
        edge in the move direction; the dominant DISTINCTIVE color there is the
        wall. Never mark the play background or the letterbox border -- doing so
        would make BFS treat every empty cell as blocked (the 0.2%-style stall)."""
        dr, dc = self.action_disp[action]
        mask = (grid == self.avatar_color)
        rs, cs = np.nonzero(mask)
        if len(rs) == 0:
            return
        h, w = grid.shape
        # backgrounds we must NEVER treat as obstacles, derived consistently.
        vals, counts = np.unique(grid, return_counts=True)
        bg = int(vals[counts.argmax()])
        border = self._border_color(grid)
        r0, r1, c0, c1 = int(rs.min()), int(rs.max()), int(cs.min()), int(cs.max())
        step_r, step_c = int(np.sign(dr)), int(np.sign(dc))
        if step_c > 0:
            band = [(r, c1 + 1) for r in range(r0, r1 + 1)]
        elif step_c < 0:
            band = [(r, c0 - 1) for r in range(r0, r1 + 1)]
        elif step_r > 0:
            band = [(r1 + 1, c) for c in range(c0, c1 + 1)]
        elif step_r < 0:
            band = [(r0 - 1, c) for c in range(c0, c1 + 1)]
        else:
            return
        tally: Dict[int, int] = {}
        for r, c in band:
            if 0 <= r < h and 0 <= c < w:
                v = int(grid[r, c])
                if v not in (bg, border, self.avatar_color):
                    tally[v] = tally.get(v, 0) + 1
        if tally:                       # empty band => blocked by boundary, mark nothing
            self.obstacle_colors.add(max(tally, key=tally.get))

    # ---- acting ----
    def ready(self, valid) -> bool:
        if self.avatar_color is None:
            return False
        return sum(1 for a in valid if base_action(a) in self.action_disp) >= 1

    def looks_mechanical(self) -> bool:
        """Game-type classifier verdict, from evidence actions we took anyway:
        True when most frame changes are NOT avatar translations (rotation /
        sprite-cycling / counter games like ls20, tr87, tu93). The threshold
        needs real evidence (>=8 changed frames) so walk-to-goal games that
        merely animate a little are never misclassified."""
        if os.environ.get("ARC_NO_MECH_GATE"):      # debug bisect switch
            return False
        if self.disp_contradictions >= 3:           # wrapping selector, not movement
            return True
        total = self.explained_moves + self.unexplained_changes
        return total >= 8 and self.unexplained_changes / total > 0.75

    def _border_color(self, grid):
        h, w = grid.shape
        if h < 2 or w < 2:
            return None
        border = np.concatenate([grid[0, :], grid[-1, :], grid[1:-1, 0], grid[1:-1, -1]])
        vals, counts = np.unique(border, return_counts=True)
        return int(vals[counts.argmax()])

    def _nearest_target(self, grid, ar, ac):
        border = self._border_color(grid)
        # candidate goal colors: not background-ish, not the avatar, not a known obstacle
        vals, counts = np.unique(grid, return_counts=True)
        bg = int(vals[counts.argmax()])   # most common overall = play-area background
        h, w = grid.shape
        cands = []                        # (structural, nearest_dist, count, cell)
        for color, cnt in zip(vals, counts):
            c = int(color)
            if c in (bg, border, self.avatar_color) or c in self.obstacle_colors \
                    or c in self.non_goal_colors:
                continue
            rs, cs = np.nonzero(grid == c)
            # Span-based candidacy (the real-game fix): a colour stretching across
            # ~the whole board (interior walls, HUD strips) or owning a large mass
            # fraction is STRUCTURE, not a goal object. Demote it -- never pick it
            # while any discrete object color exists -- but keep it as a last
            # resort so sparse boards still yield a target.
            span = max((int(rs.max()) - int(rs.min()) + 1) / h,
                       (int(cs.max()) - int(cs.min()) + 1) / w)
            structural = span >= 0.9 or cnt / grid.size >= 0.25
            d = np.abs(rs - ar) + np.abs(cs - ac)
            i = int(d.argmin())
            cands.append((structural, int(d[i]), int(cnt), (int(rs[i]), int(cs[i]))))
        if not cands:
            return None
        # Non-structural first, then nearest (the validated old order), then rarer.
        cands.sort(key=lambda t: (t[0], t[1], t[2]))
        return cands[0][3]

    def _scale(self, grid):
        """Render scale = size of the avatar block (1 logical cell). The lattice
        step for planning; makes everything scale-invariant."""
        rs, cs = np.nonzero(grid == self.avatar_color)
        if len(rs) == 0:
            return 1
        return max(1, min(rs.max() - rs.min() + 1, cs.max() - cs.min() + 1))

    def _bfs_action(self, grid, valid, ar, ac) -> Optional[str]:
        """Shortest path on the scale-lattice, avoiding learned obstacle colors.
        Each block is a point at its center; a move is valid if the destination
        center isn't an obstacle. Returns the first action of the path."""
        moves = [(a, self.action_disp[base_action(a)]) for a in valid
                 if base_action(a) in self.action_disp]
        if not moves:
            return None
        h, w = grid.shape
        border = self._border_color(grid)
        vals, counts = np.unique(grid, return_counts=True)
        bg = int(vals[counts.argmax()])
        obstacles = set(self.obstacle_colors)
        if border is not None:
            obstacles.add(border)
        targets = {int(v) for v in vals
                   if int(v) not in (bg, border, self.avatar_color)
                   and int(v) not in self.obstacle_colors
                   and int(v) not in self.non_goal_colors}
        if not targets:
            return None

        def blocked(r, c):
            return not (0 <= r < h and 0 <= c < w) or int(grid[r, c]) in obstacles

        start = (int(round(ar)), int(round(ac)))
        from collections import deque
        seen = {start}
        q = deque([(start, None)])
        while q:
            (r, c), first = q.popleft()
            if int(grid[r, c]) in targets and first is not None:
                return first
            for a, (dr, dc) in moves:
                nr, nc = r + dr, c + dc
                if (nr, nc) in seen or blocked(nr, nc):
                    continue
                seen.add((nr, nc))
                q.append(((nr, nc), a if first is None else first))
        return None

    def act(self, grid, valid, encoder) -> Optional[str]:
        if self.avatar_color is None:
            return None
        rs, cs = np.nonzero(grid == self.avatar_color)
        if len(rs) == 0:
            return None
        ar, ac = rs.mean(), cs.mean()
        target = self._nearest_target(grid, ar, ac)
        self._target = target
        if target is None:
            return None
        tr, tc = target

        # Prefer a true shortest path around known obstacles (also near-optimal
        # action count -> better efficiency score). Greedy is the fallback that
        # bumps unknown walls to LEARN they are obstacles.
        bfs = self._bfs_action(grid, valid, ar, ac)
        if bfs is not None:
            self._last_center = (ar, ac)
            return bfs

        moves = [a for a in valid if base_action(a) in self.action_disp]
        if not moves:
            return None

        # stuck detection: centroid hasn't moved -> rotate to the next-best action
        center = (ar, ac)
        if self._last_center is not None and \
                abs(center[0] - self._last_center[0]) < 0.5 and \
                abs(center[1] - self._last_center[1]) < 0.5:
            self._stuck += 1
        else:
            self._stuck = 0
            self._rank_offset = 0
        self._last_center = center
        if self._stuck >= self.STUCK_LIMIT:
            self._rank_offset += 1
            self._stuck = 0

        # Visited-lattice memory: greedy oscillation (A<->B at a wall while BFS
        # has no route) is self-defeating if revisits cost. Rank moves first by
        # how often we've SEEN the destination cell, then by goal distance.
        scale = max(1, self._scale(grid))
        cell = (int(round(ar / scale)), int(round(ac / scale)))
        self._visits[cell] = self._visits.get(cell, 0) + 1

        def rank(a):
            dr, dc = self.action_disp[base_action(a)]
            dest = (int(round((ar + dr) / scale)), int(round((ac + dc) / scale)))
            dist = abs((ar + dr) - tr) + abs((ac + dc) - tc)
            return (self._visits.get(dest, 0), dist)

        moves.sort(key=rank)
        return moves[self._rank_offset % len(moves)]


# ==========================================
# COMPONENT 5.7: EXECUTABLE WORLD MODEL PLANNER (RHAE-critical)
# ==========================================
#
# The two techniques the SOTA "Executable World Models" paper and the competition
# dossier both prescribe, aimed squarely at RHAE's SQUARED action-ratio penalty --
# spending SCORED actions to gather information is fatal (10x human actions -> 1%):
#
#   T1  EXECUTABLE WORLD MODEL + UNSCORED PLAN-THEN-EXECUTE.  Learn the transition
#       dynamics from observation, then search a WHOLE path to the goal INSIDE the
#       model (test-time compute, zero scored actions), commit to it, and execute
#       one vetted step per turn. Re-plan only when reality FALSIFIES the model (a
#       committed step didn't do what the model predicted) -- never oscillate,
#       never take a scored random move once a plan exists.
#
#   T2  FALSIFICATION / CURIOSITY-DRIVEN EXPLORATION.  When no goal path is known
#       yet, don't flail (epsilon-random burns scored actions): steer to the
#       nearest UNVISITED lattice cell so the model is completed in the fewest
#       scored actions -- the second-biggest RHAE lever after not-searching-with-
#       real-actions.
#
# ExecPlanner SUBCLASSES ReactivePlanner: it inherits the validated physics
# learning (per-action displacement, avatar/obstacle/decoy colours, goal-from-
# reward) and ADDS a POSITIONAL wall map (precise, per-cell -- vs the parent's
# coarse per-COLOUR obstacles), a committed plan queue, falsification-driven
# replanning, and directed frontier exploration. It degrades to the parent's
# greedy behaviour when it has nothing better, so the validated toys don't regress.

class ExecPlanner(ReactivePlanner):
    MAX_BFS = 6000          # lattice cells expanded before abandoning a search
    TIE_EPS = 0.05          # tiny random tie-break -> breaks deterministic livelocks
    STUCK_ESCAPE = 6        # steps of no-progress on goal-pursuit before exploring
    ESCAPE_BURST = 4        # steps of directed exploration once triggered
    EXP_TRIES = 2           # re-tests of a disputed (cell, action) before it's an anomaly
    CERT_REPLAY_GAP = 5     # min timeline growth between full-history replays (cost cap)

    def __init__(self):
        super().__init__()
        self.walls: set = set()                    # LOGICAL cells known blocked (bumped)
        self._plan: deque = deque()                # committed (expected_cell, action) steps
        self._cur_scale = 1
        self._last_cell: Optional[Tuple[int, int]] = None
        self._stuck_n = 0
        self._escape_n = 0
        # --- Schema-harness Phase 1 state (CERTIFY / experiments) ---
        self.timeline = None                       # set by MyAgent; append-only ground truth
        self._anomalies: set = set()               # (cell, action): consistently out-of-theory
        self._excused: set = set()                 # timeline indices proven one-off events
        self._cert_key = None                      # theory fingerprint of the last replay
        self._cert_idx = 0                         # timeline entries already replayed
        self._cert_green = True
        self._cert_fail = None                     # (timeline index, entry) counterexample
        self._last_replay_at = -10**9              # timeline length at last full replay
        self._exp = None                           # ((cell, action), tries_left) experiment
        self._unwalled: set = set()                # colours already un-marked this level
        # --- Schema-harness Phase 2 state (LLM theorizer) ---
        self._llm_goal: Optional[Tuple[int, int]] = None   # adopted goal hint (pixel cell)

    def reset(self):
        # __init__ re-runs on reset (via the parent), which would sever the
        # MyAgent-owned timeline reference and forget which (cell, action) pairs
        # were accepted as anomalies -- both are properties of the GAME/level
        # record, not of the attempt (same layout after GAME_OVER -> RESET).
        tl = getattr(self, "timeline", None)
        anom = getattr(self, "_anomalies", set())
        exc = getattr(self, "_excused", set())
        unw = getattr(self, "_unwalled", set())
        lg = getattr(self, "_llm_goal", None)      # same layout after GAME_OVER -> keep
        super().reset()     # re-inits fully but preserves game-type evidence
        self.timeline = tl
        self._anomalies = anom
        self._excused = exc
        self._unwalled = unw
        self._llm_goal = lg

    def new_level(self):
        super().new_level()
        # Layout changed: positional walls + committed plan are stale; physics stays.
        self.walls = set()
        self._plan = deque()
        self._last_cell = None
        self._stuck_n = 0
        self._escape_n = 0
        # Positional certification state belongs to the old layout. The timeline
        # itself is append-only; the level tag keeps old evidence out of scope.
        self._anomalies = set()
        self._excused = set()
        self._cert_key = None
        self._cert_idx = 0
        self._cert_green = True
        self._cert_fail = None
        self._last_replay_at = -10**9
        self._exp = None
        self._unwalled = set()
        self._llm_goal = None      # hint reasoned about the old layout

    # ---- lattice helpers ----
    def _log(self, rc) -> Tuple[int, int]:
        # FLOOR, not round: avatar-block centroids sit exactly on N.5 scale
        # boundaries, and banker's rounding (round(5.5)=6, round(6.5)=6) collapses
        # adjacent grid cells onto one logical cell -- which corrupts the goal-cell
        # match and desyncs the committed plan. Floor gives a clean bijective lattice.
        s = max(1, self._cur_scale)
        return (int(rc[0] // s), int(rc[1] // s))

    def _obstacle_set(self, grid):
        # The play background must NEVER be treated as a wall -- when it equals the
        # letterbox border colour (common: both 0), adding the border would block
        # every cell and BFS would always fail (the border-poisoning stall).
        vals, counts = np.unique(grid, return_counts=True)
        bg = int(vals[counts.argmax()])
        border = self._border_color(grid)
        obs = set(self.obstacle_colors)
        if border is not None and border != bg:
            obs.add(border)
        obs.discard(bg)
        if self.avatar_color is not None:
            obs.discard(self.avatar_color)
        return obs

    def _blocked(self, grid, r, c, obstacles) -> bool:
        h, w = grid.shape
        if not (0 <= r < h and 0 <= c < w):
            return True
        if self._log((r, c)) in self.walls:          # learned by falsification (bump)
            return True
        return int(grid[r, c]) in obstacles           # generalised by colour

    def _bfs_path(self, grid, valid, ar, ac, is_goal) -> Optional[List[str]]:
        """UNSCORED search through the learned model: shortest action path from the
        avatar to the first cell satisfying is_goal, routing around known walls and
        obstacle colours. Returns the whole path (T1 commits to it)."""
        moves = [(a, self.action_disp[base_action(a)]) for a in valid
                 if base_action(a) in self.action_disp]
        if not moves:
            return None
        obstacles = self._obstacle_set(grid)
        # Key the frontier by EXACT pixel (like the validated parent BFS): rounding
        # to logical cells can collapse adjacent lattice steps onto one cell (banker's
        # rounding) and sever connectivity. Walls/visits stay in logical space.
        start = (int(round(ar)), int(round(ac)))
        seen = {start}
        q = deque([(start, [])])
        while q:
            (r, c), path = q.popleft()
            if path and is_goal(r, c):
                return path
            if len(seen) > self.MAX_BFS:
                break
            for a, (dr, dc) in moves:
                # An accepted anomaly = this action's outcome at this cell is
                # KNOWN not to follow the theory -- never route a plan through it.
                if self._anomalies and (self._log((r, c)), base_action(a)) in self._anomalies:
                    continue
                nr, nc = r + dr, c + dc
                if (nr, nc) in seen or self._blocked(grid, nr, nc, obstacles):
                    continue
                seen.add((nr, nc))
                q.append(((nr, nc), path + [a]))
        return None

    def _commit(self, ar, ac, path):
        """Record the plan as (expected_from_cell, action) steps so act() can verify
        the avatar is still where the model predicted before executing each step."""
        seq = deque()
        r, c = int(round(ar)), int(round(ac))
        for a in path:
            seq.append((self._log((r, c)), a))
            dr, dc = self.action_disp[base_action(a)]
            r, c = r + dr, c + dc
        self._plan = seq

    # ---- CERTIFY (Schema Phase 1): full-history backtest of the movement theory ----
    def _explain(self, entry) -> Optional[bool]:
        """Does the CURRENT theory (per-action displacement + positional walls +
        obstacle colours) reproduce this recorded transition?
          None  = out of scope (non-movement action, avatar invisible, scene
                  event, or an accepted anomaly);
          True  = prediction matches reality;
          False = counterexample -- the record falsifies the theory."""
        a = base_action(entry.action)
        if self.avatar_color is None or a not in self.action_disp:
            return None
        m = entry.motion(self.avatar_color)
        if m is None:
            return None
        pr, pc, dr, dc = m
        if min(abs(dr), abs(dc)) > 1:       # diagonal jump = scene event, not motion
            return None
        self._cur_scale = max(1, self._scale(entry.prev))
        if (self._log((int(round(pr)), int(round(pc)))), a) in self._anomalies:
            return None                     # documented exception, out of scope
        ed, ec = self.action_disp[a]
        tr, tc = int(round(pr)) + ed, int(round(pc)) + ec
        if self._blocked(entry.prev, tr, tc, self._obstacle_set(entry.prev)):
            ped, pec = 0, 0                 # theory: move vetoed, avatar stays
        else:
            ped, pec = ed, ec
        # <=1px slack absorbs centroid rounding on multi-pixel sprites.
        # bool(): numpy comparisons yield np.bool_, and np.False_ `is False` is
        # False -- the caller's identity check would silently never fire.
        return bool(abs(dr - ped) <= 1 and abs(dc - pec) <= 1)

    def certify(self) -> bool:
        """The Schema CERTIFY gate: the movement theory must reproduce EVERY
        in-scope recorded transition of the current level before act() may
        commit a multi-step plan through it. Incremental (new entries checked
        as they arrive); any revision of the theory itself -- displacement,
        walls, obstacle colours, anomalies, excusals -- forces a full replay,
        because a revised theory must be re-verified against ALL evidence, not
        just the transition that prompted the revision. Full replays are
        throttled to every CERT_REPLAY_GAP entries so displacement jitter can't
        turn certification into an O(n^2) per-step cost."""
        if os.environ.get("ARC_NO_CERT"):   # debug bisect switch: always green
            return True
        tl = self.timeline
        if tl is None or not tl.entries:
            return True
        level = tl.entries[-1].level
        key = (self.avatar_color, tuple(sorted(self.action_disp.items())),
               frozenset(self.obstacle_colors), frozenset(self.walls),
               len(self._anomalies), len(self._excused), level)
        if key != self._cert_key and \
                len(tl.entries) - self._last_replay_at >= self.CERT_REPLAY_GAP:
            self._cert_key = key
            self._cert_idx = 0
            self._cert_fail = None
            self._cert_green = True
            self._last_replay_at = len(tl.entries)
        for i in range(self._cert_idx, len(tl.entries)):
            e = tl.entries[i]
            if e.level != level or i in self._excused:
                continue
            if self._explain(e) is False:
                self._cert_green = False
                self._cert_fail = (i, e)
        self._cert_idx = len(tl.entries)
        return self._cert_green

    def _experiment(self, grid, valid, ar, ac) -> Optional[str]:
        """Discriminating experiment (Schema 'action for discovery'): when the
        record contradicts the theory, the single most informative real action
        is to RE-RUN the disputed (cell, action) -- one scored step splits
        'one-off scene event' (excuse the recorded entry, theory stands) from
        'consistent rule the theory lacks' (after EXP_TRIES reproductions the
        pair becomes an anomaly: certification puts it out of scope and the
        planner stops routing through it). Steps toward the disputed cell are
        taken singly (no committed queue -- the theory is red here)."""
        if self._cert_fail is None:
            return None
        _, e = self._cert_fail
        a = base_action(e.action)
        m = e.motion(self.avatar_color) if self.avatar_color is not None else None
        if m is None or a not in self.action_disp:
            self._cert_fail = None
            return None
        cell = self._log((int(round(m[0])), int(round(m[1]))))
        if self._exp is None or self._exp[0] != (cell, a):
            self._exp = ((cell, a), self.EXP_TRIES)
        if self._exp[1] <= 0:
            # Reproduced EXP_TRIES times: a consistent rule outside the theory.
            # Joint state+rule revision (VIGA / WorldCoder lineage, Schema sec.4):
            # before accepting an anomaly, check whether the counterexample
            # indicts the REPRESENTATION -- the colour-obstacle / wall grounding
            # -- rather than the displacement rule. Only when no revision
            # applies does the (cell, action) become a behavioural anomaly.
            revised = self._revise_representation(e)
            self._exp = None
            self._cert_fail = None
            if not revised:
                self._anomalies.add((cell, a))
            return None
        if self._log((int(round(ar)), int(round(ac)))) == cell:
            exp_a = next((v for v in valid if base_action(v) == a), None)
            if exp_a is not None:
                self._exp = (self._exp[0], self._exp[1] - 1)
                return exp_a
            return None
        path = self._bfs_path(grid, valid, ar, ac,
                              lambda r, c: self._log((r, c)) == cell)
        return path[0] if path else None

    def _revise_representation(self, entry) -> bool:
        """VIGA/WorldCoder joint revision: a reproduced counterexample can mean
        the STATE GROUNDING is wrong, not the transition rule.
          - Theory said BLOCKED but the avatar moved through: the colour marked
            as an obstacle is actually walkable (Schema's 'the cart is a valid
            landing cell') -> un-mark it (once per level) / drop the stale wall.
          - Theory said FREE but the avatar stayed: an invisible barrier the
            colour layer can't see -> record a positional wall.
        Returns True when a revision was made; certification then re-verifies
        the WHOLE record against the revised grounding (full replay)."""
        a = base_action(entry.action)
        m = entry.motion(self.avatar_color) if self.avatar_color is not None else None
        if m is None or a not in self.action_disp:
            return False
        pr, pc, dr, dc = m
        ed, ec = self.action_disp[a]
        self._cur_scale = max(1, self._scale(entry.prev))
        tr, tc = int(round(pr)) + ed, int(round(pc)) + ec
        h, w = entry.prev.shape
        if not (0 <= tr < h and 0 <= tc < w):
            return False
        blocked_pred = self._blocked(entry.prev, tr, tc, self._obstacle_set(entry.prev))
        moved = max(abs(dr), abs(dc)) > 1
        if blocked_pred and moved:
            c = int(entry.prev[tr, tc])
            if c in self.obstacle_colors and c not in self._unwalled:
                self.obstacle_colors.discard(c)
                self._unwalled.add(c)
                self.walls.discard(self._log((tr, tc)))
                return True
            wcell = self._log((tr, tc))
            if wcell in self.walls:
                self.walls.discard(wcell)
                return True
            return False
        if not blocked_pred and not moved:
            self.walls.add(self._log((tr, tc)))
            return True
        return False

    # ---- Schema Phase 2: LLM theorizer proposals, gated by the backtest ----
    def _backtest_mismatches(self) -> int:
        """Full-replay mismatch count of the CURRENT theory against the current
        level's timeline (excused entries skipped). The theorizer's acceptance
        metric: a proposal must not explain LESS of the record than the theory
        it replaces."""
        tl = self.timeline
        if tl is None or not tl.entries:
            return 0
        level = tl.entries[-1].level
        n = 0
        for i, e in enumerate(tl.entries):
            if e.level != level or i in self._excused:
                continue
            if self._explain(e) is False:
                n += 1
        return n

    def propose_theory(self, proposal: dict, grid) -> bool:
        """Adopt an LLM-proposed theory revision ONLY if the full-history
        backtest does not get worse (Schema: the model is provisional, the
        Timeline is ground truth -- a proposal the record contradicts is
        reverted wholesale). Grounding edits are applied tentatively; the goal
        hint needs no backtest because it only REDIRECTS planning, which stays
        green-gated and per-step vetted. Returns True if anything was adopted."""
        if not isinstance(proposal, dict):
            return False
        adopted = False
        snap = (set(self.obstacle_colors), set(self.non_goal_colors), set(self.walls))
        before = self._backtest_mismatches()
        changed = False
        for c in proposal.get("not_obstacle_colors") or []:
            try:
                c = int(c)
            except (TypeError, ValueError):
                continue
            if c in self.obstacle_colors:
                self.obstacle_colors.discard(c)
                changed = True
        for c in proposal.get("obstacle_colors") or []:
            try:
                c = int(c)
            except (TypeError, ValueError):
                continue
            if c != self.avatar_color and c not in self.obstacle_colors:
                self.obstacle_colors.add(c)
                changed = True
        for c in proposal.get("decoy_colors") or []:
            try:
                c = int(c)
            except (TypeError, ValueError):
                continue
            if c != self.avatar_color:
                self.non_goal_colors.add(c)      # only affects target choice
        if changed:
            after = self._backtest_mismatches()
            if after > before:
                # The record contradicts the proposal: revert EVERYTHING it
                # touched (decoys included -- an untrustworthy theory gets no
                # partial credit).
                self.obstacle_colors, self.non_goal_colors, self.walls = snap
            else:
                adopted = True
                # Revised grounding => certification must re-verify the whole
                # record before the next multi-step commit.
                self._cert_key = None
                self._last_replay_at = -10**9
        g = proposal.get("goal")
        if isinstance(g, (list, tuple)) and len(g) == 2:
            try:
                r, c = int(g[0]), int(g[1])
            except (TypeError, ValueError):
                r = c = -1
            h, w = grid.shape
            if 0 <= r < h and 0 <= c < w:
                self._llm_goal = (r, c)
                adopted = True
        return adopted

    # ---- learning: positional wall on a falsified (blocked) move ----
    def update(self, prev, action, nxt, encoder, reward: float = 0.0):
        super().update(prev, action, nxt, encoder, reward)   # physics + colour obstacles
        if prev is None or nxt is None or prev.shape != nxt.shape:
            return
        a = base_action(action)
        if a in self.action_disp and self.avatar_color is not None and np.array_equal(prev, nxt):
            # The move was VETOED by the world -> the cell just past the avatar's
            # centre is a wall. Record it POSITIONALLY (precise) and drop the plan;
            # act() will replan around it. This is the falsification -> replan loop.
            self._cur_scale = max(1, self._scale(prev))
            rs, cs = np.nonzero(prev == self.avatar_color)
            if len(rs):
                dr, dc = self.action_disp[a]
                self.walls.add(self._log((rs.mean() + dr, cs.mean() + dc)))
            self._plan.clear()
        # Resolve a pending discriminating experiment on the outcome of its
        # disputed action (this transition is the newest timeline entry).
        if (self._exp is not None and self.timeline is not None
                and self.timeline.entries and self.avatar_color is not None):
            (cell, ea), _tries = self._exp
            if a == ea:
                i = len(self.timeline.entries) - 1
                e = self.timeline.entries[i]
                m = e.motion(self.avatar_color)
                if m is not None and \
                        self._log((int(round(m[0])), int(round(m[1])))) == cell:
                    if self._explain(e) is not False:
                        # Theory held on the re-test: the recorded contradiction
                        # was a one-off scene event -- excuse that entry only.
                        if self._cert_fail is not None:
                            self._excused.add(self._cert_fail[0])
                        self._exp = None
                        self._cert_fail = None

    # ---- acting: committed plan -> plan to goal -> frontier -> greedy ----
    def act(self, grid, valid, encoder) -> Optional[str]:
        if self.avatar_color is None:
            return None
        rs, cs = np.nonzero(grid == self.avatar_color)
        if len(rs) == 0:
            return None
        ar, ac = rs.mean(), cs.mean()
        # CERTIFY before anything else (mutates _cur_scale per replayed entry,
        # so re-derive the live scale after it).
        green = self.certify()
        self._cur_scale = max(1, self._scale(grid))
        cur = self._log((int(round(ar)), int(round(ac))))
        valid_moves = {a for a in valid if base_action(a) in self.action_disp}
        if not green:
            # Reality outranks the model: an outstanding counterexample voids
            # the committed queue (Schema's per-step mismatch rule).
            self._plan.clear()

        # Progress tracking: a genuine step changes our logical cell. If goal-pursuit
        # leaves the cell unchanged for STUCK_ESCAPE steps it is livelocked (e.g. a
        # non-integer render scale makes the fixed-displacement model oscillate).
        if cur == self._last_cell:
            self._stuck_n += 1
        else:
            self._stuck_n = 0
        self._last_cell = cur
        self._visits[cur] = self._visits.get(cur, 0) + 1

        # (1) T1 -- consume the committed plan WHILE reality still matches the model.
        if self._plan:
            exp_cell, a = self._plan[0]
            if exp_cell == cur and a in valid_moves:
                self._plan.popleft()
                return a
            self._plan.clear()          # desync (interrupt / falsified) -> replan below

        # (1b) Theory red -> spend the next scored action on the discriminating
        #      experiment (re-test the disputed transition), not on pursuing a
        #      goal through a model the record has already falsified.
        if not green:
            exp = self._experiment(grid, valid, ar, ac)
            if exp is not None:
                return exp

        # Schema Phase 2: an adopted theorizer GOAL HINT outranks the local
        # nearest-distinctive-colour heuristic (the documented real-game gap:
        # layered games' goals are not "nearest odd colour"). The hint only
        # redirects planning -- commits stay green-gated and per-step vetted --
        # and is dropped the moment reality discredits it (reached without a
        # level-up, or pursuit livelocks below).
        target = None
        if self._llm_goal is not None:
            if self._log(self._llm_goal) == cur:
                self._llm_goal = None       # arrived, no level-up -> hint was wrong
            else:
                target = self._llm_goal
        if target is None:
            target = self._nearest_target(grid, ar, ac)
        self._target = target

        # T2 -- livelock escape: goal-pursuit stopped making progress, so the belief
        # "I have a path to the goal" is FALSIFIED. Spend a short DIRECTED-exploration
        # burst to reach new ground (not epsilon-random -- RHAE squares wasted actions),
        # then resume pursuit from a fresh position/angle.
        if self._escape_n == 0 and self._stuck_n >= self.STUCK_ESCAPE:
            self._escape_n = self.ESCAPE_BURST
            self._plan.clear()
            self._llm_goal = None   # pursuit livelocked -> the hint is falsified too
        exploring = self._escape_n > 0
        if exploring:
            self._escape_n -= 1

        # (2) T1 -- goal known & not escaping: plan a WHOLE path to it through the
        #     learned model (unscored search) and commit. Optimal actions -> best RHAE.
        if target is not None and not exploring:
            tcell = self._log(target)
            path = self._bfs_path(grid, valid, ar, ac,
                                  lambda r, c: self._log((r, c)) == tcell)
            if path:
                if green:
                    self._commit(ar, ac, path)
                    return self._plan.popleft()[1]
                # ONLY PLAN WHEN GREEN: the theory has an outstanding
                # counterexample, so a committed queue would be built on a
                # falsified model -- take one vetted step instead; every real
                # transition re-enters certification before the next step.
                return path[0]
            # Goal known but no MODELLED path (unknown walls / lattice offset): defer
            # to the parent's validated greedy pursuit -- it drives toward the target,
            # bumping and LEARNING obstacles, with stuck-rotation to escape livelocks.
            return super().act(grid, valid, encoder)

        # (3) T2 -- no goal yet, or escaping a livelock: steer to the nearest UNVISITED
        #     cell (directed exploration) so the model is completed in the fewest
        #     SCORED actions.
        path = self._bfs_path(grid, valid, ar, ac,
                              lambda r, c: self._visits.get(self._log((r, c)), 0) == 0)
        if path:
            if green:
                self._commit(ar, ac, path)
                return self._plan.popleft()[1]
            return path[0]      # red: single vetted step, no committed queue

        # (4) Fallback -- whole reachable region explored / boxed in: parent's
        #     least-visited greedy with stuck-rotation breaks the livelock.
        self._escape_n = 0
        return super().act(grid, valid, encoder)


# ==========================================
# COMPONENT 5.6: CLICK PLANNER (ACTION6 games)
# ==========================================
#
# 10 of the 25 real games are click-only (available_actions=[6]) -- boards of
# tiles where clicking mutates state (e.g. ft09 cycles neighbor colors,
# lights-out style). ReactivePlanner cannot act there, and the old fallback
# emitted a bare "ACTION6" that resolved to clicking the grid CENTER forever.
# This planner clicks like an experimenter instead:
#   1) segment the frame into objects -> candidate click targets,
#   2) click every candidate at least once (discovery),
#   3) learn per-cell effect (frame changed? reward?), then keep clicking
#      ACTIVE cells with the LOWEST click count -- uniform coverage of the
#      reactive surface beats hammering one tile, and for cyclic mechanics
#      revisiting is required, so cells are never blacklisted,
#   4) a small epsilon of uniform-random clicks covers non-object targets.

class ClickPlanner:
    """Policy for click-only games (available_actions == [6]).

    There is no single click mechanic across the real games -- probing shows at
    least three families: a LOCALIZED reactive region with lose-on-bad-click
    (ft09), a handful of SPARSE inert-elsewhere buttons (lp85: 2 of 4096 cells),
    and near-universal reactivity / cursor games (r11l, vc33). Frame-change is a
    game-specific, noisy signal (vc33 changes on every click); the only reliable
    reward is level completion, which is SPARSE. So the planner is built around
    coverage + credit assignment that PERSISTS across the many GAME_OVER restarts
    a click game induces, and around learning which cells are lethal.
    """
    MAX_TARGETS = 48   # candidate objects per decision (guards vs noisy frames)
    QUANT = 4          # px; cells sharing a quantized key are "the same target"

    # -- click world-model / state-graph search (the click analogue of
    #    ReactivePlanner). For small-action-space deterministic puzzles (e.g. lp85:
    #    2 buttons slide blocks toward goals) the env IS a replayable simulator: a
    #    live RESET restores the level's exact initial state AND refills the budget
    #    (verified: level_reset() leaves action_count>0, so back-to-back RESETs never
    #    full-reset the score). So we do a breadth-first search over BOARD STATES:
    #    each state = a frame hash, each edge = a button click discovered by replaying
    #    "RESET + path-to-parent + button" and hashing the result. Deduping by state
    #    (unlike a blind product-of-buttons enumeration) collapses no-op re-clicks and
    #    the wall-parked fixed points lp85 slides into, and finds the SHORTEST winning
    #    path. Winning is detected implicitly -- a winning click bumps the score, the
    #    outer loop calls new_level(), search restarts fresh on the next layout.
    MIN_BUTTONS_FOR_SEARCH = 2 # a real button puzzle has >=2 controls; a lone
                               # reactive cell (e.g. lp85 detects 1) is not a
                               # searchable action space -> stay on coverage, which
                               # is the validated floor. Prevents the search from
                               # displacing a coverage win it cannot improve on.
    MAX_BUTTONS = 6            # only search when the reactive set is this small
    MIN_DISCOVER_CLICKS = 60   # coverage clicks before trusting the button set
    MAX_REPLAY = 30            # deepest button-path to replay from RESET (< the ~52
                               # button-click/level budget; only button clicks cost)
    MAX_SEARCH_STATES = 250    # hard cap on distinct boards before bailing to coverage.
                               # A true button puzzle's early levels have few distinct
                               # boards; a blow-up past this means the frame hash is NOT
                               # a stable state signature (some games render a live
                               # step-counter INTO the 64x64 frame, so every board hashes
                               # unique and dedup fails). Bailing keeps the coverage floor.

    def __init__(self):
        # quantized cell -> [clicks, frame-changes, rewards, losses]
        self._stats: Dict[Tuple[int, int], List[int]] = {}
        # quantized cell -> the EXACT (r,c) that actually changed the frame there.
        # Buttons are often reactive only at a specific pixel, not the cell centre,
        # so the search MUST replay the exact coord (using the centre clicks a dead
        # spot and the whole sequence no-ops).
        self._reactive: Dict[Tuple[int, int], Tuple[int, int]] = {}
        self._buttons: List[Tuple[int, int]] = []     # exact coords, level-persistent
        self._reset_search(keep_buttons=False)

    def _reset_search(self, keep_buttons: bool):
        if not keep_buttons:
            self._buttons = []
            self._reactive = {}
        # Always start a level in COVERAGE (discover), NEVER in search -- even when
        # buttons are already known. Entering search emits a RESET as the level's
        # first action, and the engine treats a RESET while its internal
        # action_count==0 (true at the start of every level, since advancing a level
        # zeroes it) as a FULL reset: that wipes levels_completed back to 0, silently
        # throwing away every level already won. This was the lp85 0/8 regression --
        # coverage won level 1, new_level() re-entered search, its first RESET
        # full-reset the score. Coverage-first guarantees >=MIN_DISCOVER_CLICKS real
        # clicks before any search RESET, so RESET does a level_reset (keeps score).
        self._phase = "discover"
        self._total_clicks = 0
        self._search_done = False
        # -- state-graph BFS bookkeeping (populated on _enter_search) --
        self._G: Dict[int, Dict[int, int]] = {}          # state -> {button_idx: child_state}
        self._dist: Dict[int, Tuple[int, ...]] = {}       # state -> shortest button-path from root
        self._queue: "deque[int]" = deque()               # BFS frontier of states
        self._root: Optional[int] = None
        self._dead_edges: set = set()                     # (state, button_idx) that lose / over-deep
        self._plan: List[Any] = []                        # current replay: ["RESET", int, int, ...]
        self._plan_i = 0
        self._pending: Optional[Tuple] = None             # ("root",) | ("expand", parent, button_idx)

    def new_level(self):
        # Real level-up: the board layout changes, so per-cell stats are stale -> wipe.
        # But button POSITIONS are UI arrows, level-invariant in practice: KEEP them so
        # L2+ skip re-discovery and search immediately (if wrong, search exhausts and
        # coverage re-discovers from the fresh stats).
        self._stats = {}
        self._reset_search(keep_buttons=True)

    def reset(self):
        # Intra-level restart (after GAME_OVER). The layout is IDENTICAL on restart,
        # so which cells are reactive / safe / lethal is still valid -- KEEP it.
        # (The old code aliased reset->new_level, wiping this every loss, which made
        # the planner permanently amnesiac on lose-happy games like ft09.)
        if self._phase == "search" and self._pending is not None and self._pending[0] == "expand":
            # A GAME_OVER interrupted the replay of "RESET + path + b". The path prefix
            # was already validated as non-lethal & deterministic, so the death is on
            # the final edge being tested: mark (parent, b) dead so search never retries
            # it. The framework has already emitted its own RESET, so drop the aborted
            # plan; the next act() builds the next plan from the frontier.
            _, parent, b = self._pending
            self._G.setdefault(parent, {})[b] = -1        # sentinel: explored, leads nowhere
            self._dead_edges.add((parent, b))
        if self._phase == "search":
            self._pending = None
            self._plan = []
            self._plan_i = 0

    @property
    def searching(self) -> bool:
        return bool(self._buttons) and self._phase == "search" and not self._search_done

    def _key(self, r: int, c: int) -> Tuple[int, int]:
        return (int(r) // self.QUANT, int(c) // self.QUANT)

    BUTTON_CLUSTER_DIST = 10   # px; reactive coords within this are ONE button sprite

    def _detect_buttons(self) -> List[Tuple[int, int]]:
        """Discrete actions of a button-driven puzzle: ONE representative exact coord
        per physical button sprite. A single button occupies several QUANT cells
        (lp85's LEFT slide is a 4-cell sprite at cols 2-6, rows 30-34), so the raw
        `self._reactive` coords over-count buttons ~3x. That inflation is fatal to the
        search: branching factor n caps the reachable depth via n**depth <=
        SEQ_FANOUT_CAP (6 fake buttons -> depth 3 only; 2 real buttons -> depth 8), so
        a 5-slide puzzle is unsearchable until the duplicates are merged. Cluster the
        reactive coords by spatial proximity (Chebyshev < BUTTON_CLUSTER_DIST) and
        return one coord per cluster. Replay still uses the EXACT pixel (cell centres
        are frequently dead spots)."""
        coords = sorted(self._reactive.values())
        clusters: List[List[Tuple[int, int]]] = []
        for rc in coords:
            for cl in clusters:
                if any(max(abs(rc[0] - m[0]), abs(rc[1] - m[1])) < self.BUTTON_CLUSTER_DIST
                       for m in cl):
                    cl.append(rc)
                    break
            else:
                clusters.append([rc])
        # representative = the cluster member closest to the cluster centroid (a
        # robust interior pixel, least likely to sit on a sprite edge / dead border).
        reps = []
        for cl in clusters:
            cr = sum(m[0] for m in cl) / len(cl)
            cc = sum(m[1] for m in cl) / len(cl)
            reps.append(min(cl, key=lambda m: (m[0] - cr) ** 2 + (m[1] - cc) ** 2))
        return sorted(reps)

    @staticmethod
    def _hash(grid: np.ndarray) -> int:
        return hash((grid.shape, grid.tobytes()))

    _DBG = bool(os.environ.get("CP_DEBUG"))

    def _enter_search(self, buttons: List[Tuple[int, int]]):
        """Switch from coverage to state-graph BFS. The first plan is a lone RESET
        that establishes the ROOT state (the level's initial board)."""
        self._buttons = buttons
        self._phase = "search"
        self._search_done = False
        self._G = {}
        self._dist = {}
        self._queue = deque()
        self._root = None
        self._dead_edges = set()
        self._pending = ("root",)
        self._plan = ["RESET"]
        self._plan_i = 0
        self._search_actions = 0
        if self._DBG:
            import sys
            print(f"[CP] enter_search buttons={buttons} after {self._total_clicks} clicks",
                  file=sys.stderr)

    def _process_result(self, h: int):
        """A replay plan finished; `h` is the hash of the board it produced. Record
        the root, or the edge (parent -> h) that the tested button just revealed."""
        kind = self._pending[0]
        if kind == "root":
            self._root = h
            self._dist[h] = ()
            self._queue.append(h)
        elif kind == "expand":
            _, parent, b = self._pending
            self._G.setdefault(parent, {})[b] = h
            if h not in self._dist:                        # a NEW board -> enqueue to expand
                self._dist[h] = self._dist[parent] + (b,)
                self._queue.append(h)
            # else: no-op / merges into a known state -> dedup drops it (this is the
            # win over blind product-BFS: parked/looping states are visited once).
        self._pending = None

    def _build_next_plan(self) -> bool:
        """Pick the next (state, untried-button) frontier edge to probe and stage its
        replay plan ["RESET", *path_to_state, button]. Returns False when the whole
        reachable graph is exhausted (search gives up -> coverage takes over)."""
        nb = len(self._buttons)
        if len(self._dist) > self.MAX_SEARCH_STATES:
            return False                                   # hash-noise blow-up -> coverage
        while self._queue:
            s = self._queue[0]
            tried = set(self._G.get(s, {}).keys())
            path = self._dist[s]
            b = None
            if len(path) + 1 <= self.MAX_REPLAY:
                for cand in range(nb):
                    if cand in tried or (s, cand) in self._dead_edges:
                        continue
                    b = cand
                    break
            if b is None:
                # fully expanded, or too deep to replay any further -> retire the node.
                self._queue.popleft()
                continue
            self._plan = ["RESET", *path, b]
            self._plan_i = 0
            self._pending = ("expand", s, b)
            return True
        return False

    def _search_step(self, grid: np.ndarray) -> Optional[str]:
        """Emit one action of the current replay plan. `grid` is the board resulting
        from the previous action; when a plan's last click has landed we hash it to
        learn the edge, then stage the next plan. Returns None when search is spent."""
        # A finished plan's result has arrived -> learn from it.
        if self._pending is not None and self._plan_i >= len(self._plan):
            self._process_result(self._hash(grid))
        # Need a fresh plan?
        if self._plan_i >= len(self._plan):
            if not self._build_next_plan():
                self._search_done = True               # graph exhausted -> pure coverage
                self._phase = "discover"
                if self._DBG:
                    import sys
                    print(f"[CP] search EXHAUSTED states={len(self._dist)} "
                          f"edges={sum(len(v) for v in self._G.values())} "
                          f"dead={len(self._dead_edges)} actions={self._search_actions}",
                          file=sys.stderr)
                return None
        step = self._plan[self._plan_i]
        self._plan_i += 1
        self._search_actions += 1
        if step == "RESET":
            return "RESET"                             # replay from the level's initial state
        b = self._buttons[step]
        return f"ACTION6_r{b[0]}_c{b[1]}"

    @staticmethod
    def _parse(action_str: str) -> Optional[Tuple[int, int]]:
        if not action_str.startswith("ACTION6_"):
            return None
        try:
            parts = action_str.split("_")
            return int(parts[1][1:]), int(parts[2][1:])
        except Exception:
            return None

    def update(self, action_str: str, changed: bool, reward: float):
        rc = self._parse(action_str)
        if rc is None:
            return
        st = self._stats.setdefault(self._key(*rc), [0, 0, 0, 0])
        st[0] += 1
        st[1] += 1 if changed else 0
        st[2] += 1 if reward > 0 else 0
        self._total_clicks += 1
        if changed:
            self._reactive[self._key(*rc)] = rc      # exact coord that worked here

    def mark_loss(self, action_str: str):
        """The click in `action_str` immediately preceded a GAME_OVER: blame its
        cell so future ranking backs off from it (persists across the restart)."""
        rc = self._parse(action_str)
        if rc is None:
            return
        st = self._stats.setdefault(self._key(*rc), [0, 0, 0, 0])
        st[3] += 1

    def is_lethal(self, r: int, c: int) -> bool:
        """True if a click at/near (r,c) has previously caused a GAME_OVER. Lets an
        external planner (e.g. an LLM click plan) refuse a target coverage already
        learned is fatal, so a suggested sequence can never score below the floor."""
        st = self._stats.get(self._key(r, c))
        return bool(st and st[3] > 0)

    def inert_cell(self) -> Optional[Tuple[int, int]]:
        """A representative coord of a cell that's been clicked, NEVER changed the
        frame, and never lost -- clicking it is a safe no-op that holds the board
        state STILL. Used to idle without mutating the board while a background LLM
        plan (reasoned about the current board) is still generating, so the plan
        lands on the same board it was computed from. None if none learned yet."""
        best = None
        for (kr, kc), st in self._stats.items():
            clicks, changes, rewards, losses = st
            if clicks > 0 and changes == 0 and losses == 0:
                cand = (kr * self.QUANT + self.QUANT // 2, kc * self.QUANT + self.QUANT // 2)
                # prefer the most-confirmed-inert cell (highest click count)
                if best is None or clicks > best[0]:
                    best = (clicks, cand)
        return best[1] if best else None

    def _lattice(self, h: int, w: int) -> List[Tuple[int, int]]:
        """Cell-centre grid at QUANT resolution -- guarantees sparse-button games
        (whole interactive surface = a couple of cells) get systematically swept,
        which pure object-centre targeting would miss."""
        half = self.QUANT // 2
        return [(r, c) for r in range(half, h, self.QUANT)
                for c in range(half, w, self.QUANT)]

    def act(self, grid: np.ndarray, encoder: StateEncoder, epsilon: float) -> str:
        # If a small button set has been discovered, drive the state-graph BFS
        # (env-as-simulator via RESET). Otherwise cover the board to discover it.
        if self.searching:
            step = self._search_step(grid)
            if step is not None:
                return step                            # None => search just exhausted; fall through
        if (not self._search_done and self._phase == "discover"
                and self._total_clicks >= self.MIN_DISCOVER_CLICKS):
            buttons = self._detect_buttons()
            if self.MIN_BUTTONS_FOR_SEARCH <= len(buttons) <= self.MAX_BUTTONS:
                self._enter_search(buttons)
                step = self._search_step(grid)
                if step is not None:
                    return step
        return self._coverage_click(grid, encoder, epsilon)

    def _coverage_click(self, grid: np.ndarray, encoder: StateEncoder, epsilon: float) -> str:
        h, w = grid.shape
        # Exploration: bias toward a lattice cell we've NEVER clicked (remembered
        # coverage) rather than un-remembered uniform-random that re-hits cells.
        if random.random() < epsilon:
            untried = [rc for rc in self._lattice(h, w)
                       if self._key(*rc) not in self._stats]
            if untried:
                r, c = random.choice(untried)
                return f"ACTION6_r{r}_c{c}"
            return f"ACTION6_r{random.randrange(h)}_c{random.randrange(w)}"

        # Candidates: object centres (small first -- the board panel is the biggest
        # object and rarely interactive) UNION the coverage lattice.
        objs = sorted(encoder.objects(grid), key=lambda o: o.size)[:self.MAX_TARGETS]
        cands = [(int(round(o.center[0])), int(round(o.center[1]))) for o in objs]
        cands += self._lattice(h, w)
        seen: set = set()
        uniq: List[Tuple[int, int]] = []
        for rc in cands:
            k = self._key(*rc)
            if k in seen:
                continue
            seen.add(k)
            uniq.append(rc)
        if not uniq:
            uniq = [(h // 2, w // 2)]

        def rank(rc):
            st = self._stats.get(self._key(*rc))
            if st is None:
                return (0, 0.0, 0, random.random())          # never tried: coverage
            clicks, changes, rewards, losses = st
            loss_rate = losses / clicks if clicks else 0.0
            if rewards > 0:                                    # exploit what paid off
                return (-1, -rewards, loss_rate, random.random())
            dead = 0 if changes > 0 else 1                     # changed-safe before inert
            # non-rewarding tried cells: least lethal, then least explored, last.
            return (1 + dead, loss_rate, clicks, random.random())

        r, c = min(uniq, key=rank)
        return f"ACTION6_r{r}_c{c}"


# ==========================================
# COMPONENT 5.8: STATE GRAPH + DEAD-ACTION TRACKER
# ==========================================
#
# The games are DETERMINISTIC: the same action from the same frame always
# yields the same next frame, so an explicit graph of observed transitions is
# an EXACT world model -- no learning error, no training. It makes exploration
# DIRECTED: a (state, action) pair whose outcome is already recorded is never
# re-spent, and when the current state is exhausted the agent walks already-
# proven edges to the nearest state that still has an untested action (Blind
# Squirrel's whole strategy -- 6.7% in the preview with nothing else). RHAE
# squares wasted actions, so eliminating re-tests is pure gain. If a hidden
# timer/counter makes states never repeat, every state is novel and the graph
# degrades gracefully to "try each action once per new state".
#
# ACTION6 clicks are excluded throughout: they are parameterized by pixel, so
# per-state bookkeeping over the bare action is meaningless -- the ClickPlanner
# owns that surface with its own per-cell stats.

class DeadActionTracker:
    """Flags actions that have NEVER changed the frame anywhere in the level.

    Per-(state, action) inertness is already captured by the graph (a no-change
    transition is a self-loop edge, never re-suggested). What the graph cannot
    see is an action that is inert from EVERY state (e.g. an unused ACTION5 in
    this game): after MIN_TRIALS no-change outcomes with zero changes ever,
    exploration stops spending scored actions on it. One observed change
    whitelists the action for the rest of the level.
    """
    MIN_TRIALS = 3

    def __init__(self):
        self._no_change: Dict[str, int] = {}
        self._changed: set = set()

    def update(self, action: str, changed: bool):
        a = base_action(action)
        if a == "ACTION6":
            return                      # clicks: per-cell, ClickPlanner's domain
        if changed:
            self._changed.add(a)
        else:
            self._no_change[a] = self._no_change.get(a, 0) + 1

    def is_dead(self, action: str) -> bool:
        a = base_action(action)
        if a == "ACTION6":
            return False
        return a not in self._changed and self._no_change.get(a, 0) >= self.MIN_TRIALS


class StateGraph:
    """Exact directed graph of observed frames and transitions (per level).

    Persists across intra-level RESETs on purpose: the level layout is
    unchanged, so every recorded edge is still true and a restart replays into
    known territory instead of rediscovering it with scored actions.

    HUD/TIMER NORMALIZATION: games like tr87 (move-budget bar) and ls20 (timer)
    mutate a few fixed cells on EVERY action, so raw frame hashes never repeat
    and the graph degrades to "everything is novel". Cells that change in most
    transitions are HUD, not world -- they are learned online and masked out of
    the state hash. Raw frames and ensure-events are logged so the whole graph
    can be REBUILT under a new mask (exactness is preserved, nothing is guessed).
    """
    MAX_BFS_STATES = 4000               # 1500-action budget -> never near this
    MASK_RATE = 0.5                     # cell changing in >50% of transitions = HUD
    MASK_CHECK_EVERY = 25               # recompute mask (maybe rebuild) this often
    MASK_MIN_PER_ACT = 5                # samples before an action confirms/vetoes
    # HUD strips (timer / step-counter / budget bar) are CLOCKS: they tick as a
    # function of elapsed actions, NOT of what the agent does -- so the gaps
    # between a HUD line's changes are perfectly regular (tr87's bar: exactly
    # every 2nd transition; a timer: every transition). World lines (avatar
    # paths, cycled sprites, cursors) change when specific actions are taken,
    # which under randomized balanced exploration gives irregular gaps. A line
    # is HUD iff it changed often enough AND one gap length dominates.
    MASK_LINE_MIN_CHG = 10              # changes before periodicity is judged
    MASK_LINE_PERIODIC = 0.9            # modal gap must cover >=90% of gaps
    MAX_LOG = 4000                      # raw-log cap (budget is 1500 actions)

    def __init__(self):
        self.adj: Dict[int, Dict[str, int]] = {}    # hash -> {action: next_hash}
        self.untested: Dict[int, set] = {}          # hash -> actions never tried there
        self._ensures: List[Tuple[np.ndarray, tuple]] = []   # raw (grid, actions) log
        self._trans: List[Tuple[np.ndarray, str, np.ndarray]] = []
        self._change: Optional[np.ndarray] = None   # per-cell change counts
        self._rowchg: Optional[np.ndarray] = None   # transitions where row had a change
        self._colchg: Optional[np.ndarray] = None
        self._row_last: Optional[np.ndarray] = None  # transition idx of last row change
        self._col_last: Optional[np.ndarray] = None
        self._row_gaps: List[Dict[int, int]] = []   # per row: gap length -> count
        self._col_gaps: List[Dict[int, int]] = []
        # Per-action stats: action -> [cell counts, row counts, col counts, n].
        # HUD ticks under EVERY action; world cells the current policy happens to
        # hammer (graph explore repeating ACTION1 on a sprite-cycler) change under
        # SOME actions only. Requiring the rate under each well-sampled action
        # keeps the mask a property of the game, not of the policy.
        self._act: Dict[str, list] = {}
        self._mask: Optional[np.ndarray] = None     # True = ignore cell in the hash
        self._n = 0
        self.grids: Dict[int, np.ndarray] = {}      # hash -> a raw grid of that state
        # (state, action) pairs a world model predicted to be no-ops: treated as
        # tested by imagination (unscored). Cleared on mask rebuilds (stale hashes).
        self._noop_skip: set = set()

    def _hash(self, grid: np.ndarray) -> int:
        m = self._mask
        if m is not None and m.shape == grid.shape:
            g = grid.copy()
            g[m] = -1
            return hash((g.shape, g.tobytes()))
        return hash((grid.shape, grid.tobytes()))

    def changed_masked(self, prev: np.ndarray, nxt: np.ndarray) -> bool:
        """Did the WORLD change (ignoring HUD/timer cells)? A move that only
        ticked the HUD did nothing the agent should credit as an effect."""
        if prev.shape != nxt.shape:
            return True
        d = prev != nxt
        if self._mask is not None and self._mask.shape == d.shape:
            d = d & ~self._mask
        return bool(d.any())

    def _ensure(self, h: int, acts) -> None:
        if h not in self.untested:
            self.untested[h] = set()
            self.adj[h] = {}
        for a in acts:
            if a not in self.adj[h]:
                self.untested[h].add(a)

    def ensure_state(self, grid: np.ndarray, valid: List[str]):
        """Register the current frame's state; unseen actions become untested."""
        acts = tuple(sorted({base_action(a) for a in valid
                             if base_action(a) != "ACTION6"}))
        if len(self._ensures) < self.MAX_LOG:
            self._ensures.append((grid, acts))
        h = self._hash(grid)
        self.grids.setdefault(h, grid)
        self._ensure(h, acts)

    def record(self, prev_grid: np.ndarray, action: str, next_grid: np.ndarray):
        a = base_action(action)
        if a == "ACTION6":
            return
        if len(self._trans) < self.MAX_LOG:
            self._trans.append((prev_grid, a, next_grid))
        if prev_grid.shape == next_grid.shape:
            if self._change is None or self._change.shape != prev_grid.shape:
                self._change = np.zeros(prev_grid.shape, dtype=np.float32)
                self._rowchg = np.zeros(prev_grid.shape[0], dtype=np.float32)
                self._colchg = np.zeros(prev_grid.shape[1], dtype=np.float32)
                self._row_last = np.full(prev_grid.shape[0], -1, dtype=np.int64)
                self._col_last = np.full(prev_grid.shape[1], -1, dtype=np.int64)
                self._row_gaps = [{} for _ in range(prev_grid.shape[0])]
                self._col_gaps = [{} for _ in range(prev_grid.shape[1])]
                self._act = {}
                self._n = 0
            d = prev_grid != next_grid
            self._change += d
            rhit, chit = d.any(axis=1), d.any(axis=0)
            self._rowchg += rhit
            self._colchg += chit
            for i in np.nonzero(rhit)[0]:
                if self._row_last[i] >= 0:
                    gap = self._n - int(self._row_last[i])
                    self._row_gaps[i][gap] = self._row_gaps[i].get(gap, 0) + 1
                self._row_last[i] = self._n
            for j in np.nonzero(chit)[0]:
                if self._col_last[j] >= 0:
                    gap = self._n - int(self._col_last[j])
                    self._col_gaps[j][gap] = self._col_gaps[j].get(gap, 0) + 1
                self._col_last[j] = self._n
            per = self._act.get(a)
            if per is None:
                per = self._act[a] = [np.zeros(prev_grid.shape, dtype=np.float32),
                                      np.zeros(prev_grid.shape[0], dtype=np.float32),
                                      np.zeros(prev_grid.shape[1], dtype=np.float32), 0]
            per[0] += d
            per[1] += d.any(axis=1)
            per[2] += d.any(axis=0)
            per[3] += 1
            self._n += 1
            if self._n % self.MASK_CHECK_EVERY == 0:
                self._update_mask()
        hp, hn = self._hash(prev_grid), self._hash(next_grid)
        self.grids.setdefault(hp, prev_grid)
        self.grids.setdefault(hn, next_grid)
        self._ensure(hp, ())                        # no-op if already known
        self.adj[hp][a] = hn
        self.untested[hp].discard(a)

    def _update_mask(self):
        n = max(1, self._n)
        new = (self._change / n) > self.MASK_RATE
        # Per-cell confirmation (needs >=2 well-sampled actions): a HUD cell
        # ticks under all of them; a cell only the current policy hammers
        # (repeated ACTION1 on a sprite-cycler) is quiet under another action.
        sampled = [p for p in self._act.values() if p[3] >= self.MASK_MIN_PER_ACT]
        if len(sampled) >= 2:
            for cc, rc, colcnt, na in sampled:
                new &= (cc / na) > self.MASK_RATE
        else:
            new[:] = False
        # HUD strips: clock-periodic rows/cols (see MASK_LINE_* above).
        def _periodic(gaps: Dict[int, int]) -> bool:
            tot = sum(gaps.values())
            return (tot + 1 >= self.MASK_LINE_MIN_CHG
                    and max(gaps.values()) / tot >= self.MASK_LINE_PERIODIC)
        new[[i for i, g in enumerate(self._row_gaps) if g and _periodic(g)], :] = True
        new[:, [j for j, g in enumerate(self._col_gaps) if g and _periodic(g)]] = True
        if not new.any():
            new = None
        old = self._mask
        if (old is None and new is None) or \
                (old is not None and new is not None and np.array_equal(old, new)):
            return
        self._mask = new
        # Every stored hash may have changed -> rebuild EXACTLY from the raw logs.
        self.adj = {}
        self.untested = {}
        self.grids = {}
        self._noop_skip = set()
        for grid, acts in self._ensures:
            h = self._hash(grid)
            self.grids.setdefault(h, grid)
            self._ensure(h, acts)
        for prev, a, nxt in self._trans:
            hp, hn = self._hash(prev), self._hash(nxt)
            self.grids.setdefault(hp, prev)
            self.grids.setdefault(hn, nxt)
            self._ensure(hp, ())
            self.adj[hp][a] = hn
            self.untested[hp].discard(a)

    def nearest_untested_grid(self, grid: np.ndarray,
                              dead: Optional[DeadActionTracker] = None,
                              noop=None):
        return self.nearest_untested(self._hash(grid), dead, noop)

    def nearest_untested(self, h: int, dead: Optional[DeadActionTracker] = None,
                         noop=None):
        """UNSCORED BFS through known edges to the closest state that still has
        an untested (non-dead) action. Returns (action_path_to_it, untested_action)
        or None if the whole recorded graph is exhausted.

        `noop(grid, action) -> bool` is an optional world-model predicate: pairs
        it confidently predicts to be no-ops are tested by IMAGINATION instead of
        by a scored action (sticky in _noop_skip). Callers must fall back to a
        noop-less pass when this returns None -- the model may be wrong, so
        skipped pairs are re-offered before declaring true exhaustion."""
        seen = {h}
        q = deque([(h, [])])
        while q:
            cur, path = q.popleft()
            avail = [a for a in self.untested.get(cur, ())
                     if dead is None or not dead.is_dead(a)]
            if noop is not None and avail:
                g = self.grids.get(cur)
                if g is not None:
                    keep = []
                    for a in avail:
                        if (cur, a) in self._noop_skip:
                            continue
                        if noop(g, a):
                            self._noop_skip.add((cur, a))
                        else:
                            keep.append(a)
                    avail = keep
            if avail:
                # Least-GLOBALLY-used first. Plain sorted() hammered ACTION1
                # forever on games where every frame is novel (tr87: 284 of 297
                # presses) -- depth-first down one button. Balancing usage
                # explores the action product space and feeds every action
                # enough samples for the HUD-mask veto. Ties break RANDOMLY
                # (seeded RNG): a fixed name-order tie-break degenerates into a
                # strict A1-A2-A3-A4 round-robin, which stamps a fake clock
                # period onto world rows and fools the periodicity mask.
                return path, min(avail, key=lambda a: (
                    self._act.get(a, (0, 0, 0, 0))[3], random.random()))
            if len(seen) > self.MAX_BFS_STATES:
                break
            for a, nxt in self.adj.get(cur, {}).items():
                if nxt not in seen:
                    seen.add(nxt)
                    q.append((nxt, path + [a]))
        return None


class PatchWorldModel:
    """Learned LOCAL rewrite rules (COMPONENT 5.9): an executable world model.

    ARC-AGI-3 dynamics are local (moving onto track cells, walls, cursor steps,
    sprite cycles), so a cell's next value is usually a function of the action
    and the cell's 5x5 neighborhood. Those rules are learned from every recorded
    transition and -- unlike the exact StateGraph -- GENERALIZE across positions:
    a wall learned at one spot predicts a never-tried move into a wall elsewhere.
    Use: predict an untested action's outcome with unscored compute instead of
    spending a scored action to observe it (the RHAE currency exchange).

    The prediction target is transition EFFECTIVENESS (did any unmasked cell
    change), not per-cell values: games with global side-effects (tu93's
    platforms move by a history-dependent rule after every VALID move) make
    per-cell outcomes unpredictable, but ineffectiveness is still decided at one
    local site (the avatar pressed against a non-track cell). A (state, action)
    is called a no-op only when EVERY unmasked cell's patch is known (seen >=
    MIN_SEEN times under that action) AND the state contains a BLOCKING WITNESS:
    a patch that has only ever preceded ineffective transitions (>= WITNESS_MIN
    times). Unknown contexts always fall back to real testing, and StateGraph's
    caller-side exhaustion fallback re-offers model-skipped pairs, so
    completeness is never lost to a wrong rule.
    HUD cells (StateGraph mask) are neutralized both as patch context and in the
    effectiveness label -- a ticking budget bar would otherwise make every
    transition look effective (the same trap that broke raw state hashing).
    """
    K = 2                    # neighborhood radius -> 5x5 patches
    MIN_SEEN = 2             # occurrences before a patch counts as known
    WITNESS_MIN = 2          # all-ineffective occurrences to certify a blocker
    PAD = -2                 # out-of-grid / HUD fill (never a real color)
    MEMO_REFRESH = 25        # re-evaluate a cached False after this much new data

    def __init__(self):
        # (action, patch_bytes) -> [effective_count, ineffective_count]
        self.stats: Dict[Tuple[str, bytes], list] = {}
        self.trained = 0
        self._memo: Dict[Tuple[int, str], Tuple[int, bool]] = {}

    def _patches(self, grid: np.ndarray, mask: Optional[np.ndarray]) -> np.ndarray:
        g = np.array(grid, dtype=np.int32)          # own copy; PAD needs signed
        if mask is not None and mask.shape == g.shape:
            g[mask] = self.PAD
        k = self.K
        p = np.pad(g, k, mode="constant", constant_values=self.PAD)
        w = np.lib.stride_tricks.sliding_window_view(p, (2 * k + 1, 2 * k + 1))
        return np.ascontiguousarray(w.reshape(g.size, -1))

    @staticmethod
    def _live(mask: Optional[np.ndarray], shape) -> np.ndarray:
        if mask is not None and mask.shape == shape:
            return np.nonzero(~mask.ravel())[0]
        return np.arange(int(np.prod(shape)))

    def learn(self, prev: np.ndarray, action: str, nxt: np.ndarray,
              mask: Optional[np.ndarray] = None):
        a = base_action(action)
        if a == "ACTION6" or prev.shape != nxt.shape:
            return
        d = prev != nxt
        if mask is not None and mask.shape == d.shape:
            d = d & ~mask
        eff = bool(d.any())
        pat = self._patches(prev, mask)
        stats = self.stats
        for i in self._live(mask, prev.shape):
            key = (a, pat[i].tobytes())
            s = stats.get(key)
            if s is None:
                s = stats[key] = [0, 0]
            s[0 if eff else 1] += 1
        self.trained += 1

    def predict_noop(self, grid: np.ndarray, action: str,
                     mask: Optional[np.ndarray] = None) -> bool:
        """True iff every live cell's context is known AND a blocking witness
        (an always-ineffective patch) is present."""
        a = base_action(action)
        if a == "ACTION6":
            return False
        key = (hash(grid.tobytes()), a)
        memo = self._memo.get(key)
        if memo is not None and (memo[1] or
                                 self.trained - memo[0] < self.MEMO_REFRESH):
            return memo[1]
        pat = self._patches(grid, mask)
        stats = self.stats
        witness = False
        result = True
        for i in self._live(mask, grid.shape):
            s = stats.get((a, pat[i].tobytes()))
            if s is None or s[0] + s[1] < self.MIN_SEEN:
                result = False                  # unknown context -> test for real
                break
            if s[0] == 0 and s[1] >= self.WITNESS_MIN:
                witness = True
        result = result and witness
        self._memo[key] = (self.trained, result)
        return result


# ==========================================
# COMPONENT 5.10: TIMELINE  (append-only ground truth; Schema-harness Phase 1)
# ==========================================
#
# The Schema-harness discipline: keep an APPEND-ONLY, immutable record of every
# real transition. Learned models are provisional and get revised; the Timeline
# never does -- it is the ground truth every model must be CERTIFIED against
# (full-history backtest) before the agent is allowed to commit a multi-step
# plan inside that model. It deliberately survives GAME_OVER -> RESET
# (deterministic games: evidence from a failed attempt still constrains the
# theory) and is only SEGMENTED by level tag -- never cleared -- on level-up,
# because a new layout obsoletes positional facts but not the record itself.

class TimelineEntry:
    __slots__ = ("prev", "action", "nxt", "reward", "level", "_mv")

    def __init__(self, prev, action, nxt, reward, level):
        self.prev = prev
        self.action = action
        self.nxt = nxt
        self.reward = reward
        self.level = level
        self._mv = None      # cached (avatar_color, motion) -- replay cost amortiser

    def motion(self, color):
        """Observed centroid motion of `color` across this transition, or None
        when the colour is absent on either side / shapes differ (out of a
        movement theory's scope). Cached per colour: backtests replay the whole
        history every time the theory is revised, and the np.nonzero scans are
        the dominant cost."""
        if self._mv is not None and self._mv[0] == color:
            return self._mv[1]
        res = None
        if (self.prev is not None and self.nxt is not None
                and self.prev.shape == self.nxt.shape):
            prs, pcs = np.nonzero(self.prev == color)
            nrs, ncs = np.nonzero(self.nxt == color)
            if len(prs) and len(nrs):
                pr, pc = prs.mean(), pcs.mean()
                res = (pr, pc, nrs.mean() - pr, ncs.mean() - pc)
        self._mv = (color, res)
        return res


class Timeline:
    def __init__(self):
        self.entries: List[TimelineEntry] = []

    def append(self, prev, action, nxt, reward, level):
        self.entries.append(TimelineEntry(prev, action, nxt, reward, level))

    def __len__(self):
        return len(self.entries)


# ==========================================
# COMPONENT 6: ACT & MAIN AGENT
# ==========================================

class MyAgent(Agent):
    MAX_ACTIONS = 1500

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        gid = getattr(self, "game_id", "local")
        # Kaggle: fresh time-based randomness per game. Local eval: a caller can pin
        # ARC_AGENT_SEED to make the exploration RNG reproducible (the game wrapper's
        # own --seed does NOT control the agent's clicks), so results are comparable
        # across code changes instead of flickering on luck.
        _seed_env = os.environ.get("ARC_AGENT_SEED")
        if _seed_env is not None:
            seed = int(_seed_env) + (hash(gid) % 1_000_000)
        else:
            seed = int(time.time() * 1e6) + (hash(gid) % 1_000_000)
        random.seed(seed); np.random.seed(seed % (2**32 - 1))
        if torch is not None:
            torch.manual_seed(seed)

        self.encoder = StateEncoder()
        self.llm = get_local_llm()
        self.memory = MemoryManager()
        self.world_model = WorldModelManager(self.llm, self.encoder)
        self.counter = StateCounter()
        self.planner = MCTSPlanner(self.world_model, self.encoder, self.counter)
        self.rplanner = ExecPlanner()   # executable world model + unscored plan (COMPONENT 5.7)
        # Append-only ground truth (COMPONENT 5.10). Owned here, never cleared:
        # ExecPlanner certifies its movement theory against it before planning.
        self.timeline = Timeline()
        self.rplanner.timeline = self.timeline
        self.cplanner = ClickPlanner()
        self.sgraph = StateGraph()      # exact transition graph (COMPONENT 5.8)
        self.dead = DeadActionTracker()
        # Local rewrite rules (COMPONENT 5.9). Persists across levels ON PURPOSE:
        # the rules are the game's physics, not the level's layout.
        self.pwm = PatchWorldModel()
        self.gate = ExploitGate()

        self.transitions: List[Transition] = []
        self.last_grid: Optional[np.ndarray] = None
        self.last_action_str: str = ""
        self.last_score = 0
        self.action_x = 0
        self.action_y = 0
        self.level_steps = 0
        self.epsilon = EPSILON_START

        self.llm_calls_this_level = 0
        self.synth_inflight = False
        self.synth_started_at = 0.0
        self.synth_generation = 0
        self._last_enum_at = -1

        # LLM click-reasoning plan (click-only CSP games). A budgeted background
        # call fills an ordered queue of pixel targets; choose_action consumes one
        # per step. Empty / LLM-absent => the ClickPlanner coverage floor drives.
        self._click_plan: "deque[Tuple[int, int]]" = deque()
        self._click_inflight = False
        self._click_src_hash: Optional[int] = None
        self._last_click_plan_at = -10 ** 9
        # Schema Phase 2: budgeted background THEORIZER for movement games. A
        # completed proposal is adopted on the MAIN thread (the backtest mutates
        # certification state act() also uses), gated by the full-timeline
        # backtest in ExecPlanner.propose_theory.
        self._theory_proposal: Optional[dict] = None
        self._theory_inflight = False
        self._last_theorize_at = -10 ** 9
        self._win_recorded = False
        self.last_checkpoint = time.time()

        # Preload the 27B in the background so the first budgeted call is instant.
        if self.llm.available and not self.llm.is_ready():
            threading.Thread(target=self.llm.ensure_loaded, daemon=True).start()

    # ---- helpers ----
    @staticmethod
    def _epsilon_for(step: int) -> float:
        return max(EPSILON_FLOOR, EPSILON_START * (EPSILON_DECAY ** step))   # Issue 5

    def _reset_level_state(self):
        self.transitions.clear()
        self.last_grid = None
        self.last_action_str = ""
        self.llm_calls_this_level = 0
        self.synth_inflight = False
        self.synth_generation += 1        # orphan any in-flight synth thread
        self.level_steps = 0
        self.epsilon = EPSILON_START
        self._last_enum_at = -1
        self._win_recorded = False
        self._theory_proposal = None
        self._theory_inflight = False
        self._last_theorize_at = -10 ** 9
        self.world_model.reset_level()
        self.rplanner.reset()
        self.cplanner.reset()
        self._discard_click_plan()
        self.gate = ExploitGate()

    def _synth_watchdog(self):
        """Issue 6: a stalled LLM thread can't be killed, but it CAN be orphaned --
        clear the inflight flag and bump the generation so its completion is ignored
        for flag bookkeeping (a late candidate still verifies on fresh data)."""
        if self.synth_inflight and time.time() - self.synth_started_at > SYNTH_MAX_INFLIGHT_S:
            print("[synth] watchdog: abandoning stalled LLM synthesis thread")
            self.synth_generation += 1
            self.synth_inflight = False

    # ---- LLM click reasoning (click-only CSP games) ----
    _CLICK_PLAN_MIN_INTERVAL = 40   # steps between plan requests (throttle the budget)
    _CLICK_PLAN_MIN_OBS = 20        # coverage clicks before the LLM has data to reason on

    def _discard_click_plan(self):
        self._click_plan.clear()
        self._click_src_hash = None

    @staticmethod
    def _grid_to_text(grid: np.ndarray, max_side: int = 64) -> str:
        """Compact char-grid for the LLM: one char per cell, '.' for the modal
        background, 0-9/a-z for other colours. Pixel coords are preserved (no
        downscaling), so a coord the LLM returns maps straight to a click."""
        g = grid
        if g.shape[0] > max_side or g.shape[1] > max_side:
            g = g[:max_side, :max_side]
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[counts.argmax()]) if len(vals) else 0
        alpha = "0123456789abcdefghijklmnopqrstuvwxyz"
        rows = []
        for row in g:
            rows.append("".join("." if int(v) == bg else alpha[int(v) % len(alpha)]
                                for v in row))
        return "\n".join(rows)

    def _maybe_request_click_plan(self, grid: np.ndarray):
        """Fire a budgeted background LLM call to plan clicks. No-op unless the LLM
        is loaded, budget remains, no plan/request is already pending, and enough
        board evidence exists. choose_action never blocks on this."""
        if (not self.llm.is_ready() or self._click_inflight or self._click_plan
                or self.llm_calls_this_level >= LLM_CALL_BUDGET_PER_LEVEL
                or self.level_steps - self._last_click_plan_at < self._CLICK_PLAN_MIN_INTERVAL
                or self.cplanner._total_clicks < self._CLICK_PLAN_MIN_OBS):
            return
        self._click_inflight = True
        self._last_click_plan_at = self.level_steps
        self.llm_calls_this_level += 1
        gen = self.synth_generation
        src_hash = int(hash((grid.shape, grid.tobytes())))
        board_text = self._grid_to_text(grid)
        obs = self.memory.observation_digest(self.transitions)
        reactive = ", ".join(f"({r},{c})" for r, c in
                             list(self.cplanner._reactive.values())[:12])
        hyp = self.memory.l2_hypothesis

        def _job():
            try:
                clicks = self.llm.plan_clicks(board_text, obs, reactive, hyp)
            except Exception as e:
                print(f"[click-plan] failed: {e}")
                clicks = []
            finally:
                # Only adopt if this request wasn't orphaned by a level-reset.
                if gen == self.synth_generation and clicks:
                    h, w = grid.shape
                    self._click_plan = deque(
                        (max(0, min(h - 1, int(r))), max(0, min(w - 1, int(c))))
                        for r, c in clicks)
                    self._click_src_hash = src_hash
                self._click_inflight = False
        threading.Thread(target=_job, daemon=True).start()

    # ---- Schema Phase 2: budgeted LLM theorizer (movement games) ----
    _THEORIZE_AFTER = 120       # scored actions on a level before the theorizer may fire
    _THEORIZE_INTERVAL = 120    # min steps between theorize requests (budget throttle)

    def _maybe_request_theory(self, grid: np.ndarray):
        """Fire a budgeted background THEORIZE call when the local loop is
        demonstrably stuck (many scored actions on this level, no level-up).
        The result is stored as a pending proposal; choose_action adopts it on
        the main thread via ExecPlanner.propose_theory, which backtests it
        against the full timeline. choose_action never blocks on this; with no
        LLM (local eval) this is a no-op, so the measured no-LLM core is
        untouched."""
        if (not self.llm.is_ready() or self._theory_inflight
                or self._theory_proposal is not None
                or self.llm_calls_this_level >= LLM_CALL_BUDGET_PER_LEVEL
                or self.level_steps < self._THEORIZE_AFTER
                or self.level_steps - self._last_theorize_at < self._THEORIZE_INTERVAL):
            return
        rp = self.rplanner
        if rp.avatar_color is None:
            return
        self._theory_inflight = True
        self._last_theorize_at = self.level_steps
        self.llm_calls_this_level += 1
        gen = self.synth_generation
        board_text = self._grid_to_text(grid)
        theory = (f"avatar_color={rp.avatar_color}, "
                  f"action_displacements={rp.action_disp}, "
                  f"obstacle_colors={sorted(rp.obstacle_colors)}, "
                  f"decoy_colors={sorted(rp.non_goal_colors)}, "
                  f"positional_walls={len(rp.walls)} cells, "
                  f"anomalous_cells={sorted(rp._anomalies)}")
        obs = self.memory.observation_digest(self.transitions)
        trouble = ("certification RED: a recorded transition contradicts the theory"
                   if not rp._cert_green else
                   f"{self.level_steps} actions on this level without a level-up; "
                   f"target pursuit is not finding the goal")

        def _job():
            try:
                res = self.llm.theorize(board_text, theory, obs, trouble)
            except Exception as e:
                print(f"[theorize] failed: {e}")
                res = None
            finally:
                # Only adopt if this request wasn't orphaned by a level change.
                if gen == self.synth_generation and res:
                    self._theory_proposal = res
                self._theory_inflight = False
        threading.Thread(target=_job, daemon=True).start()

    def _next_click_from_plan(self, grid: np.ndarray) -> Optional[str]:
        """Pop the next planned target. The plan is an ORDERED sequence reasoned
        about the board as it was when the request fired, so it is only valid if it
        STARTS from that same board. We validate the start ONCE: on the first
        consume, if the live board no longer hashes to the request-time snapshot
        (coverage mutated it while the LLM was generating), the ordered assumption is
        broken -> drop the whole plan and fall back to coverage. After the start is
        validated, each planned click legitimately mutates the board, so we stop
        re-checking. Lethal targets coverage already learned are skipped so a plan
        can never trip a GAME_OVER coverage would have avoided (score below floor)."""
        if not self._click_plan:
            return None
        if self._click_src_hash is not None:               # start not yet validated
            if int(hash((grid.shape, grid.tobytes()))) != self._click_src_hash:
                self._discard_click_plan()                 # stale start -> abandon
                return None
            self._click_src_hash = None                    # validated; don't re-check
        while self._click_plan:
            r, c = self._click_plan.popleft()
            if self.cplanner.is_lethal(r, c):
                continue                                   # skip a known-fatal target
            return f"ACTION6_r{r}_c{c}"
        return None

    def _valid_actions(self, latest_frame) -> List[str]:
        avail = getattr(latest_frame, "available_actions", None)
        if avail:
            out = []
            for a in avail:
                name = getattr(a, "name", None)          # arcengine GameAction enum
                if name is None:
                    try:
                        name = f"ACTION{int(a)}"          # raw API action id
                    except (TypeError, ValueError):
                        name = str(a)
                if name in VALID_ACTION_NAMES:
                    out.append(name)
            if out:
                return out
        return [f"ACTION{i}" for i in range(1, 8)]

    @staticmethod
    def _read_score(latest_frame) -> int:
        # arcengine >=0.9.3 renamed score -> levels_completed; support both.
        score = getattr(latest_frame, "levels_completed", None)
        if score is None:
            score = getattr(latest_frame, "score", 0)
        try:
            return int(score or 0)
        except (TypeError, ValueError):
            return 0

    @staticmethod
    def _grid_from(frame_obj) -> Optional[np.ndarray]:
        raw = getattr(frame_obj, "grid", None)
        if raw is None:
            raw = getattr(frame_obj, "frame", None)
        if raw is None:
            return None
        try:
            arr = np.array(raw, dtype=np.int32)
        except Exception:
            return None
        if arr.ndim == 3:               # arcengine FrameData.frame is a LIST of grids
            arr = arr[-1]               # (animation frames); the last one is current
        return arr if arr.size and arr.ndim == 2 else None

    def _parse_grid(self, latest_frame, frames) -> Optional[np.ndarray]:
        grid = self._grid_from(latest_frame)
        if grid is None and frames:
            for f in reversed(frames[:-1] if frames and frames[-1] is latest_frame else frames):
                grid = self._grid_from(f)
                if grid is not None:
                    break
        return grid

    def parse_action(self, action_str: str):
        if action_str.startswith("ACTION6"):
            r = c = None
            try:
                parts = action_str.split("_")
                r = int(parts[1][1:])
                c = int(parts[2][1:])
            except Exception:
                if self.last_grid is not None:
                    r, c = self.last_grid.shape[0] // 2, self.last_grid.shape[1] // 2
                else:
                    r = c = 0
            self.action_x, self.action_y = int(c), int(r)     # API convention: x=col, y=row
            act = GameAction.ACTION6
            if hasattr(act, "set_data"):
                try:
                    act.set_data({"x": int(c), "y": int(r)})
                except Exception:
                    pass
            return act
        return getattr(GameAction, action_str, GameAction.ACTION1)

    def _graph_explore(self, grid: np.ndarray, valid: List[str]) -> Optional[str]:
        """Graph-directed exploration (COMPONENT 5.8): the single step that makes
        progress toward the nearest (state, action) pair with an UNKNOWN outcome.
        Recomputed every call -- the games are deterministic, so re-running BFS
        from the live state stays consistent, and if reality diverges (hidden
        timer -> novel state) the novel state's own untested actions take over."""
        if os.environ.get("ARC_NO_GRAPH"):          # debug bisect switch
            return None
        noop = None
        if not os.environ.get("ARC_NO_PWM"):        # debug bisect switch
            noop = lambda g, a: self.pwm.predict_noop(g, a, self.sgraph._mask)
        res = self.sgraph.nearest_untested_grid(grid, self.dead, noop=noop)
        if res is None and noop is not None:
            # Imagination may have pruned everything. Completeness fallback:
            # re-offer the model-skipped pairs for real testing.
            res = self.sgraph.nearest_untested_grid(grid, self.dead)
        if res is None:
            return None
        path, a = res
        step = path[0] if path else a
        return step if step in valid else None

    def _explore_action(self, grid: np.ndarray, valid: List[str]) -> str:
        # Directed beats random: never re-spend a scored action on a known outcome.
        a = self._graph_explore(grid, valid)
        if a is not None:
            return a
        # Graph exhausted (or action not currently valid): fall back to novelty /
        # random, skipping actions proven inert everywhere this level.
        alive = [a for a in valid if not self.dead.is_dead(a)] or valid
        if random.random() < self.epsilon or self.world_model.active_model is None:
            return random.choice(alive)
        best_a, best_nov = alive[0], -1.0
        for a in alive:
            nov = self.counter.peek_bonus(self.world_model.predict(grid, a))
            if nov > best_nov:
                best_nov, best_a = nov, a
        return best_a

    def _checkpoint(self):
        path = "/kaggle/working/agent_checkpoint.pkl" if os.path.isdir("/kaggle/working") else "agent_checkpoint.pkl"
        try:
            with open(path, "wb") as f:
                pickle.dump({"action_counter": getattr(self, "action_counter", 0),
                             "hypothesis": self.memory.l2_hypothesis,
                             "mode": self.gate.mode}, f)
        except Exception as e:
            print(f"Checkpoint failed: {e}")
        self.last_checkpoint = time.time()

    def _record_transition(self, prev, action, nxt, reward):
        # Ground truth first (append-only; level tag = the level the action was
        # taken IN, i.e. before this transition's reward is applied).
        self.timeline.append(prev, action, nxt, reward, self.last_score)
        diff = self.encoder.encode_transition(prev, nxt, action)
        t = Transition(prev, action, nxt, reward, diff, time.time())
        self.transitions.append(t)
        if len(self.transitions) > 200:
            self.transitions.pop(0)
        self.memory.add_transition(t)

    # ---- interface ----
    def is_done(self, frames, latest_frame) -> bool:
        if latest_frame.state == GameState.WIN:
            # Roadmap V3 fix: capture the transition that CAUSED the win with the
            # REAL final grid from the environment -- never a duplicated last_grid.
            if not self._win_recorded and self.last_grid is not None and self.last_action_str:
                self._win_recorded = True
                final = self._parse_grid(latest_frame, frames)
                if final is not None and not np.array_equal(final, self.last_grid):
                    self._record_transition(self.last_grid, self.last_action_str, final, 1.0)
                elif self.transitions:
                    # WIN frame carries no (new) grid: credit the previous
                    # action's already-recorded transition instead of fabricating one.
                    self.transitions[-1].reward = 1.0
                self.memory.l1_episodes.append(f"WIN via {self.last_action_str}.")
            return True
        return getattr(self, "action_counter", 0) >= self.MAX_ACTIONS

    def choose_action(self, frames, latest_frame):
        if time.time() - self.last_checkpoint > CHECKPOINT_INTERVAL_S:
            self._checkpoint()
        self._synth_watchdog()

        grid = self._parse_grid(latest_frame, frames)
        if grid is None:
            return GameAction.ACTION1
        score = self._read_score(latest_frame)

        # 1) Learn from the previous step. Reward = a level was completed.
        if self.last_grid is not None and self.last_action_str:
            reward = 1.0 if score > self.last_score else 0.0
            self._record_transition(self.last_grid, self.last_action_str, grid, reward)
            # Only learn physics WITHIN a level. A level-up swaps the whole grid
            # (new avatar spawn), so prev->nxt is a discontinuity, not a move --
            # feeding it to the planner corrupts the learned displacements.
            if reward == 0:
                self.rplanner.update(self.last_grid, self.last_action_str, grid, self.encoder, reward)
                # Exact world model: record the observed edge (deterministic games,
                # so this is ground truth, not an estimate).
                self.sgraph.record(self.last_grid, self.last_action_str, grid)
                self.dead.update(self.last_action_str,
                                 changed=self.sgraph.changed_masked(self.last_grid, grid))
                self.pwm.learn(self.last_grid, self.last_action_str, grid,
                               self.sgraph._mask)
            self.cplanner.update(self.last_action_str,
                                 changed=not np.array_equal(self.last_grid, grid),
                                 reward=reward)
            self.counter.get_bonus(grid)
            m = self.world_model.active_model
            if m and not np.array_equal(self.last_grid, grid):
                pred = self.world_model.predict(self.last_grid, self.last_action_str)
                hit = 1.0 if np.array_equal(pred, grid) else 0.0
                m.accuracy = 0.9 * m.accuracy + 0.1 * hit
                self.gate.update(hit)
                if m.accuracy < DEMOTE_ACCURACY:
                    self.world_model.demote(m)
                    self.gate = ExploitGate()

        if os.environ.get("CP_DEBUG") and score < self.last_score:
            import sys
            print(f"[CP] SCORE REGRESSION {self.last_score}->{score} "
                  f"prev_action={self.last_action_str!r} state={latest_frame.state} "
                  f"cp_phase={getattr(self.cplanner,'_phase',None)} "
                  f"searching={self.cplanner.searching}", file=sys.stderr)

        # Level-up: refresh per-level budgets, keep the verified physics model.
        if score != self.last_score:
            if score > self.last_score:
                print(f"[level] score {self.last_score} -> {score}; refreshing per-level budgets")
                self.llm_calls_this_level = 0
                self.level_steps = 0
                self.gate = ExploitGate()     # re-earn EXPLOIT on the new layout
                # Keep learned physics (avatar + per-action displacement) across
                # levels, but the layout changed -> forget positional stuck/target
                # AND the dead-cell blacklist (those coords belong to the old map).
                self.rplanner.new_level()
                self.cplanner.new_level()
                # New layout -> every recorded state/edge belongs to the old map.
                self.sgraph = StateGraph()
                self.dead = DeadActionTracker()
                self._discard_click_plan()
                self._theory_proposal = None      # reasoned about the old layout
                self._last_theorize_at = -10 ** 9
            self.last_score = score

        # 2) Terminal handling.
        if latest_frame.state == GameState.GAME_OVER:
            # Blame the click that killed us so the ClickPlanner backs off that cell
            # on restart (its stats survive the reset -- same layout, same hazards).
            if self.last_action_str.startswith("ACTION6"):
                self.cplanner.mark_loss(self.last_action_str)
            if self.llm.is_ready() and self.llm_calls_this_level < LLM_CALL_BUDGET_PER_LEVEL:
                self.llm_calls_this_level += 1
                self.memory.l2_hypothesis = self.llm.reflect(
                    self.memory.observation_digest(self.transitions), "GAME_OVER")
            self._reset_level_state()
            return GameAction.RESET
        if latest_frame.state == GameState.NOT_PLAYED:
            return GameAction.RESET

        valid = self._valid_actions(latest_frame)
        self.level_steps += 1
        self.epsilon = self._epsilon_for(self.level_steps)
        # Register the current state so its untried actions are known to the graph.
        # (Survives intra-level RESETs by design -- the layout is unchanged.)
        self.sgraph.ensure_state(grid, valid)

        # Click-only games (available_actions == [6]) can never learn a movement
        # world-model, and (b2) always sets the action for them, so the MCTS/
        # world-model path never runs -- firing world-model synth there would waste
        # the per-level LLM budget on a model nothing consumes. Route that budget to
        # click reasoning instead.
        click_only = (not any(a in ("ACTION1", "ACTION2", "ACTION3", "ACTION4") for a in valid)
                      and any(a.startswith("ACTION6") for a in valid))

        # Schema Phase 2 (THEORIZE): adopt a completed theorizer proposal on the
        # main thread, then maybe fire the next budgeted background request.
        # Movement games only -- mechanical games are graph-driven and click
        # games route the same budget to click reasoning instead.
        if not click_only and not os.environ.get("ARC_NO_THEORIZE"):
            if self._theory_proposal is not None:
                prop, self._theory_proposal = self._theory_proposal, None
                if self.rplanner.propose_theory(prop, grid):
                    print(f"[theorize] adopted: {str(prop.get('hypothesis', ''))[:100]}")
                else:
                    print("[theorize] proposal rejected by full-history backtest")
            if not self.rplanner.looks_mechanical():
                self._maybe_request_theory(grid)

        # 3) Discover the world model. Enumeration first (cheap, sync, re-run only
        #    when NEW evidence arrived); LLM as budgeted background fallback.
        n_changed = sum(1 for t in self.transitions if t.changed)
        if (not click_only and self.world_model.active_model is None
                and n_changed != self._last_enum_at and synth_ready(self.transitions)):
            self._last_enum_at = n_changed
            if not self.world_model.try_enumerative(self.transitions):
                if (self.llm.is_ready() and not self.synth_inflight
                        and self.llm_calls_this_level < LLM_CALL_BUDGET_PER_LEVEL):
                    self.synth_inflight = True
                    self.synth_started_at = time.time()
                    self.llm_calls_this_level += 1
                    gen = self.synth_generation
                    obs = self.memory.observation_digest(self.transitions)
                    def _job():
                        try:
                            self.world_model.synthesize_with_llm(obs, lambda: list(self.transitions))
                        finally:
                            if gen == self.synth_generation:
                                self.synth_inflight = False
                    threading.Thread(target=_job, daemon=True).start()

        # 4) Act.
        #    Priority: (a) bootstrap -- try each move action once to learn its
        #    displacement; (b) reactive goal-seek once physics is known (scale-
        #    invariant, the main solver); (c) fall back to world-model EXPLOIT /
        #    curiosity EXPLORE for non-movement games.
        action_str = None
        move_actions = [a for a in valid if a in ("ACTION1", "ACTION2", "ACTION3", "ACTION4")]
        untried = [a for a in move_actions if a not in self.rplanner.action_disp]
        # (a) Keep probing UNLEARNED moves until their displacement is known. A move
        #     blocked on its first try (avatar spawned against a wall) must be retried
        #     later from open space, else e.g. "up" is never learned and snake mazes
        #     become unsolvable. Rotate so each unlearned move gets fresh positions.
        if untried and (not self.rplanner.ready(valid) or self.level_steps % 6 == 0):
            action_str = untried[self.level_steps % len(untried)]
        elif (self.rplanner.ready(valid) and not self.rplanner.looks_mechanical()
                and random.random() > self.epsilon):
            # Game-type classifier: MECHANICAL games (rotation/cycling/counters --
            # frame changes a displacement model can't explain) must NOT be driven
            # by ExecPlanner; they fall through to (c), where the state graph
            # explores untested (state, action) pairs systematically.
            # ExecPlanner (COMPONENT 5.7) is the PRIMARY driver: T1 unscored optimal
            # planning + T2 directed frontier / falsification-driven livelock escape.
            # The decaying-epsilon gate keeps a small escape hatch to the count-based
            # novelty explorer (path c) for pathologies the planner can't break on its
            # own (e.g. keydoor's non-integer render scale). Epsilon decays per level,
            # so this costs few scored actions once physics is learned.
            action_str = self.rplanner.act(grid, valid, self.encoder)   # (b) goal-seek

        # (b2) Click-driven games: no movement actions at all, but ACTION6 is
        #      available. Three layers, ONE active at a time (search XOR LLM-plan
        #      XOR coverage), with coverage as the guaranteed floor:
        #        - the ClickPlanner's env-as-simulator search when it's running
        #          (its deterministic RESET-replay must not be interleaved);
        #        - otherwise an LLM-reasoned click plan for the CSP/logic puzzles
        #          blind coverage can't crack (budgeted, filled in the background);
        #        - otherwise coverage discovers the reactive surface.
        if action_str is None and click_only:
            others = [a for a in valid if not a.startswith("ACTION6")]
            if self.cplanner.searching:
                action_str = self.cplanner.act(grid, self.encoder, self.epsilon)
            else:
                # Only reason with the LLM while the search isn't driving (keeps
                # one reasoning layer active); coverage evidence feeds the prompt.
                self._maybe_request_click_plan(grid)
                action_str = self._next_click_from_plan(grid)
                if action_str is None and self._click_inflight:
                    # A plan is generating (reasoned about THIS board). Idle on a
                    # known-inert cell so the board doesn't move under it -- then it
                    # lands on a matching board and its ordered start validates.
                    hold = self.cplanner.inert_cell()
                    if hold is not None:
                        action_str = f"ACTION6_r{hold[0]}_c{hold[1]}"
                # Don't inject a stray ACTION5/7 mid-plan/replay -- it desyncs the
                # deterministic sequences the plan / search depend on.
                if action_str is None and others and random.random() < 0.15:
                    action_str = random.choice(others)   # probe ACTION5/7 occasionally
                elif action_str is None:
                    action_str = self.cplanner.act(grid, self.encoder, self.epsilon)

        if action_str is None:                           # (c)
            if self.gate.should_exploit() and self.world_model.active_model is not None:
                action_str = self.planner.search(grid, valid, n_iterations=60)
            else:
                action_str = self._explore_action(grid, valid)

        # Any path that yields a bare "ACTION6" (random explore, MCTS edge cases)
        # would resolve to a fixed center click -- give it real coordinates.
        if action_str == "ACTION6":
            action_str = self.cplanner.act(grid, self.encoder, self.epsilon)

        self.last_grid = grid
        self.last_action_str = action_str
        return self.parse_action(action_str)


# ==========================================
# SELF-TESTS (offline; no LLM / no sentence-transformers required)
# ==========================================
if __name__ == "__main__":
    print("Running component isolation tests...")
    enc = StateEncoder()
    g = np.zeros((10, 10), dtype=np.int32); g[2:5, 2:5] = 1
    assert "square" in enc.encode_state(g).lower()

    # Object cache: same grid -> same cached list object (Issue 3)
    assert enc.objects(g) is enc.objects(g)

    # --- ReactivePlanner: scale-invariant learning + BFS pathing ---
    rp = ReactivePlanner()
    # Avatar = color 2 (a 2x2 block, i.e. scale 2), jumps 2 px right under ACTION4.
    p = np.zeros((12, 12), dtype=np.int32); p[2:4, 2:4] = 2
    n = np.zeros((12, 12), dtype=np.int32); n[2:4, 4:6] = 2
    rp.update(p, "ACTION4", n, enc)
    assert rp.avatar_color == 2 and rp.action_disp["ACTION4"] == (0, 2)   # learned 2px, scale-invariant
    # A blocked move (no change) while a wall color sits in the destination -> obstacle learned.
    pb = np.zeros((12, 12), dtype=np.int32); pb[2:4, 2:4] = 2; pb[2:4, 4:6] = 8
    rp.update(pb, "ACTION4", pb, enc)
    assert 8 in rp.obstacle_colors
    # BFS routes AROUND a wall: avatar bottom-left, goal top-right, wall column between,
    # gap at the top. Teach all four moves first (scale 1 here for clarity).
    rp2 = ReactivePlanner(); rp2.avatar_color = 2
    rp2.action_disp = {"ACTION1": (-1, 0), "ACTION2": (1, 0), "ACTION3": (0, -1), "ACTION4": (0, 1)}
    rp2.obstacle_colors = {8}
    board = np.zeros((5, 5), dtype=np.int32)
    board[1:5, 2] = 8            # wall column 2, rows 1..4 (gap at row 0)
    board[4, 0] = 2              # avatar bottom-left
    board[0, 4] = 4              # goal top-right
    a = rp2.act(board, ["ACTION1", "ACTION2", "ACTION3", "ACTION4"], enc)
    assert a == "ACTION1", f"BFS should head up toward the gap, got {a}"   # not blindly right into the wall

    # --- ExecPlanner (COMPONENT 5.7): executable world model + unscored planning ---
    M4 = ["ACTION1", "ACTION2", "ACTION3", "ACTION4"]
    DISP = {"ACTION1": (-1, 0), "ACTION2": (1, 0), "ACTION3": (0, -1), "ACTION4": (0, 1)}
    # T1: commit a WHOLE path around a wall (not just the first step). A letterbox ring
    # (colour 5) is the border so the interior background (0) stays walkable; a colour-8
    # wall splits the interior with a single gap on the top interior row.
    ep = ExecPlanner(); ep.avatar_color = 2; ep.action_disp = dict(DISP); ep.obstacle_colors = {8}
    eb = np.zeros((20, 20), dtype=np.int32); eb[0, :] = eb[-1, :] = eb[:, 0] = eb[:, -1] = 5
    eb[2:19, 10] = 8                     # interior wall down col 10 (gap at interior row 1)
    eb[18, 1] = 2                        # avatar: interior bottom-left
    eb[1, 18] = 4                        # goal:   interior top-right (across the wall)
    a = ep.act(eb, M4, enc)
    assert a in ("ACTION1", "ACTION4"), f"ExecPlanner must head toward the gap/goal, got {a}"
    assert len(ep._plan) > 1, "T1: a whole multi-step path must be committed, not one step"
    # T1 falsification: a blocked move (no frame change) records a POSITIONAL wall cell
    # and DROPS the committed plan, so act() replans around it next turn.
    ep2 = ExecPlanner(); ep2.avatar_color = 2; ep2.action_disp = {"ACTION4": (0, 1)}
    pb = np.zeros((6, 6), dtype=np.int32); pb[2, 2] = 2
    ep2._plan = deque([((0, 0), "ACTION4")])                 # a stale committed step
    ep2.update(pb, "ACTION4", pb, enc)                       # ACTION4 vetoed (no change)
    assert (2, 3) in ep2.walls, f"a blocked move must learn the wall cell, got {ep2.walls}"
    assert len(ep2._plan) == 0, "a falsified move must drop the committed plan"
    # T2 curiosity: no goal on the board -> steer to an UNVISITED cell (directed), never
    # return None-and-flail. Letterbox ring keeps the interior walkable.
    ep3 = ExecPlanner(); ep3.avatar_color = 2; ep3.action_disp = dict(DISP)
    fb = np.zeros((12, 12), dtype=np.int32); fb[0, :] = fb[-1, :] = fb[:, 0] = fb[:, -1] = 5
    fb[6, 6] = 2                                             # only the avatar, no goal
    af = ep3.act(fb, M4, enc)
    assert af in M4, f"T2 frontier must pick a directed move, got {af}"

    # --- Schema Phase 1 (COMPONENT 5.10): Timeline + CERTIFY + experiments ---
    # Grids: avatar colour 2, single pixel (scale 1). tg1->tgJ is a 5px jump the
    # theory (disp (0,1)) cannot explain -- and too large for the parent to
    # re-learn as a displacement, so it stays a stable counterexample.
    tg0 = np.zeros((6, 8), dtype=np.int32); tg0[3, 1] = 2
    tg1 = np.zeros((6, 8), dtype=np.int32); tg1[3, 2] = 2
    tgJ = np.zeros((6, 8), dtype=np.int32); tgJ[3, 7] = 2
    tgR = np.zeros((6, 8), dtype=np.int32); tgR[3, 3] = 2
    # (1) Consistent history certifies GREEN; a contradicting entry turns RED.
    tl = Timeline()
    ep4 = ExecPlanner(); ep4.avatar_color = 2; ep4.action_disp = {"ACTION4": (0, 1)}
    ep4.timeline = tl
    tl.append(tg0, "ACTION4", tg1, 0.0, 0)
    assert ep4.certify(), "a consistent record must certify green"
    tl.append(tg1, "ACTION4", tgJ, 0.0, 0)
    assert not ep4.certify(), "a contradicting record must certify red"
    # (2) RED voids the committed queue, and the next scored action is the
    #     DISCRIMINATING EXPERIMENT (re-run the disputed action at its cell).
    ep4._plan = deque([((3, 2), "ACTION4")])
    a4 = ep4.act(tg1, ["ACTION4"], enc)
    assert len(ep4._plan) == 0, "red certification must void the committed queue"
    assert a4 == "ACTION4", f"red -> re-test the disputed (cell, action), got {a4}"
    # (3) Re-test outcome A: theory HOLDS -> the recorded contradiction was a
    #     one-off scene event; the entry is excused and certification is green.
    tl2 = Timeline()
    ep5 = ExecPlanner(); ep5.CERT_REPLAY_GAP = 0
    ep5.avatar_color = 2; ep5.action_disp = {"ACTION4": (0, 1)}; ep5.timeline = tl2
    tl2.append(tg1, "ACTION4", tgJ, 0.0, 0)
    assert not ep5.certify()
    assert ep5.act(tg1, ["ACTION4"], enc) == "ACTION4"
    tl2.append(tg1, "ACTION4", tgR, 0.0, 0)          # re-test: moved 1px = theory
    ep5.update(tg1, "ACTION4", tgR, enc)
    assert 0 in ep5._excused, "a held re-test must excuse the one-off entry"
    assert ep5.certify(), "excused entries must leave certification green"
    # (4) Re-test outcome B: the contradiction REPRODUCES -> after EXP_TRIES the
    #     (cell, action) is an anomaly: out of certification scope, and BFS
    #     refuses to route plans through it.
    tl3 = Timeline()
    ep6 = ExecPlanner(); ep6.CERT_REPLAY_GAP = 0
    ep6.avatar_color = 2; ep6.action_disp = {"ACTION4": (0, 1)}; ep6.timeline = tl3
    tl3.append(tg1, "ACTION4", tgJ, 0.0, 0)
    assert not ep6.certify()
    for _ in range(ExecPlanner.EXP_TRIES):
        assert ep6.act(tg1, ["ACTION4"], enc) == "ACTION4"   # issue the experiment
        tl3.append(tg1, "ACTION4", tgJ, 0.0, 0)              # ...jump reproduces
        ep6.update(tg1, "ACTION4", tgJ, enc)
    ep6.act(tg1, ["ACTION4"], enc)                # tries exhausted -> accept anomaly
    assert ((3, 2), "ACTION4") in ep6._anomalies, "reproduced contradiction -> anomaly"
    assert ep6.certify(), "anomalous (cell, action) must be out of certification scope"
    # (5) VIGA/WorldCoder joint revision: 'predicted-wall but reality walked
    #     through' indicts the REPRESENTATION (the colour-obstacle abstraction),
    #     not the rule -- the colour is un-marked, no anomaly is recorded, and
    #     the whole record re-certifies green under the revised grounding.
    gV1 = np.zeros((8, 10), dtype=np.int32)
    gV1[3:5, 2:4] = 2       # avatar 2x2 (scale 2), centroid (3.5, 2.5)
    gV1[3:5, 4:6] = 8       # colour marked as wall -- actually walkable (a cart)
    gV2 = np.zeros((8, 10), dtype=np.int32)
    gV2[3:5, 4:6] = 2       # the avatar drove straight onto it (dc = +2)
    tl4 = Timeline()
    ep7 = ExecPlanner(); ep7.CERT_REPLAY_GAP = 0
    ep7.avatar_color = 2; ep7.action_disp = {"ACTION4": (0, 2)}
    ep7.obstacle_colors = {8}; ep7.timeline = tl4
    tl4.append(gV1, "ACTION4", gV2, 0.0, 0)
    assert not ep7.certify(), "walked-through-'wall' must be a counterexample"
    for _ in range(ExecPlanner.EXP_TRIES):
        assert ep7.act(gV1, ["ACTION4"], enc) == "ACTION4"   # experiment re-tests
        tl4.append(gV1, "ACTION4", gV2, 0.0, 0)              # ...and it reproduces
        ep7.update(gV1, "ACTION4", gV2, enc)
    ep7.act(gV1, ["ACTION4"], enc)                # exhausted -> revise, not anomaly
    assert 8 not in ep7.obstacle_colors, "the colour-obstacle grounding must be indicted"
    assert not ep7._anomalies, "representation revision must pre-empt the anomaly"
    assert ep7.certify(), "the record must re-certify green under the revised grounding"

    # --- Schema Phase 2: budgeted LLM theorizer, backtest-gated adoption ---
    # (a) REJECT: the record shows a move into colour 7 was VETOED (avatar 2x2,
    #     disp (0,2) -- only "7 is a wall" explains the veto). A proposal
    #     un-marking 7 increases mismatches and must be reverted wholesale.
    gP1 = np.zeros((8, 10), dtype=np.int32)
    gP1[3:5, 2:4] = 2       # avatar 2x2 (scale 2)
    gP1[3:5, 4:6] = 7       # a REAL wall
    tlP = Timeline()
    ep8 = ExecPlanner(); ep8.CERT_REPLAY_GAP = 0
    ep8.avatar_color = 2; ep8.action_disp = {"ACTION4": (0, 2)}
    ep8.obstacle_colors = {7}; ep8.timeline = tlP
    tlP.append(gP1, "ACTION4", gP1, 0.0, 0)       # veto: avatar stayed put
    assert ep8.certify()
    assert not ep8.propose_theory({"not_obstacle_colors": [7], "goal": None,
                                   "decoy_colors": [9]}, gP1), \
        "a proposal the record contradicts must be rejected"
    assert 7 in ep8.obstacle_colors and 9 not in ep8.non_goal_colors, \
        "a rejected proposal must be reverted wholesale (no partial credit)"
    # (b) ACCEPT: the record shows the avatar drove THROUGH colour 8 (red);
    #     un-marking 8 explains the record -> adopted, and the goal hint lands.
    tlQ = Timeline()
    ep9 = ExecPlanner(); ep9.CERT_REPLAY_GAP = 0
    ep9.avatar_color = 2; ep9.action_disp = {"ACTION4": (0, 2)}
    ep9.obstacle_colors = {8}; ep9.timeline = tlQ
    tlQ.append(gV1, "ACTION4", gV2, 0.0, 0)       # walked through the "wall"
    assert not ep9.certify()
    assert ep9.propose_theory({"not_obstacle_colors": [8], "goal": [1, 8],
                               "decoy_colors": [9]}, gV1)
    assert 8 not in ep9.obstacle_colors and 9 in ep9.non_goal_colors
    assert ep9.certify(), "adoption must re-certify against the full record"
    assert ep9._llm_goal == (1, 8)
    # (c) The adopted goal hint REDIRECTS planning; reaching the hinted cell
    #     without a level-up drops it (reality outranks the hypothesis).
    epA = ExecPlanner(); epA.avatar_color = 2; epA.action_disp = dict(DISP)
    hb = np.zeros((12, 12), dtype=np.int32)
    hb[0, :] = hb[-1, :] = hb[:, 0] = hb[:, -1] = 5
    hb[6, 2] = 2                                  # avatar; NO distinctive goal colour
    assert epA.propose_theory({"goal": [6, 9]}, hb)
    aH = epA.act(hb, M4, enc)
    assert aH == "ACTION4", f"the hinted goal must drive planning, got {aH}"
    assert epA._target == (6, 9)
    hb2 = np.zeros((12, 12), dtype=np.int32)
    hb2[0, :] = hb2[-1, :] = hb2[:, 0] = hb2[:, -1] = 5
    hb2[6, 9] = 2                                 # avatar AT the hinted cell
    epA.act(hb2, M4, enc)
    assert epA._llm_goal is None, "reaching the hint without reward must drop it"
    # (d) LocalLLM.theorize parses a JSON proposal out of noisy generation.
    class _StubLLM(LocalLLM):
        def __init__(self):
            super().__init__(model_path="/nonexistent")
        def generate(self, *a, **k):
            return ('noise {"hypothesis": "h", "goal": [3, 4], '
                    '"not_obstacle_colors": [8]} tail')
    resT = _StubLLM().theorize("....", "t", "e", "stuck")
    assert resT and resT["goal"] == [3, 4] and resT["not_obstacle_colors"] == [8]

    # --- ClickPlanner: reward-exploit > coverage > active-safe > dead, avoid lethal ---
    # 8x8 grid at QUANT=4 -> exactly 4 lattice keys (0,0),(0,1),(1,0),(1,1); check the
    # quantized KEY of each pick (object-centre coords may share a key, so don't assert
    # exact px). New contract differs from the old one on purpose (see class docstring).
    cg = np.zeros((8, 8), dtype=np.int32)
    keyof = lambda s: ClickPlanner()._key(*ClickPlanner._parse(s))
    # (1) A cell that PAID OFF (reward) is re-selected over untried coverage cells.
    cp = ClickPlanner()
    cp.update("ACTION6_r2_c2", changed=True, reward=1.0)      # key (0,0) rewarded
    assert keyof(cp.act(cg, enc, epsilon=0.0)) == (0, 0), "reward cell must be exploited"
    # (2) Among non-rewarding TRIED cells (all lattice keys used up), active > dead.
    cp2 = ClickPlanner()
    cp2.update("ACTION6_r2_c2", changed=False, reward=0.0)    # (0,0) dead
    cp2.update("ACTION6_r2_c6", changed=True, reward=0.0)     # (0,1) active
    cp2.update("ACTION6_r6_c2", changed=False, reward=0.0)    # (1,0) dead
    cp2.update("ACTION6_r6_c6", changed=False, reward=0.0)    # (1,1) dead
    assert keyof(cp2.act(cg, enc, epsilon=0.0)) == (0, 1), "active cell preferred over dead"
    # (3) A lethal (loss-marked) active cell loses to an equally-active clean cell.
    cp3 = ClickPlanner()
    cp3.update("ACTION6_r2_c2", changed=True, reward=0.0); cp3.mark_loss("ACTION6_r2_c2")  # (0,0) active+lethal
    cp3.update("ACTION6_r2_c6", changed=True, reward=0.0)     # (0,1) active clean
    cp3.update("ACTION6_r6_c2", changed=False, reward=0.0)    # (1,0) dead
    cp3.update("ACTION6_r6_c6", changed=False, reward=0.0)    # (1,1) dead
    assert keyof(cp3.act(cg, enc, epsilon=0.0)) == (0, 1), "lethal cell must be avoided"
    # (4) Intra-level reset() KEEPS board knowledge; new_level() wipes it.
    cp3.reset();      assert cp3._stats, "reset() must preserve click stats across restarts"
    cp3.new_level();  assert not cp3._stats, "new_level() must wipe stats on real level-up"

    # --- ClickPlanner state-graph BFS: env-as-simulator world model over buttons ---
    # Mirror real lp85: two well-separated buttons, each a MULTI-CELL sprite whose
    # several reactive coords must cluster into ONE button (raw coords over-count
    # buttons ~3x, which throttled the old blind product-BFS).
    A, B = "ACTION6_r30_c2", "ACTION6_r30_c58"     # cluster representatives (interior pixels)

    def _register_buttons(cp):
        # button B (RIGHT sprite, cols ~58): two adjacent reactive cells -> 1 cluster
        for _ in range(20): cp.update("ACTION6_r30_c58", changed=True, reward=0.0)
        for _ in range(20): cp.update("ACTION6_r34_c58", changed=True, reward=0.0)
        # button A (LEFT sprite, cols ~2-6): three adjacent cells -> 1 cluster
        for _ in range(20): cp.update("ACTION6_r30_c2", changed=True, reward=0.0)
        for _ in range(20): cp.update("ACTION6_r30_c6", changed=True, reward=0.0)
        for _ in range(20): cp.update("ACTION6_r34_c2", changed=True, reward=0.0)

    cps = ClickPlanner()
    gg = np.zeros((64, 64), dtype=np.int32)
    assert not cps.searching and cps.act(gg, enc, 0.0).startswith("ACTION6_r")  # coverage until discovered
    _register_buttons(cps)
    assert cps._detect_buttons() == [(30, 2), (30, 58)], \
        f"multi-cell sprites must cluster to 2 buttons, got {cps._detect_buttons()}"

    # A deterministic 1-D slide puzzle as the "env": button A (index 0) slides toward
    # position 0 and PARKS there (a wall fixed point); button B (index 1) slides toward
    # goal position 3 and parks there. RESET returns to position 0 (the root). Each
    # position renders to a distinct grid so states hash apart. This is the exact shape
    # the state-graph BFS must handle: no-op parked clicks + reconverging paths that a
    # blind product-BFS would re-explore combinatorially.
    GOAL = 3
    pos = [0]
    def _sim_grid():
        g = np.zeros((64, 64), dtype=np.int32); g[0, 0] = pos[0]; return g
    def _apply(a):
        if a == "RESET":            pos[0] = 0
        elif a == A:                pos[0] = max(pos[0] - 1, 0)       # slide left, park at 0
        elif a == B:                pos[0] = min(pos[0] + 1, GOAL)    # slide right, park at goal

    emitted = []
    for _ in range(400):
        a = cps.act(_sim_grid(), enc, 0.0)
        emitted.append(a)
        _apply(a)
        if cps._search_done:
            break
    assert emitted[0] == "RESET", f"search must establish the root via a RESET first, got {emitted[0]}"
    assert A in emitted and B in emitted, "both buttons must be probed by the BFS"
    # Dedup is the whole point: 4 reachable boards (positions 0..3) despite the parked
    # no-op re-clicks and the A/B paths that reconverge. A blind product-BFS would keep
    # re-expanding those; the state graph visits each distinct board exactly once.
    assert len(cps._dist) == 4, f"state graph must dedup to 4 distinct boards, got {len(cps._dist)}"
    assert cps._search_done and not cps.searching, "BFS must exhaust the reachable graph then yield to coverage"
    # Every discovered edge is deterministic-correct: from every non-root board, button A
    # steps one closer to 0 (recorded child hash matches the pos-1 board).
    root_h = cps._root
    assert cps._dist[root_h] == () and cps._G[root_h][1] != root_h, "B from root must reach a NEW board"

    # Post-exhaustion the planner is a pure coverage clicker again (never spins on RESET).
    assert cps.act(_sim_grid(), enc, 0.0).startswith("ACTION6_r"), "exhausted search -> coverage clicks"

    # GAME_OVER mid-replay: reset() must blame the final (parent, button) edge as dead
    # so the BFS never retries the losing click, and drop the interrupted plan.
    cpd = ClickPlanner(); _register_buttons(cpd)
    cpd._enter_search(cpd._detect_buttons())
    cpd.act(np.zeros((64, 64), dtype=np.int32), enc, 0.0)         # emits RESET (root plan)
    cpd.act(_sim_grid(), enc, 0.0)                                 # processes root, stages an expand plan
    assert cpd._pending is not None and cpd._pending[0] == "expand", "should be probing an edge"
    parent, b = cpd._pending[1], cpd._pending[2]
    cpd.reset()                                                    # simulate GAME_OVER on that edge
    assert (parent, b) in cpd._dead_edges, "a fatal replay edge must be marked dead"
    assert cpd._plan == [] and cpd._pending is None, "the interrupted plan must be dropped"

    # Button positions persist across level-up (UI arrows are level-invariant), but a
    # new level starts in COVERAGE, never search: a search RESET as the level's first
    # action is a FULL reset that wipes levels_completed (the lp85 0/8 regression).
    cps.new_level()
    assert cps._buttons == [(30, 2), (30, 58)] and not cps.searching, "new level must not start in search"
    assert cps.act(gg, enc, 0.0).startswith("ACTION6_r"), "first action of a level must be a click, not RESET"
    # search re-engages only after re-accumulating discover clicks (>=1 click => the
    # engine's action_count>0, so a later RESET is a level_reset that keeps the score).
    for _ in range(cps.MIN_DISCOVER_CLICKS):
        cps.update("ACTION6_r30_c2", changed=True, reward=0.0)
    assert cps.act(gg, enc, 0.0) == "RESET" and cps.searching, "search resumes after coverage budget"

    # --- StateGraph + DeadActionTracker (COMPONENT 5.8) ---
    sg = StateGraph()
    g0 = np.zeros((4, 4), dtype=np.int32)
    g1 = g0.copy(); g1[0, 0] = 1
    g2 = g0.copy(); g2[1, 1] = 2
    h0 = sg._hash(g0)
    sg.ensure_state(g0, ["ACTION1", "ACTION2", "ACTION6"])
    assert "ACTION6" not in sg.untested[h0], "clicks must never enter the graph"
    res = sg.nearest_untested_grid(g0)
    assert res is not None and res[0] == [] and res[1] in ("ACTION1", "ACTION2"), \
        f"current state's own untested action first (tie-break random), got {res}"
    sg.record(g0, "ACTION1", g1)                                  # h0 --A1--> h1
    sg.ensure_state(g1, ["ACTION1", "ACTION2"])
    sg.record(g1, "ACTION1", g2)                                  # h1 --A1--> h2
    res = sg.nearest_untested_grid(g0)
    assert res == ([], "ACTION2"), "untested at the CURRENT state beats a walk"
    sg.record(g0, "ACTION2", g0)                                  # A2 inert at h0 (self-loop)
    # h0 exhausted -> BFS must walk the proven A1 edge to h1, where A2 is untested.
    res = sg.nearest_untested_grid(g0)
    assert res == (["ACTION1"], "ACTION2"), f"BFS to nearest untested failed: {res}"

    # HUD normalization: a cell mutating on EVERY transition (timer / budget bar)
    # must be masked out of the hash, so world-identical frames collapse to ONE
    # state and inert moves read as self-loops instead of endless novelty.
    sgh = StateGraph()
    def _hud(world, t):
        g = np.zeros((6, 6), dtype=np.int32); g[0, 0] = world; g[5, 5] = t; return g
    frames = [_hud(0, t) for t in range(StateGraph.MASK_CHECK_EVERY + 2)]
    acts3 = ["ACTION1", "ACTION2", "ACTION3"]
    sgh.ensure_state(frames[0], acts3)
    for t in range(StateGraph.MASK_CHECK_EVERY):    # A1/A2 alternate, both only tick HUD
        sgh.record(frames[t], "ACTION1" if t % 2 == 0 else "ACTION2", frames[t + 1])
        sgh.ensure_state(frames[t + 1], acts3)
    assert sgh._mask is not None and sgh._mask[5, 5] and not sgh._mask[0, 0], \
        "the always-changing HUD cell (and only it) must be masked"
    assert len(sgh.untested) == 1, \
        f"world-identical frames must collapse to 1 state, got {len(sgh.untested)}"
    assert not sgh.changed_masked(frames[0], frames[1]), "HUD-only change = no world change"
    wchg = frames[0].copy(); wchg[0, 0] = 9
    assert sgh.changed_masked(frames[0], wchg), "a real world change must still count"
    res = sgh.nearest_untested_grid(frames[-1])
    assert res == ([], "ACTION3"), f"collapsed state: A1/A2 tested (self-loops), A3 next, got {res}"
    # HUD strip: a draining budget bar changes a DIFFERENT cell each transition
    # (no single cell crosses MASK_RATE) but its row changes every time -> the
    # whole row is masked and world-identical frames still collapse.
    sgb = StateGraph()
    def _bar(world, t):
        g = np.zeros((6, 40), dtype=np.int32); g[0, 0] = world
        g[5, :40 - t] = 8                                         # bar drains right-to-left
        return g
    bframes = [_bar(0, t) for t in range(StateGraph.MASK_CHECK_EVERY + 2)]
    sgb.ensure_state(bframes[0], ["ACTION1", "ACTION2"])
    for t in range(StateGraph.MASK_CHECK_EVERY):    # bar drains under BOTH actions
        sgb.record(bframes[t], "ACTION1" if t % 2 == 0 else "ACTION2", bframes[t + 1])
        sgb.ensure_state(bframes[t + 1], ["ACTION1", "ACTION2"])
    assert sgb._mask is not None and sgb._mask[5].all() and not sgb._mask[0, 0], \
        "the draining-bar row (and not the world) must be masked"
    assert len(sgb.untested) == 1, \
        f"bar-only changes must collapse to 1 state, got {len(sgb.untested)}"
    # Policy-conditioned world changes must NOT be masked: a sprite cycled by A1
    # but untouched by A2 changes under only ONE action -> it is world, not HUD.
    sgp = StateGraph()
    spr = 0
    a1_steps = {0, 1, 3, 6, 10, 15, 21}     # irregular gaps -> not a periodic clock
    for t in range(StateGraph.MASK_CHECK_EVERY):
        a = np.zeros((6, 6), dtype=np.int32); a[2, 2] = 3 + spr
        b = a.copy()
        if t in a1_steps:
            spr = 1 - spr
            b[2, 2] = 3 + spr                           # A1 cycles the sprite
        sgp.record(a, "ACTION1" if t in a1_steps else "ACTION2", b)  # A2 quiet
    assert sgp._mask is None or not sgp._mask[2, 2], \
        "a cell quiet under one well-sampled action must stay unmasked"

    # --- PatchWorldModel (COMPONENT 5.9): local rules predict untested actions ---
    pwm = PatchWorldModel()
    def _cor(n_track, pos):
        # corridor: track cells (2) at cols 1..n_track, avatar (7) at pos, row 3
        g = np.zeros((7, 15), dtype=np.int32)
        g[3, 1:n_track + 1] = 2
        g[3, pos] = 7
        return g
    for _ in range(PatchWorldModel.MIN_SEEN):
        for p in range(1, 8):                         # long corridor: valid right-moves
            pwm.learn(_cor(10, p), "ACTION4", _cor(10, p + 1))
        pwm.learn(_cor(5, 5), "ACTION4", _cor(5, 5))  # short corridor: blocked at end
    assert not pwm.predict_noop(_cor(10, 9), "ACTION4"), \
        "a move onto track must not be predicted as a no-op"
    assert pwm.predict_noop(_cor(10, 10), "ACTION4"), \
        "blocked-at-wall must generalize from the short corridor to an unseen position"
    assert not pwm.predict_noop(np.random.randint(0, 9, (7, 15)), "ACTION4"), \
        "unknown patches must never be called no-ops"
    # Graph integration: predicted no-ops are tested by imagination (unscored),
    # and a noop-less pass re-offers them (completeness fallback).
    sgn = StateGraph()
    gN = np.zeros((4, 4), dtype=np.int32)
    sgn.ensure_state(gN, ["ACTION3", "ACTION4"])
    hN = sgn._hash(gN)
    res = sgn.nearest_untested(hN, noop=lambda g, a: a == "ACTION4")
    assert res == ([], "ACTION3"), f"predicted no-op must be skipped, got {res}"
    sgn.record(gN, "ACTION3", gN)
    assert sgn.nearest_untested(hN, noop=lambda g, a: a == "ACTION4") is None, \
        "imagination-pruned graph reads exhausted"
    assert sgn.nearest_untested(hN) == ([], "ACTION4"), \
        "fallback without the model re-offers the skipped pair"

    # A globally-dead action is excluded from graph suggestions.
    dt = DeadActionTracker()
    for _ in range(DeadActionTracker.MIN_TRIALS):
        dt.update("ACTION2", changed=False)
    assert dt.is_dead("ACTION2")
    assert sg.nearest_untested(h0, dt) is None, "dead actions must not be suggested"
    dt.update("ACTION2", changed=True)                            # one change whitelists it
    assert not dt.is_dead("ACTION2")
    for _ in range(9):
        dt.update("ACTION6_r1_c1", changed=False)
    assert not dt.is_dead("ACTION6_r1_c1"), "clicks are never globally dead"

    # --- Game-type classifier: mechanical games bypass the displacement planner ---
    rpm = ReactivePlanner()
    base_g = np.zeros((16, 16), dtype=np.int32); base_g[4, 4] = 3
    for k in range(8):        # 8 changed frames, none a clean translation (cycling)
        nxt_g = np.zeros((16, 16), dtype=np.int32); nxt_g[4, 4] = 3 + (k % 2) + 1
        rpm.update(base_g, "ACTION1", nxt_g, enc)
        base_g = nxt_g
    assert rpm.looks_mechanical(), "8/8 unexplained changes must classify as mechanical"
    rpw = ReactivePlanner()   # clean translations -> walk-to-goal, planner stays on
    wg = np.zeros((12, 12), dtype=np.int32); wg[2:4, 2:4] = 2
    for _ in range(8):
        nw = np.roll(wg, 2, axis=1)
        rpw.update(wg, "ACTION4", nw, enc)
        wg = nw
    assert not rpw.looks_mechanical(), "clean translations must NOT classify as mechanical"
    # A wrapping selector (tr87 cursor) LOOKS like clean translations, but its
    # displacement flips sign at every wrap -> contradictions flag it mechanical.
    rpc = ReactivePlanner()
    cg = np.zeros((12, 40), dtype=np.int32)
    pos = [2, 12, 22]         # 3 selector slots; cursor wraps 22 -> 2
    for k in range(12):
        a = np.zeros_like(cg); a[5, pos[k % 3]:pos[k % 3] + 8] = 6
        b = np.zeros_like(cg); b[5, pos[(k + 1) % 3]:pos[(k + 1) % 3] + 8] = 6
        rpc.update(a, "ACTION3", b, enc)
    assert rpc.disp_contradictions >= 3 and rpc.looks_mechanical(), \
        f"sign-flipping selector must classify mechanical ({rpc.disp_contradictions} flips)"

    # --- Span-based goal candidacy: structure never outranks a discrete object ---
    rps = ReactivePlanner(); rps.avatar_color = 2
    sb = np.zeros((20, 20), dtype=np.int32)
    sb[10, :] = 7            # full-width interior wall (span 100%): structural
    sb[2, 2] = 2             # avatar
    sb[17, 17] = 4           # the real goal object -- far away, 1 pixel
    tgt = rps._nearest_target(sb, 2.0, 2.0)
    assert tgt == (17, 17), f"structural colour must be demoted, got {tgt}"
    # ...but on a board with ONLY structure left, it is still a last-resort target.
    sb2 = np.zeros((20, 20), dtype=np.int32); sb2[10, :] = 7; sb2[2, 2] = 2
    assert rps._nearest_target(sb2, 2.0, 2.0) is not None, "structure stays a last resort"

    # DSL primitives + no-exec compile
    prog = [{"action": "*", "op": "translate", "args": {"dr": 1, "dc": 0}}]
    fn = compile_program(prog, enc)
    assert fn is not None and np.array_equal(fn(g, "ACTION1")[3, 2:5], np.array([1, 1, 1]))
    assert compile_program([{"op": "no_such_op"}], enc) is None

    # Object-scoped ops
    og = np.zeros((6, 6), dtype=np.int32); og[1, 1] = 2; og[4, 4] = 3
    moved = GridDSL.translate_color(og, 0, color=2, dr=0, dc=1)
    assert moved[1, 2] == 2 and moved[1, 1] == 0 and moved[4, 4] == 3   # obstacle untouched
    slid = GridDSL.slide_color(og, 0, color=2, direction="down")
    assert slid[5, 1] == 2 and slid[1, 1] == 0                          # slid to the wall
    blocked = np.zeros((6, 6), dtype=np.int32); blocked[0, 0] = 2; blocked[3, 0] = 5
    slid2 = GridDSL.slide_color(blocked, 0, color=2, direction="down")
    assert slid2[2, 0] == 2 and slid2[3, 0] == 5                        # stops above the 5
    swapped = GridDSL.swap_colors(og, 0, a=2, b=3)
    assert swapped[1, 1] == 3 and swapped[4, 4] == 2

    # Synthesis trigger (Issue 2): 4 same-action OR 8 total with >=2 actions of >=2
    def _t(act):
        p = np.zeros((2, 2), dtype=np.int32); n = np.ones((2, 2), dtype=np.int32)
        return Transition(p, act, n, 0.0, "", time.time())
    assert synth_ready([_t("ACTION1")] * 3) is False
    assert synth_ready([_t("ACTION1")] * 4) is True
    assert synth_ready([_t("ACTION1")] * 3 + [_t("ACTION2")] * 3) is False       # 6 total
    assert synth_ready([_t(f"ACTION{i}") for i in (1, 2, 3, 4)] * 2) is True     # 8 total, 4x2

    # Build a ground-truth "ACTION3 = gravity down" world and recover it WITHOUT any LLM
    truth = compile_program([{"action": "ACTION3", "op": "gravity", "args": {"direction": "down"}}], enc)
    trans, cur = [], np.zeros((8, 8), dtype=np.int32)
    for i in range(6):
        cur = cur.copy(); cur[0, i % 8] = (i % 3) + 1     # scatter some pixels up top
        nxt = truth(cur, "ACTION3")
        trans.append(Transition(cur, "ACTION3", nxt, 0.0, enc.encode_transition(cur, nxt, "ACTION3"), time.time()))
        cur = nxt
    syn = EnumerativeSynthesizer(enc)
    recovered = syn.synthesize(trans)
    assert recovered is not None and "gravity" in recovered.hypothesis, "enumerator failed to recover rule"

    # Recover an OBJECT-scoped rule: "ACTION2 moves the color-2 avatar right by 1"
    # (a static color-3 obstacle defeats the global translate hypothesis)
    ctrans = []
    for i in range(1, 6):
        p = np.zeros((8, 8), dtype=np.int32); p[3, i] = 2; p[7, 7] = 3
        n = np.zeros((8, 8), dtype=np.int32); n[3, i + 1] = 2; n[7, 7] = 3
        ctrans.append(Transition(p, "ACTION2", n, 0.0, enc.encode_transition(p, n, "ACTION2"), time.time()))
    rec2 = syn.synthesize(ctrans)
    assert rec2 is not None and rec2.spec[0]["op"] == "translate_color" and rec2.spec[0]["args"]["color"] == 2, \
        f"expected translate_color, got {rec2.spec if rec2 else None}"

    # Verifier: correct model promotes, identity trap rejected (roadmap V2)
    wmm = WorldModelManager(LocalLLM(model_path="/nonexistent"), enc)
    assert wmm.llm.available is False                      # missing model dir -> LLM cleanly disabled
    assert wmm.verify(recovered.fn, trans) >= VERIFY_PROMOTE_THRESHOLD
    identity = compile_program([{"action": "*", "op": "identity"}], enc)
    assert wmm.verify(identity, trans) < VERIFY_PROMOTE_THRESHOLD
    assert wmm.try_enumerative(trans) is True and wmm.active_model is not None

    # Demotion bans the rule so enumeration must find a DIFFERENT hypothesis
    wmm.demote(wmm.active_model)
    assert wmm.active_model is None and len(wmm.banned_rules) > 0
    again = wmm.enumerator.synthesize(trans, wmm.banned_rules)
    assert again is None or again.spec != recovered.spec

    # ExploitGate hysteresis (Issue 4): 5 consecutive hits to enter, <0.5 avg to exit
    gate = ExploitGate()
    for _ in range(4):
        gate.update(1.0)
    assert not gate.should_exploit()
    gate.update(1.0)
    assert gate.should_exploit()
    for _ in range(10):
        gate.update(0.0)
    assert not gate.should_exploit()

    # Epsilon decay (Issue 5)
    assert MyAgent._epsilon_for(0) == EPSILON_START
    assert MyAgent._epsilon_for(10_000) == EPSILON_FLOOR
    assert MyAgent._epsilon_for(50) < MyAgent._epsilon_for(10)

    # MCTS returns a valid action, ACTION6 as one node, no LLM
    wmm2 = WorldModelManager(LocalLLM(model_path="/nonexistent"), enc)
    assert wmm2.try_enumerative(trans) is True
    planner = MCTSPlanner(wmm2, enc, StateCounter())
    act = planner.search(g, ["ACTION1", "ACTION2", "ACTION6"], n_iterations=20)
    assert act.startswith("ACTION")

    # Agent-level checks (stand-in Agent, no LLM/embedder needed)
    agent = MyAgent(game_id="local-test")
    assert agent.llm.available is False or True            # constructed either way

    # parse_action: ACTION6 coordinates, x=col / y=row
    a6 = agent.parse_action("ACTION6_r3_c5")
    assert a6 == GameAction.ACTION6 and agent.action_x == 5 and agent.action_y == 3

    # _valid_actions accepts enums / ints / strings, drops RESET and junk
    class _F: pass
    fr = _F(); fr.available_actions = [1, 2, "ACTION3", "RESET", "JUNK"]
    assert agent._valid_actions(fr) == ["ACTION1", "ACTION2", "ACTION3"]

    # _read_score prefers arcengine >=0.9.3 levels_completed, falls back to score
    fs = _F(); fs.levels_completed = 3
    assert MyAgent._read_score(fs) == 3
    fs2 = _F(); fs2.score = 2
    assert MyAgent._read_score(fs2) == 2
    assert MyAgent._read_score(_F()) == 0

    # _parse_grid handles arcengine's 3D frame (list of animation grids)
    fr2 = _F(); fr2.frame = [np.zeros((4, 4), dtype=int).tolist(), (np.ones((4, 4), dtype=int) * 7).tolist()]
    got = agent._parse_grid(fr2, [fr2])
    assert got is not None and got[0, 0] == 7

    # is_done (WIN): records the CAUSING transition with the REAL final grid
    prev = np.zeros((4, 4), dtype=np.int32); prev[0, 0] = 2
    final = np.zeros((4, 4), dtype=np.int32); final[3, 3] = 2
    agent.last_grid = prev; agent.last_action_str = "ACTION5"
    fwin = _F(); fwin.state = GameState.WIN; fwin.frame = final.tolist(); fwin.levels_completed = 1
    assert agent.is_done([fwin], fwin) is True
    t_last = agent.transitions[-1]
    assert t_last.reward == 1.0 and np.array_equal(t_last.next, final) and t_last.action == "ACTION5"
    n_before = len(agent.transitions)
    agent.is_done([fwin], fwin)                            # second call must not double-record
    assert len(agent.transitions) == n_before

    # is_done (WIN, no grid in frame): retro-tags the previous transition's reward
    agent2 = MyAgent(game_id="local-test-2")
    agent2._record_transition(prev, "ACTION4", final, 0.0)
    agent2.last_grid = final; agent2.last_action_str = "ACTION4"
    fwin2 = _F(); fwin2.state = GameState.WIN               # no .frame / .grid at all
    assert agent2.is_done([fwin2], fwin2) is True
    assert agent2.transitions[-1].reward == 1.0

    # Synth watchdog (Issue 6): stale inflight flag is cleared, generation bumped
    agent.synth_inflight = True
    agent.synth_started_at = time.time() - SYNTH_MAX_INFLIGHT_S - 1
    gen0 = agent.synth_generation
    agent._synth_watchdog()
    assert agent.synth_inflight is False and agent.synth_generation == gen0 + 1

    # --- LLM click-reasoning plan: trigger -> background fill -> ordered consume ---
    # The real 27B never loads in local eval (its path is a Kaggle mount), so this is
    # the ONLY place the click-plan pipeline is exercised: a mock LLM stands in for it.
    class _MockLLM:
        available = True
        def is_ready(self): return True
        def plan_clicks(self, board_text, observations, reactive, hypothesis):
            # returns one in-range and one OUT-of-range coord to prove clamping
            return [(5, 7), (2, 3), (999, -4)]

    ca = MyAgent(game_id="click-plan-test")
    ca.llm = _MockLLM()
    ca.level_steps = 100                                   # past the throttle interval
    ca.cplanner._total_clicks = ca._CLICK_PLAN_MIN_OBS     # enough evidence to reason
    board = np.zeros((32, 32), dtype=np.int32); board[5, 7] = 4; board[2, 3] = 6

    # grid->text keeps pixel coords and marks background as '.'
    txt = ca._grid_to_text(board)
    assert txt.count("\n") == 31 and txt.splitlines()[5][7] != "." and txt.splitlines()[0][0] == "."

    ca._maybe_request_click_plan(board)
    t_wait = time.time()
    while ca._click_inflight and time.time() - t_wait < 5:
        time.sleep(0.01)
    assert not ca._click_inflight and len(ca._click_plan) == 3, "plan must be filled off-thread"
    assert ca.llm_calls_this_level == 1, "a plan request must consume exactly one budget unit"
    # consumed in ORDER, out-of-range coord clamped into [0,31]
    assert ca._next_click_from_plan(board) == "ACTION6_r5_c7"
    assert ca._next_click_from_plan(board) == "ACTION6_r2_c3"
    assert ca._next_click_from_plan(board) == "ACTION6_r31_c0"     # (999,-4) clamped
    assert ca._next_click_from_plan(board) is None                 # exhausted -> caller falls to coverage

    # budget exhausted => no new request fires (coverage floor stays in control)
    ca._discard_click_plan()
    ca.llm_calls_this_level = LLM_CALL_BUDGET_PER_LEVEL
    ca.level_steps += ca._CLICK_PLAN_MIN_INTERVAL + 1
    ca._maybe_request_click_plan(board)
    assert not ca._click_inflight and not ca._click_plan, "no plan when budget is spent"

    # a real (absent) LLM never triggers the pipeline: is_ready() is False locally
    ca.llm = get_local_llm()
    ca.llm_calls_this_level = 0
    ca._maybe_request_click_plan(board)
    assert not ca._click_plan, "LLM-absent must leave coverage as the sole click policy"

    # stale-start guard: a plan reasoned about board A must be DROPPED if the live
    # board has changed by the time it lands (coverage mutated it while generating).
    ca._click_plan = deque([(1, 1), (2, 2)])
    ca._click_src_hash = int(hash((board.shape, board.tobytes())))
    other = board.copy(); other[0, 0] = 9
    assert ca._next_click_from_plan(other) is None and not ca._click_plan, \
        "a plan must not execute from a board it wasn't reasoned about"
    # matching start validates ONCE, then ordered clicks proceed as the board mutates
    ca._click_plan = deque([(1, 1), (2, 2)])
    ca._click_src_hash = int(hash((board.shape, board.tobytes())))
    assert ca._next_click_from_plan(board) == "ACTION6_r1_c1"       # validates on match
    assert ca._next_click_from_plan(other) == "ACTION6_r2_c2"       # board moved -> still runs
    # lethal target in a plan is skipped (never scores below the coverage floor)
    ca.cplanner.update("ACTION6_r3_c3", changed=True, reward=0.0); ca.cplanner.mark_loss("ACTION6_r3_c3")
    ca._click_plan = deque([(3, 3), (7, 7)]); ca._click_src_hash = None
    assert ca._next_click_from_plan(board) == "ACTION6_r7_c7", "a known-lethal target must be skipped"

    # --- Blocker 2: the choose_action (b2) wiring, driven with a mock LLM ---
    class _CF:                                    # minimal click-only frame
        def __init__(self, g, lvl=0, st=None):
            self.frame = g.tolist(); self.available_actions = [6]
            self.levels_completed = lvl
            self.state = st if st is not None else GameState.NOT_FINISHED
    cb = MyAgent(game_id="click-choose-test")
    cb.llm = _MockLLM()
    fg = np.zeros((16, 16), dtype=np.int32); fg[4, 4] = 2
    # (i) a filled plan is emitted BY choose_action as a real ACTION6 click
    cb._click_plan = deque([(8, 8)])
    cb._click_src_hash = int(hash((fg.shape, fg.tobytes())))
    out = cb.choose_action([_CF(fg)], _CF(fg))
    assert out == GameAction.ACTION6 and cb.action_x == 8 and cb.action_y == 8, \
        "choose_action must emit the planned click for a click-only game"
    # (ii) when the ClickPlanner SEARCH is driving, the LLM plan is suppressed
    cs = MyAgent(game_id="click-suppress-test"); cs.llm = _MockLLM()
    _register_buttons(cs.cplanner); cs.cplanner._enter_search(cs.cplanner._detect_buttons())
    assert cs.cplanner.searching
    cs._click_plan = deque([(8, 8)])
    _ = cs.choose_action([_CF(fg)], _CF(fg))
    assert list(cs._click_plan) == [(8, 8)], "search must not consume the LLM plan (one layer at a time)"
    # (iii) a real level-up clears any pending plan
    cl = MyAgent(game_id="click-levelup-test"); cl.llm = _MockLLM()
    cl.last_score = 0; cl.last_grid = fg.copy(); cl.last_action_str = "ACTION6_r8_c8"
    cl._click_plan = deque([(8, 8)])
    _ = cl.choose_action([_CF(fg, lvl=1)], _CF(fg, lvl=1))
    assert not cl._click_plan, "level-up must discard a stale plan from the previous layout"

    print("All isolation tests passed. Agent iteration 6 is ready (offline / in-process).")


Writing /kaggle/working/my_agent.py


# Testing

In [5]:
%%writefile /kaggle/working/Kaggle_test.py
"""Kaggle_test.py -- 5-game accuracy/RHAE check for my_agent.py, LLM-free.

WHY THIS FILE EXISTS
    The submission notebook writes the agent with `%%writefile /kaggle/working/my_agent.py`.
    Locally (and in a plain Kaggle CPU/GPU session without the model datasets) the
    Qwen ~27B cannot load, but MyAgent degrades gracefully to its no-LLM planners --
    which are exactly the components that move the score. This cell runs those
    planners against 5 representative public games (one per game "type") and
    reports levels completed AND the real competition metric, RHAE.

HOW TO RUN (pick one)
    1) Notebook cell (easiest): paste this entire file into a cell AFTER the
       `%%writefile /kaggle/working/my_agent.py` cell and run it. Tune CONFIG below.
    2) Notebook, keeping it as a file:
           %%writefile /kaggle/working/Kaggle_test.py   (this file)
       then in the next cell:
           !PYTHONIOENCODING=utf-8 PYTHONHASHSEED=0 python -u /kaggle/working/Kaggle_test.py
    3) Locally, from ARC_AGI EXPERMENTATIONS/:
           PYTHONIOENCODING=utf-8 PYTHONHASHSEED=0 python -u Kaggle_test.py
    CLI overrides (2/3 only): --games lp85 tr87  --seed 0  --max-actions 1500

    Reproducibility note: MyAgent derives its RNG seed from ARC_AGENT_SEED +
    hash(game_id), and Python randomizes str hashes per process. Routes 2/3 pin it
    via PYTHONHASHSEED=0. Route 1 (in-kernel) cannot pin it retroactively, so runs
    are deterministic *within* one kernel session but may differ across restarts --
    a warning is printed when that applies.

GAME SOURCES
    Expects the public game files laid out as  <root>/<id>/<hash>/<id>.py  (the
    eval/real_games layout) or flat  <root>/<id>.py.  Roots searched, first hit:
        CONFIG["GAMES_DIR"], $ARC_GAMES_DIR, ./eval/real_games,
        /kaggle/working/real_games,
        /kaggle/input/competitions/*/environment_files   (the official files),
        /kaggle/input/*/real_games, /kaggle/input/*/*/real_games
    Human baselines (games_index.json) are found SEPARATELY -- the competition
    environment_files do NOT include them, so on Kaggle add
    eval/real_games/games_index.json to one of your datasets (any depth under
    /kaggle/input is found) or set CONFIG["BASELINES_FILE"]. Without it every
    level_score reads 0.

THE 5 DEFAULT GAMES (one per type)
    lp85  click-only [6]           ClickPlanner: stats + coverage + sequence search
    cn04  mixed [1-6]              the other level-banking game; regression sentinel
    wa30  pure movement [1-5]      ExecPlanner T1/T2 + Timeline/CERTIFY on a real game
    tr87  mechanical [1-4]         StateGraph + HUD masking (wrapping selectors)
    ls20  mechanical/movement edge Schema Phase-1 certify-loop boundary case

RHAE (the competition metric -- NOT level fraction)
    level_score = min(1.15, human_actions / ai_actions) ** 2      (cap ~1.32)
    game_score  = sum(level_score_i * i) / sum(i for i = 1..win_levels)
    overall     = mean over games. Squared ratio: 10x human actions -> ~1% score.
"""
import os
import sys
import glob
import json
import time
import random
import logging
import traceback

import numpy as np

# ============================== CONFIG ======================================
# Tune everything here. CLI flags (when run as a script) override these.
CONFIG = {
    # Which games to run. Any of the 25 public ids works if its files are present.
    "GAMES": ["lp85", "cn04", "wa30", "tr87", "ls20"],

    # RNG seed: seeds random / np.random and is exported as ARC_AGENT_SEED.
    "SEED": 0,

    # Per-game budget of REAL (scored) actions. Competition allows more, but
    # 1500 keeps the whole 5-game run ~20-25 min; raise for a fuller picture.
    "MAX_ACTIONS": 1500,

    # Per-game wall-clock cap in seconds (0 = no cap). Useful on Kaggle so one
    # slow game can't eat the session; the game is scored with whatever levels
    # it banked before the cap.
    "TIME_CAP_SECS": 0,

    # Where the game files live. "" = auto-discover (see GAME SOURCES above).
    "GAMES_DIR": "",

    # Path to games_index.json (human baselines). "" = auto-discover: games
    # root, ./eval/real_games, /kaggle/working, then anywhere under
    # /kaggle/input. Without baselines RHAE is 0 even for completed levels.
    "BASELINES_FILE": "",

    # Where my_agent.py lives. "" = auto: next to this file, else /kaggle/working.
    "AGENT_DIR": "",

    # Print a progress line every N actions during a game (0 = quiet).
    "PROGRESS_EVERY": 250,
}
# ============================================================================

try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:            # pasted into a notebook cell: no __file__
    HERE = os.getcwd()


def _in_notebook():
    return "ipykernel" in sys.modules


def _setup_paths(cfg):
    """Make `from my_agent import MyAgent` resolve, wherever we're running."""
    cands = [cfg["AGENT_DIR"], HERE, "/kaggle/working", os.getcwd()]
    for c in cands:
        if c and os.path.exists(os.path.join(c, "my_agent.py")):
            if c not in sys.path:
                sys.path.insert(0, c)
            return c
    raise FileNotFoundError(
        "my_agent.py not found. Run the %%writefile my_agent.py cell first, "
        "or set CONFIG['AGENT_DIR'].")


def find_games_root(cfg):
    cands = []
    if cfg["GAMES_DIR"]:
        cands.append(cfg["GAMES_DIR"])
    env = os.environ.get("ARC_GAMES_DIR")
    if env:
        cands.append(env)
    cands.append(os.path.join(HERE, "eval", "real_games"))
    cands.append("/kaggle/working/real_games")
    cands += sorted(glob.glob("/kaggle/input/competitions/*/environment_files"))
    cands += sorted(glob.glob("/kaggle/input/*/real_games"))
    cands += sorted(glob.glob("/kaggle/input/*/*/real_games"))
    for c in cands:
        if c and os.path.isdir(c):
            return c
    raise FileNotFoundError(
        "No games root found. Set CONFIG['GAMES_DIR'] (or $ARC_GAMES_DIR), or "
        "upload eval/real_games as a Kaggle dataset.")


def find_game_file(root, gid):
    hits = sorted(glob.glob(os.path.join(root, gid, "*", f"{gid}.py")))
    if not hits:
        hits = sorted(glob.glob(os.path.join(root, f"{gid}.py")))
    return hits[0] if hits else None


def load_baselines(cfg, root):
    """game id -> per-level human baseline actions list (games_index.json).
    Searched independently of the games root: the official competition
    environment_files ship the games but NOT the baselines."""
    cands = []
    if cfg.get("BASELINES_FILE"):
        cands.append(cfg["BASELINES_FILE"])
    cands += [os.path.join(root, "games_index.json"),
              os.path.join(os.path.dirname(root), "games_index.json"),
              os.path.join(HERE, "eval", "real_games", "games_index.json"),
              "/kaggle/working/games_index.json"]
    if os.path.isdir("/kaggle/input"):
        for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):
            cands += sorted(glob.glob(f"/kaggle/input/{depth}/games_index.json"))
    for idx in cands:
        if idx and os.path.exists(idx):
            with open(idx, "r", encoding="utf-8") as f:
                data = json.load(f)
            print(f"baselines:  {idx}", flush=True)
            return {g["id"]: list(g.get("baseline_steps", [])) for g in data}
    print("baselines:  NOT FOUND -- level_score/RHAE will read 0.0000 even for "
          "completed levels.\n            Add eval/real_games/games_index.json "
          "to a dataset or set CONFIG['BASELINES_FILE'].", flush=True)
    return {}


def class_name_from_file(path):
    import importlib.util
    from arcengine import ARCBaseGame
    spec = importlib.util.spec_from_file_location("_probe_" + os.path.basename(path), path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    for name, obj in vars(mod).items():
        if isinstance(obj, type) and issubclass(obj, ARCBaseGame) and obj is not ARCBaseGame:
            return name
    raise RuntimeError(f"No ARCBaseGame subclass in {path}")


def run_game(game_file, cfg):
    """The exact Kaggle framework loop (choose_action / is_done), plus per-level
    action bookkeeping: the action count at the FIRST banking of each level, so
    per-level RHAE can be computed (a GAME_OVER->RESET later never un-banks it)."""
    from arc_agi.local_wrapper import LocalEnvironmentWrapper
    from arc_agi.models import EnvironmentInfo
    from my_agent import MyAgent

    seed, max_actions = cfg["SEED"], cfg["MAX_ACTIONS"]
    time_cap = cfg["TIME_CAP_SECS"]
    progress_every = cfg["PROGRESS_EVERY"]

    gid = os.path.splitext(os.path.basename(game_file))[0]
    info = EnvironmentInfo(game_id=gid, class_name=class_name_from_file(game_file),
                           local_dir=os.path.dirname(os.path.abspath(game_file)))
    lg = logging.getLogger("kaggle_test")
    lg.setLevel(logging.ERROR)
    if not lg.handlers:
        lg.addHandler(logging.StreamHandler(sys.stderr))

    wrapper = LocalEnvironmentWrapper(info, lg, scorecard_id="local", seed=seed)
    frame0 = wrapper.observation_space
    if frame0 is None:
        return {"game": gid, "error": "failed to load/reset game"}

    win_levels = int(getattr(frame0, "win_levels", 0) or 0)
    os.environ["ARC_AGENT_SEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    agent = MyAgent(game_id=gid)
    frames = [frame0]

    t0 = time.time()
    best = 0
    actions_at_level = {}          # level number (1-indexed) -> action_counter when banked
    stall = None
    
    os.makedirs("traces", exist_ok=True)
    trace_file = os.path.join("traces", f"trace_{gid}_{int(time.time())}.jsonl")
    with open(trace_file, "w", encoding="utf-8") as tf:
        tf.write(json.dumps({"game": gid, "seed": seed, "max_actions": max_actions, "win_levels": win_levels}) + "\n")
        
        while not agent.is_done(frames, frames[-1]) and agent.action_counter <= max_actions:
            if time_cap and time.time() - t0 > time_cap:
                stall = f"time cap {time_cap}s hit"
                break
            try:
                action = agent.choose_action(frames, frames[-1])
            except Exception as e:
                stall = f"choose_action raised: {e!r}"
                traceback.print_exc()
                break
            data = {}
            if hasattr(action, "is_complex") and action.is_complex():
                data = {"x": int(action.action_data.x), "y": int(action.action_data.y)}
            resp = wrapper.step(action, data=data)
            if resp is None:
                stall = "wrapper.step returned None"
                break
            frames.append(resp)
            agent.action_counter += 1
            lv = int(getattr(resp, "levels_completed", 0) or 0)
            while lv > best:               # bank each newly completed level exactly once
                best += 1
                actions_at_level[best] = agent.action_counter
            
            # --- Detailed Tracking Dump ---
            trace_entry = {
                "step": agent.action_counter,
                "time_offset_s": round(time.time() - t0, 3),
                "levels_completed": lv,
                "chosen_action": getattr(agent, "last_action_str", str(action)),
            }
            if hasattr(agent, "world_model") and agent.world_model.active_model:
                trace_entry["active_hypothesis"] = getattr(agent.world_model.active_model, "hypothesis", None)
                trace_entry["active_model_source"] = getattr(agent.world_model.active_model, "source", None)
            if hasattr(agent, "eplanner"):
                trace_entry["eplanner_plan_len"] = len(getattr(agent.eplanner, "_plan", []))
                trace_entry["eplanner_walls"] = list(getattr(agent.eplanner, "walls", []))
                trace_entry["eplanner_target"] = getattr(agent.eplanner, "_target", None)
            if hasattr(agent, "cplanner"):
                trace_entry["cplanner_searching"] = getattr(agent.cplanner, "searching", False)
            if hasattr(agent, "llm_calls_this_level"):
                trace_entry["llm_calls_this_level"] = agent.llm_calls_this_level
            
            # Dump the grid to visualize what happened
            if hasattr(resp, "frame") and resp.frame:
                trace_entry["grid"] = resp.frame
            elif hasattr(resp, "grid") and resp.grid:
                trace_entry["grid"] = resp.grid.tolist() if hasattr(resp.grid, "tolist") else resp.grid

            def _np_default(o):
                if isinstance(o, np.ndarray): return o.tolist()
                if isinstance(o, np.integer): return int(o)
                if isinstance(o, np.floating): return float(o)
                raise TypeError(f"Object of type {o.__class__.__name__} is not JSON serializable")

            tf.write(json.dumps(trace_entry, default=_np_default) + "\n")
            tf.flush()
            
            if progress_every and agent.action_counter % progress_every == 0:
                print(f"    ... {gid}: {agent.action_counter}/{max_actions} actions, "
                      f"levels {best}/{win_levels}, {time.time() - t0:.0f}s", flush=True)

    state = getattr(frames[-1], "state", None)
    return {
        "game": gid,
        "levels": best,
        "win_levels": win_levels,
        "actions": agent.action_counter,
        "actions_at_level": actions_at_level,
        "state": getattr(state, "name", str(state)),
        "secs": round(time.time() - t0, 1),
        "stall": stall,
    }


def rhae(result, baseline_steps):
    """Competition scoring for one game. Per-level AI cost = actions between the
    previous bank and this bank (deaths/resets in between are paid for)."""
    win = result["win_levels"]
    if not win:
        return 0.0, []
    denom = sum(range(1, win + 1))
    per_level = []
    total = 0.0
    prev = 0
    for i in range(1, result["levels"] + 1):
        ai = result["actions_at_level"][i] - prev
        prev = result["actions_at_level"][i]
        human = baseline_steps[i - 1] if i - 1 < len(baseline_steps) else None
        if human is None or ai <= 0:
            score = 0.0
        else:
            score = min(1.15, human / ai) ** 2
        per_level.append((i, ai, human, score))
        total += score * i
    return total / denom, per_level


def main(config=None):
    cfg = dict(CONFIG)
    if config:
        cfg.update(config)

    # CLI overrides -- only when run as a real script. parse_known_args() also
    # tolerates ipykernel's injected `-f kernel.json`, but in a notebook we
    # prefer CONFIG as the single source of truth, so skip argv entirely there.
    if not _in_notebook():
        import argparse
        ap = argparse.ArgumentParser()
        ap.add_argument("--games", nargs="*", default=cfg["GAMES"])
        ap.add_argument("--seed", type=int, default=cfg["SEED"])
        ap.add_argument("--max-actions", type=int, default=cfg["MAX_ACTIONS"])
        ap.add_argument("--time-cap", type=int, default=cfg["TIME_CAP_SECS"])
        args, _ = ap.parse_known_args()
        cfg.update(GAMES=args.games, SEED=args.seed,
                   MAX_ACTIONS=args.max_actions, TIME_CAP_SECS=args.time_cap)

    if _in_notebook() and os.environ.get("PYTHONHASHSEED") is None:
        print("NOTE: PYTHONHASHSEED is not pinned in this kernel -- results are "
              "deterministic within this session but may differ after a kernel "
              "restart. Use the `!python` route for exact reproducibility.\n",
              flush=True)

    try:
        from arc_agi.local_wrapper import LocalEnvironmentWrapper  # noqa: F401
    except ImportError:
        print("arc_agi is not installed. Run:  pip install arc-agi python-dotenv",
              flush=True)
        return None

    try:
        agent_dir = _setup_paths(cfg)
        root = find_games_root(cfg)
    except FileNotFoundError as e:
        print(f"SETUP ERROR: {e}", flush=True)
        return None

    print(f"agent:      {os.path.join(agent_dir, 'my_agent.py')}", flush=True)
    print(f"games root: {root}", flush=True)
    baselines = load_baselines(cfg, root)
    print(f"running {len(cfg['GAMES'])} games, seed={cfg['SEED']}, "
          f"max_actions={cfg['MAX_ACTIONS']}"
          + (f", time_cap={cfg['TIME_CAP_SECS']}s" if cfg["TIME_CAP_SECS"] else "")
          + "\n", flush=True)

    scores, results = [], []
    for raw_gid in cfg["GAMES"]:
        if os.path.isfile(raw_gid):
            gf = raw_gid
            gid = os.path.splitext(os.path.basename(gf))[0]
        else:
            gid = raw_gid
            gf = find_game_file(root, gid)
            
        if gf is None:
            print(f"  {gid:<6} MISSING (no game file under {root})", flush=True)
            scores.append(0.0)
            continue
        r = run_game(gf, cfg)
        results.append(r)
        if "error" in r:
            print(f"  {gid:<6} ERROR: {r['error']}", flush=True)
            scores.append(0.0)
            continue
        gs, per_level = rhae(r, baselines.get(gid, []))
        scores.append(gs)
        stall = f"  [{r['stall']}]" if r["stall"] else ""
        print(f"  {gid:<6} levels {r['levels']}/{r['win_levels']}  "
              f"actions={r['actions']}  RHAE={gs:.4f}  "
              f"state={r['state']}  {r['secs']}s{stall}", flush=True)
        for i, ai, human, s in per_level:
            print(f"         L{i}: ai={ai}  human={human}  level_score={s:.4f}",
                  flush=True)

    if scores:
        print(f"\n  MEAN RHAE over {len(scores)} games: "
              f"{sum(scores) / len(scores):.4f}", flush=True)
        print("  (competition overall = mean over all 25 games; these 5 are one "
              "per type)", flush=True)
    return results


if __name__ == "__main__":
    main()


Writing /kaggle/working/Kaggle_test.py


# Running The Test

In [6]:
!PYTHONIOENCODING=utf-8 PYTHONHASHSEED=0 python -u Kaggle_test.py 
#--games lp85 --seed 0 2>&1

agent:      /kaggle/working/my_agent.py
games root: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files
baselines:  /kaggle/input/datasets/banwait13/models/games_index.json
running 5 games, seed=0, max_actions=1500

Embedder init failed (Path /kaggle/input/notebooks/banwait13/datasets-for-arc-agi/models/all-MiniLM-L6-v2 not found); L3 -> flat storage.
You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
[level] score 0 -> 1; refreshing per-level budgets
[LLM] load failed (The checkpoint you are trying to load has model type `qwen3_5` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version th

In [7]:
# 1. IF KAGGLE IS RUNNING THE OFFICIAL TEST:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    # Wait for the hidden game server to wake up
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the agent framework into our working folder
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Inject our custom agent (from Cell 2) into their framework
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Override the default settings to point to our agent
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    # Setup the offline network connections
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Fire up the agent!
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

# 2. IF YOU ARE JUST SAVING THE NOTEBOOK WHILE CODING:
else:
    # Make a dummy submission file so Kaggle doesn't crash on save
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print("Created dummy submission file. Setup complete!")

Created dummy submission file. Setup complete!
